# Public research notebook

This notebook is a cleaned public version of the original
research workflow.

## Execution model

- All filesystem paths are relative to the repository root.
- No external mounted filesystem is required.
- Stored cell outputs have been removed.
- Generated files are written below the local `results/`
  directory.
- The archival source notebook remains unchanged.


In [ ]:
# Portable repository configuration
#
# The notebook assumes that it is executed from the repository
# root or from a cloned copy of the repository.

from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()

# Move upward when the notebook is launched from a nested folder.
if REPOSITORY_ROOT.name in {
    "lorenz",
    "rossler",
    "duffing",
    "kuramoto",
    "stuart_landau",
    "coupled_map_lattice",
}:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parents[1]

DATA_DIR = REPOSITORY_ROOT / "data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
# ============================================================
# KURAMOTO DATA GENERATION — BASELINE DATASET
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Parameters
# ----------------------------

SEED = 42
rng = np.random.default_rng(SEED)

N = 300                  # number of oscillators
T = 80.0                 # total simulation time
dt = 0.05                # time step
steps = int(T / dt)

K_values = np.linspace(0.1, 5.0, 25)

omega_mean = 0.0
omega_std = 1.0

# Natural frequencies
omega = rng.normal(omega_mean, omega_std, N)

# Initial phases
theta0 = rng.uniform(0, 2*np.pi, N)

# ----------------------------
# Kuramoto simulator
# ----------------------------

def kuramoto_step(theta, omega, K, dt):
    """
    Fully connected Kuramoto model:
    dtheta_i/dt = omega_i + (K/N) * sum_j sin(theta_j - theta_i)
    """
    phase_diff = theta[None, :] - theta[:, None]
    coupling = (K / len(theta)) * np.sum(np.sin(phase_diff), axis=1)
    return theta + dt * (omega + coupling)

def order_parameter(theta):
    """
    R(t) = |mean(exp(i theta_j))|
    """
    z = np.mean(np.exp(1j * theta))
    return np.abs(z), np.angle(z)

def simulate_kuramoto(K, theta0, omega, steps, dt):
    theta = theta0.copy()

    theta_history = np.zeros((steps, len(theta)))
    R_history = np.zeros(steps)
    psi_history = np.zeros(steps)

    for t in range(steps):
        theta = kuramoto_step(theta, omega, K, dt)
        theta = np.mod(theta, 2*np.pi)

        R, psi = order_parameter(theta)

        theta_history[t] = theta
        R_history[t] = R
        psi_history[t] = psi

    return theta_history, R_history, psi_history

# ----------------------------
# Run K scan
# ----------------------------

kuramoto_data = {}

for K in K_values:
    print(f"Running K = {K:.3f}")
    theta_hist, R_hist, psi_hist = simulate_kuramoto(K, theta0, omega, steps, dt)

    kuramoto_data[float(K)] = {
        "theta": theta_hist,
        "R": R_hist,
        "psi": psi_hist,
        "mean_R": float(np.mean(R_hist[int(0.2*steps):])),
        "var_R": float(np.var(R_hist[int(0.2*steps):]))
    }

# ----------------------------
# Summary table
# ----------------------------

summary_rows = []

for K, data in kuramoto_data.items():
    summary_rows.append({
        "K": K,
        "mean_R": data["mean_R"],
        "var_R": data["var_R"]
    })

kuramoto_summary_df = pd.DataFrame(summary_rows)

print("\nKURAMOTO BASELINE SUMMARY")
display(kuramoto_summary_df)

# ----------------------------
# Save summary
# ----------------------------

kuramoto_summary_df.to_csv("kuramoto_baseline_summary.csv", index=False)

print("\nSaved file:")
print("kuramoto_baseline_summary.csv")
print("\nREADY.")

In [ ]:
# ============================================================
# KURAMOTO — FIRST DELTA-WINDOW TEST
# low / transition / high synchronization regimes
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# SELECT REGIMES
# ------------------------------------------------------------

selected_K = {
    "low": 1.121,
    "transition": 1.529,
    "high": 3.163
}

# ------------------------------------------------------------
# DOMAIN DEFINITIONS
# ------------------------------------------------------------

N_PHASE_BINS = 8

def build_phase_labels(psi_series, n_bins=N_PHASE_BINS):
    """
    Convert global phase psi(t) into discrete phase sectors.
    """
    psi_mod = np.mod(psi_series, 2*np.pi)

    edges = np.linspace(0, 2*np.pi, n_bins + 1)

    labels = np.digitize(psi_mod, edges) - 1
    labels[labels == n_bins] = n_bins - 1

    return labels

# ------------------------------------------------------------
# DWELL TIMES
# ------------------------------------------------------------

def compute_dwell_times(labels):

    dwell = []

    current = labels[0]
    length = 1

    for x in labels[1:]:

        if x == current:
            length += 1
        else:
            dwell.append(length)
            current = x
            length = 1

    dwell.append(length)

    return np.array(dwell)

# ------------------------------------------------------------
# T_global(Δ)
# ------------------------------------------------------------

def compute_T_global(labels, delta_grid):

    N = len(labels)

    T = []

    for delta in delta_grid:

        compatible = 0

        for i in range(N - 1):

            if abs(labels[i+1] - labels[i]) <= delta:
                compatible += 1

        T.append(compatible / (N - 1))

    return np.array(T)

# ------------------------------------------------------------
# T_local(Δ)
# ------------------------------------------------------------

def compute_T_local(labels, delta_grid, window=20):

    N = len(labels)

    T_local = []

    for delta in delta_grid:

        local_vals = []

        for i in range(N - window):

            segment = labels[i:i+window]

            comp = 0

            for j in range(window - 1):

                if abs(segment[j+1] - segment[j]) <= delta:
                    comp += 1

            local_vals.append(comp / (window - 1))

        T_local.append(np.mean(local_vals))

    return np.array(T_local)

# ------------------------------------------------------------
# ANALYSIS
# ------------------------------------------------------------

delta_grid = np.linspace(0, 7, 100)

results = []

fig, axes = plt.subplots(3, 3, figsize=(16, 14))

for row_idx, (regime, K) in enumerate(selected_K.items()):

    closest_K = min(
        kuramoto_data.keys(),
        key=lambda x: abs(x - K)
    )

    data = kuramoto_data[closest_K]

    psi = data["psi"]
    R = data["R"]

    labels = build_phase_labels(psi)
    dwell = compute_dwell_times(labels)
    # ------------------------
    # ORIGINAL
    # ------------------------

    labels = build_phase_labels(psi)

    dwell = compute_dwell_times(labels)

    T_global = compute_T_global(labels, delta_grid)
    T_local = compute_T_local(labels, delta_grid)

    dT = np.gradient(T_global, delta_grid)

    delta_star = delta_grid[np.argmax(dT)]

    local_global = T_local / (T_global + 1e-12)

    # ------------------------
    # SHUFFLED
    # ------------------------

    shuffled_labels = labels.copy()
    np.random.shuffle(shuffled_labels)

    shuffled_dwell = compute_dwell_times(shuffled_labels)

    # ========================================================
    # PLOT 1 — T_global
    # ========================================================

    ax = axes[row_idx, 0]

    ax.plot(delta_grid, T_global, label="original")
    ax.axvline(delta_star, linestyle="--", alpha=0.7)

    ax.set_title(f"{regime} K={K:.3f} — T_global")
    ax.set_xlabel("Δ")
    ax.set_ylabel("T_global")
    ax.grid(True)

    # ========================================================
    # PLOT 2 — dwell histogram
    # ========================================================

    ax = axes[row_idx, 1]

    ax.hist(dwell, bins=30, alpha=0.6, label="original")
    ax.hist(shuffled_dwell, bins=30, alpha=0.6, label="shuffled")

    ax.set_yscale("log")

    ax.set_title(f"{regime} — dwell histogram")
    ax.set_xlabel("dwell time")
    ax.legend()

    # ========================================================
    # PLOT 3 — dwell CCDF
    # ========================================================

    ax = axes[row_idx, 2]

    def ccdf(x):

        x_sorted = np.sort(x)

        y = 1.0 - np.arange(len(x_sorted)) / len(x_sorted)

        return x_sorted, y

    x1, y1 = ccdf(dwell)
    x2, y2 = ccdf(shuffled_dwell)

    ax.plot(x1, y1, label="original")
    ax.plot(x2, y2, label="shuffled")

    ax.set_yscale("log")

    ax.set_title(f"{regime} — dwell CCDF")
    ax.set_xlabel("dwell time")
    ax.set_ylabel("P(dwell ≥ x)")
    ax.legend()

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    results.append({

        "regime": regime,
        "K": K,

        "mean_R": data["mean_R"],
        "var_R": data["var_R"],

        "delta_star": delta_star,
        "max_dT": np.max(dT),

        "local_global_at_delta_star":
            local_global[np.argmax(dT)],

        "mean_dwell_original":
            np.mean(dwell),

        "mean_dwell_shuffled":
            np.mean(shuffled_dwell),

        "dwell_contrast":
            np.mean(dwell) / np.mean(shuffled_dwell),

        "transition_count_original":
            len(dwell),

        "transition_count_shuffled":
            len(shuffled_dwell)
    })

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# SUMMARY TABLE
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n" + "="*80)
print("KURAMOTO DELTA-WINDOW SUMMARY")
print("="*80)

display(results_df)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

results_df.to_csv(
    "kuramoto_delta_window_summary.csv",
    index=False
)

print("\nSaved file:")
print("kuramoto_delta_window_summary.csv")

print("\nDONE.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# KURAMOTO PHASE-DISTANCE Δ-WINDOW
# Delta computed from phase distances between oscillators
# ============================================================

def phase_distance_matrix(theta):
    """
    Circular phase distance between oscillators at one time step.
    theta: shape (N,)
    returns condensed pairwise distances
    """
    diff = theta[:, None] - theta[None, :]
    dist = np.abs(np.angle(np.exp(1j * diff)))
    iu = np.triu_indices_from(dist, k=1)
    return dist[iu]


def phase_distance_series(theta_series, sample_every=5, max_pairs=20000, seed=123):
    """
    Build time series of pairwise phase-distance samples.
    theta_series: shape (T, N)
    returns array of sampled phase distances over time
    """
    rng = np.random.default_rng(seed)
    all_dists = []

    for t in range(0, theta_series.shape[0], sample_every):
        d = phase_distance_matrix(theta_series[t])

        if len(d) > max_pairs:
            idx = rng.choice(len(d), size=max_pairs, replace=False)
            d = d[idx]

        all_dists.append(d)

    return np.concatenate(all_dists)


def compute_T_delta(distances, delta_grid):
    """
    T(Δ) = fraction of phase distances <= Δ
    """
    return np.array([np.mean(distances <= d) for d in delta_grid])


def compute_delta_star(distances, n_grid=200):
    """
    Δ* = point of maximum slope dT/dΔ
    """
    delta_grid = np.linspace(0, np.pi, n_grid)
    T = compute_T_delta(distances, delta_grid)
    dT = np.gradient(T, delta_grid)

    idx = np.argmax(dT)

    return {
        "delta_grid": delta_grid,
        "T": T,
        "dT": dT,
        "delta_star": delta_grid[idx],
        "max_dT": dT[idx]
    }


def phase_sector_labels(theta_series, n_sectors=8):
    """
    Label global phase position by mean phase sector.
    """
    mean_phase = np.angle(np.mean(np.exp(1j * theta_series), axis=1))
    mean_phase = (mean_phase + 2*np.pi) % (2*np.pi)
    labels = np.floor(mean_phase / (2*np.pi / n_sectors)).astype(int)
    return labels


def dwell_times(labels, dt=1.0):
    """
    Consecutive residence times in same label.
    """
    labels = np.asarray(labels)
    runs = []
    current = labels[0]
    count = 1

    for x in labels[1:]:
        if x == current:
            count += 1
        else:
            runs.append(count * dt)
            current = x
            count = 1

    runs.append(count * dt)
    return np.array(runs)


def shuffled_dwell(labels, dt=1.0, seed=123):
    rng = np.random.default_rng(seed)
    shuffled = np.array(labels).copy()
    rng.shuffle(shuffled)
    return dwell_times(shuffled, dt=dt)


# ============================================================
# REQUIREMENT:
# kuramoto_data should already exist from previous cell:
# kuramoto_data[K]["theta"] or ["psi"]
# ============================================================

if "kuramoto_data" not in globals():
    raise RuntimeError("Required Kuramoto data are missing.")

# Detect theta key
example_key = list(kuramoto_data.keys())[0]
example_data = kuramoto_data[example_key]

if "theta" in example_data:
    theta_key = "theta"
elif "psi" in example_data:
    theta_key = "psi"
else:
    raise KeyError("Nie found ani 'theta', ani 'psi' w kuramoto_data[K].")

print("Detected phase key:", theta_key)

# Pick regimes automatically from previous baseline if possible
if "kuramoto_summary_df" in globals():
    low_K = float(kuramoto_summary_df.loc[kuramoto_summary_df["mean_R"].idxmin(), "K"])

    # transition = max variance R
    transition_K = float(kuramoto_summary_df.loc[kuramoto_summary_df["var_R"].idxmax(), "K"])

    high_K = float(kuramoto_summary_df.loc[kuramoto_summary_df["mean_R"].idxmax(), "K"])

    selected_K = {
        "low": low_K,
        "transition": transition_K,
        "high": high_K
    }
else:
    K_values = sorted(list(kuramoto_data.keys()))
    selected_K = {
        "low": K_values[len(K_values)//4],
        "transition": K_values[len(K_values)//2],
        "high": K_values[-1]
    }

print("Selected K regimes:")
print(selected_K)

# ============================================================
# MAIN ANALYSIS
# ============================================================

rows = []
all_results = {}

for regime, K in selected_K.items():
    # nearest K safety
    K_real = min(kuramoto_data.keys(), key=lambda x: abs(float(x) - float(K)))
    data = kuramoto_data[K_real]
    theta = np.asarray(data[theta_key])

    print(f"\nRunning regime={regime}, K={K_real}")

    # phase-distance Δ-window
    distances = phase_distance_series(
        theta,
        sample_every=5,
        max_pairs=15000,
        seed=123
    )

    delta_res = compute_delta_star(distances, n_grid=250)

    # dwell-time on phase sectors
    labels = phase_sector_labels(theta, n_sectors=8)
    dwell_orig = dwell_times(labels, dt=1.0)
    dwell_shuf = shuffled_dwell(labels, dt=1.0, seed=123)

    mean_dwell_orig = np.mean(dwell_orig)
    mean_dwell_shuf = np.mean(dwell_shuf)
    dwell_contrast = mean_dwell_orig / mean_dwell_shuf if mean_dwell_shuf > 0 else np.nan

    transition_count_orig = len(dwell_orig) - 1
    transition_count_shuf = len(dwell_shuf) - 1

    mean_R = data.get("mean_R", np.nan)
    var_R = data.get("var_R", np.nan)

    if np.isnan(mean_R) and "R" in data:
        mean_R = np.mean(data["R"])
        var_R = np.var(data["R"])

    rows.append({
        "regime": regime,
        "K": K_real,
        "mean_R": mean_R,
        "var_R": var_R,
        "phase_delta_star": delta_res["delta_star"],
        "phase_max_dT": delta_res["max_dT"],
        "mean_dwell_original": mean_dwell_orig,
        "mean_dwell_shuffled": mean_dwell_shuf,
        "dwell_contrast": dwell_contrast,
        "transition_count_original": transition_count_orig,
        "transition_count_shuffled": transition_count_shuf
    })

    all_results[regime] = {
        "K": K_real,
        "theta": theta,
        "distances": distances,
        "delta_res": delta_res,
        "dwell_original": dwell_orig,
        "dwell_shuffled": dwell_shuf,
        "labels": labels
    }

# ============================================================
# SUMMARY TABLE
# ============================================================

phase_distance_df = pd.DataFrame(rows)

print("\n" + "="*80)
print("KURAMOTO PHASE-DISTANCE Δ-WINDOW SUMMARY")
print("="*80)
display(phase_distance_df)

phase_distance_df.to_csv("kuramoto_phase_distance_delta_window_summary.csv", index=False)
print("\nSaved file:")
print("kuramoto_phase_distance_delta_window_summary.csv")

# ============================================================
# PLOTS
# ============================================================

fig, axes = plt.subplots(len(selected_K), 3, figsize=(15, 4 * len(selected_K)))

if len(selected_K) == 1:
    axes = np.array([axes])

for row_idx, (regime, res) in enumerate(all_results.items()):
    delta_grid = res["delta_res"]["delta_grid"]
    T = res["delta_res"]["T"]
    dT = res["delta_res"]["dT"]
    delta_star = res["delta_res"]["delta_star"]

    dwell_orig = res["dwell_original"]
    dwell_shuf = res["dwell_shuffled"]

    # T(Δ)
    ax = axes[row_idx, 0]
    ax.plot(delta_grid, T, label="T_phase(Δ)")
    ax.axvline(delta_star, linestyle="--", label="phase Δ*")
    ax.set_title(f"{regime} | K={res['K']:.3f} | phase T(Δ)")
    ax.set_xlabel("Δ phase distance")
    ax.set_ylabel("T(Δ)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # dT/dΔ
    ax = axes[row_idx, 1]
    ax.plot(delta_grid, dT)
    ax.axvline(delta_star, linestyle="--", label="phase Δ*")
    ax.set_title(f"{regime} | dT/dΔ")
    ax.set_xlabel("Δ phase distance")
    ax.set_ylabel("dT/dΔ")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # dwell CCDF
    ax = axes[row_idx, 2]

    def ccdf(x):
        x = np.sort(np.asarray(x))
        y = 1.0 - np.arange(len(x)) / len(x)
        return x, y

    xo, yo = ccdf(dwell_orig)
    xs, ys = ccdf(dwell_shuf)

    ax.step(xo, yo, where="post", label="original")
    ax.step(xs, ys, where="post", label="shuffled")
    ax.set_yscale("log")
    ax.set_title(f"{regime} | dwell CCDF")
    ax.set_xlabel("dwell time")
    ax.set_ylabel("P(dwell ≥ x)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("kuramoto_phase_distance_delta_window_plots.png", dpi=200)
plt.show()

print("\nSaved plot:")
print("kuramoto_phase_distance_delta_window_plots.png")

print("\nDONE.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# STUART–LANDAU / AMPLITUDE + PHASE Δ-WINDOW TEST
# ============================================================

np.random.seed(42)

# ----------------------------
# Parameters
# ----------------------------
N = 80
T = 250
dt = 0.03
steps = int(T / dt)
burn = int(0.4 * steps)

K_values = np.linspace(0.1, 5.0, 25)
lambda_amp = 1.0

omega = np.random.normal(0.0, 1.0, N)

# ----------------------------
# Utilities
# ----------------------------

def wrapped_phase_distance(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

def simulate_stuart_landau(K):
    """
    Complex Stuart–Landau oscillators:
    dz_i/dt = (1 + iω_i - |z_i|^2)z_i + K/N * Σ(z_j - z_i)
    """
    z = (0.5 + 0.2*np.random.randn(N)) * np.exp(1j * np.random.uniform(0, 2*np.pi, N))
    Z_record = []

    for t in range(steps):
        coupling = K * (np.mean(z) - z)
        dz = (1 + 1j*omega - np.abs(z)**2) * z + coupling
        z = z + dt * dz

        if t >= burn:
            Z_record.append(z.copy())

    return np.array(Z_record)

def order_parameter(z_series):
    theta = np.angle(z_series)
    R = np.abs(np.mean(np.exp(1j * theta), axis=1))
    return R

def amplitude_phase_distance_matrix(theta, amp):
    """
    Pairwise Δ_ij = sqrt(phase_distance^2 + lambda*(amplitude difference)^2)
    for each time step, flattened.
    """
    all_d = []

    for t in range(len(theta)):
        th = theta[t]
        A = amp[t]

        dtheta = wrapped_phase_distance(th[:, None], th[None, :])
        dA = A[:, None] - A[None, :]

        D = np.sqrt(dtheta**2 + lambda_amp * dA**2)

        iu = np.triu_indices(N, k=1)
        all_d.append(D[iu])

    return np.concatenate(all_d)

def local_amplitude_phase_distances(theta, amp):
    """
    Temporally local distance between consecutive system states.
    Uses oscillator-wise amplitude+phase distance and averages over oscillators.
    """
    dlist = []

    for t in range(len(theta)-1):
        dtheta = wrapped_phase_distance(theta[t+1], theta[t])
        dA = amp[t+1] - amp[t]
        d = np.sqrt(dtheta**2 + lambda_amp * dA**2)
        dlist.append(np.mean(d))

    return np.array(dlist)

def transition_curve(distances, grid):
    return np.array([np.mean(distances <= d) for d in grid])

def delta_star_from_curve(grid, Tcurve):
    dT = np.gradient(Tcurve, grid)
    idx = np.argmax(dT)
    return grid[idx], dT[idx]

def dwell_times_from_phase(theta, n_sectors=12):
    """
    Domain = mean phase sector of the whole oscillator population.
    This is simple and robust for first test.
    """
    mean_phase = np.angle(np.mean(np.exp(1j * theta), axis=1))
    mean_phase = (mean_phase + 2*np.pi) % (2*np.pi)

    sectors = np.floor(mean_phase / (2*np.pi/n_sectors)).astype(int)

    dwells = []
    current = sectors[0]
    count = 1

    for s in sectors[1:]:
        if s == current:
            count += 1
        else:
            dwells.append(count)
            current = s
            count = 1

    dwells.append(count)
    return np.array(dwells), sectors

def shuffled_dwell(theta, n_sectors=12):
    idx = np.random.permutation(len(theta))
    theta_shuf = theta[idx]
    return dwell_times_from_phase(theta_shuf, n_sectors=n_sectors)[0]

def random_control_like(theta, amp):
    theta_r = np.random.uniform(-np.pi, np.pi, theta.shape)
    amp_r = np.random.permutation(amp.flatten()).reshape(amp.shape)
    return theta_r, amp_r

def ar1_control_like(theta, amp, phi=0.9):
    """
    Simple AR(1)-like control on phase and amplitude.
    Not a perfect circular AR model, but good as first negative/memory control.
    """
    Tn, Nn = theta.shape
    theta_ar = np.zeros_like(theta)
    amp_ar = np.zeros_like(amp)

    theta_ar[0] = np.random.uniform(-np.pi, np.pi, Nn)
    amp_ar[0] = np.mean(amp) + np.std(amp) * np.random.randn(Nn)

    for t in range(1, Tn):
        theta_ar[t] = phi * theta_ar[t-1] + np.sqrt(1-phi**2) * np.random.randn(Nn)
        theta_ar[t] = (theta_ar[t] + np.pi) % (2*np.pi) - np.pi

        amp_ar[t] = phi * amp_ar[t-1] + np.sqrt(1-phi**2) * np.std(amp) * np.random.randn(Nn)

    amp_ar = np.abs(amp_ar)
    return theta_ar, amp_ar

# ----------------------------
# Main scan
# ----------------------------

results = []

saved_examples = {}

for K in K_values:
    print(f"Running K = {K:.3f}")

    z_series = simulate_stuart_landau(K)
    theta = np.angle(z_series)
    amp = np.abs(z_series)

    R = order_parameter(z_series)
    mean_R = np.mean(R)
    var_R = np.var(R)

    # distances
    global_d = amplitude_phase_distance_matrix(theta, amp)
    local_d = local_amplitude_phase_distances(theta, amp)

    max_d = np.percentile(global_d, 99.5)
    grid = np.linspace(0.001, max_d, 160)

    T_global = transition_curve(global_d, grid)
    T_local = transition_curve(local_d, grid)

    delta_star, max_dT = delta_star_from_curve(grid, T_global)
    lg_ratio = np.interp(delta_star, grid, T_local) / max(np.interp(delta_star, grid, T_global), 1e-12)

    # dwell original
    dw_orig, sectors = dwell_times_from_phase(theta)
    mean_dwell_orig = np.mean(dw_orig)
    transition_orig = len(dw_orig) - 1

    # shuffled
    dw_shuf = shuffled_dwell(theta)
    mean_dwell_shuf = np.mean(dw_shuf)
    transition_shuf = len(dw_shuf) - 1

    dwell_contrast = mean_dwell_orig / max(mean_dwell_shuf, 1e-12)
    transition_contrast = transition_shuf / max(transition_orig, 1)

    # random control
    theta_rand, amp_rand = random_control_like(theta, amp)
    dw_rand, _ = dwell_times_from_phase(theta_rand)
    mean_dwell_rand = np.mean(dw_rand)

    # AR(1)
    theta_ar, amp_ar = ar1_control_like(theta, amp)
    dw_ar, _ = dwell_times_from_phase(theta_ar)
    mean_dwell_ar = np.mean(dw_ar)

    results.append({
        "K": K,
        "mean_R": mean_R,
        "var_R": var_R,
        "delta_star_amp_phase": delta_star,
        "max_dT": max_dT,
        "local_global_at_delta_star": lg_ratio,
        "mean_dwell_original": mean_dwell_orig,
        "mean_dwell_shuffled": mean_dwell_shuf,
        "mean_dwell_random": mean_dwell_rand,
        "mean_dwell_ar1": mean_dwell_ar,
        "dwell_contrast_original_over_shuffle": dwell_contrast,
        "transition_count_original": transition_orig,
        "transition_count_shuffled": transition_shuf,
        "transition_contrast_shuffle_over_original": transition_contrast
    })

    # save examples near low / transition / high
    if abs(K - 0.5) < 0.12 or abs(K - 1.5) < 0.12 or abs(K - 3.0) < 0.12:
        saved_examples[round(K,3)] = {
            "grid": grid,
            "T_global": T_global,
            "T_local": T_local,
            "dw_orig": dw_orig,
            "dw_shuf": dw_shuf,
            "R": R
        }

summary_df = pd.DataFrame(results)
summary_df.to_csv("stuart_landau_amp_phase_delta_summary.csv", index=False)

print("\nSTUART–LANDAU AMP+PHASE Δ-WINDOW SUMMARY")
display(summary_df)

# ----------------------------
# Plots
# ----------------------------

plt.figure(figsize=(8,5))
plt.plot(summary_df["K"], summary_df["mean_R"], marker="o")
plt.xlabel("Coupling K")
plt.ylabel("Mean synchronization R")
plt.title("Stuart–Landau: synchronization vs K")
plt.grid(True)
plt.savefig("stuart_landau_sync_vs_K.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(summary_df["K"], summary_df["delta_star_amp_phase"], marker="o")
plt.xlabel("Coupling K")
plt.ylabel("Amplitude+phase Δ*")
plt.title("Stuart–Landau: amplitude+phase Δ* vs K")
plt.grid(True)
plt.savefig("stuart_landau_delta_star_vs_K.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(summary_df["K"], summary_df["dwell_contrast_original_over_shuffle"], marker="o")
plt.xlabel("Coupling K")
plt.ylabel("Dwell contrast original / shuffled")
plt.title("Stuart–Landau: dwell contrast vs K")
plt.grid(True)
plt.savefig("stuart_landau_dwell_contrast_vs_K.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(summary_df["K"], summary_df["local_global_at_delta_star"], marker="o")
plt.axhline(1.0, linestyle="--")
plt.xlabel("Coupling K")
plt.ylabel("Local/global at Δ*")
plt.title("Stuart–Landau: local/global vs K")
plt.grid(True)
plt.savefig("stuart_landau_local_global_vs_K.png", dpi=300, bbox_inches="tight")
plt.show()

# CCDF helper
def ccdf(data):
    x = np.sort(data)
    y = 1.0 - np.arange(len(x)) / len(x)
    return x, y

plt.figure(figsize=(8,5))
for K, ex in saved_examples.items():
    x, y = ccdf(ex["dw_orig"])
    plt.step(x, y, where="post", label=f"original K={K}")

plt.yscale("log")
plt.xlabel("Dwell time")
plt.ylabel("P(Dwell ≥ x)")
plt.title("Stuart–Landau: dwell-time CCDF original")
plt.legend()
plt.grid(True)
plt.savefig("stuart_landau_dwell_ccdf_original.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10,8))
for i, (K, ex) in enumerate(saved_examples.items(), 1):
    plt.subplot(len(saved_examples), 1, i)
    plt.plot(ex["grid"], ex["T_global"], label="T_global")
    plt.plot(ex["grid"], ex["T_local"], label="T_local")
    plt.title(f"K={K} amplitude+phase Δ-window")
    plt.xlabel("Δ")
    plt.ylabel("T(Δ)")
    plt.grid(True)
    plt.legend()

plt.tight_layout()
plt.savefig("stuart_landau_delta_window_examples.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved files:")
print("stuart_landau_amp_phase_delta_summary.csv")
print("stuart_landau_sync_vs_K.png")
print("stuart_landau_delta_star_vs_K.png")
print("stuart_landau_dwell_contrast_vs_K.png")
print("stuart_landau_local_global_vs_K.png")
print("stuart_landau_dwell_ccdf_original.png")
print("stuart_landau_delta_window_examples.png")
print("\nDONE.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# STUART–LANDAU AMP+PHASE — LAMBDA SCAN
# ============================================================

np.random.seed(123)

# ----------------------------
# Parameters
# ----------------------------
N = 80
T = 220
dt = 0.03
steps = int(T / dt)
burn = int(0.4 * steps)

K_values = np.linspace(0.1, 5.0, 25)
lambda_values = [0.0, 0.25, 0.5, 1.0, 2.0]

omega = np.random.normal(0.0, 1.0, N)

# ----------------------------
# Utilities
# ----------------------------

def wrapped_phase_distance(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

def simulate_stuart_landau(K):
    z = (0.5 + 0.2*np.random.randn(N)) * np.exp(1j * np.random.uniform(0, 2*np.pi, N))
    Z_record = []

    for t in range(steps):
        coupling = K * (np.mean(z) - z)
        dz = (1 + 1j*omega - np.abs(z)**2) * z + coupling
        z = z + dt * dz

        if t >= burn:
            Z_record.append(z.copy())

    return np.array(Z_record)

def order_parameter(z_series):
    theta = np.angle(z_series)
    R = np.abs(np.mean(np.exp(1j * theta), axis=1))
    return R

def amplitude_phase_global_distances(theta, amp, lam):
    all_d = []
    Tn, Nn = theta.shape
    iu = np.triu_indices(Nn, k=1)

    for t in range(Tn):
        th = theta[t]
        A = amp[t]

        dtheta = wrapped_phase_distance(th[:, None], th[None, :])
        dA = A[:, None] - A[None, :]

        D = np.sqrt(dtheta**2 + lam * dA**2)
        all_d.append(D[iu])

    return np.concatenate(all_d)

def amplitude_phase_local_distances(theta, amp, lam):
    dlist = []

    for t in range(len(theta)-1):
        dtheta = wrapped_phase_distance(theta[t+1], theta[t])
        dA = amp[t+1] - amp[t]
        d = np.sqrt(dtheta**2 + lam * dA**2)
        dlist.append(np.mean(d))

    return np.array(dlist)

def transition_curve(distances, grid):
    return np.array([np.mean(distances <= d) for d in grid])

def delta_star_from_curve(grid, Tcurve):
    dT = np.gradient(Tcurve, grid)
    idx = np.argmax(dT)
    return grid[idx], dT[idx]

def dwell_times_from_phase(theta, n_sectors=12):
    mean_phase = np.angle(np.mean(np.exp(1j * theta), axis=1))
    mean_phase = (mean_phase + 2*np.pi) % (2*np.pi)

    sectors = np.floor(mean_phase / (2*np.pi/n_sectors)).astype(int)

    dwells = []
    current = sectors[0]
    count = 1

    for s in sectors[1:]:
        if s == current:
            count += 1
        else:
            dwells.append(count)
            current = s
            count = 1

    dwells.append(count)
    return np.array(dwells)

def shuffled_dwell(theta, n_sectors=12):
    idx = np.random.permutation(len(theta))
    theta_shuf = theta[idx]
    return dwell_times_from_phase(theta_shuf, n_sectors=n_sectors)

# ----------------------------
# Precompute trajectories for each K
# ----------------------------

trajectories = {}

for K in K_values:
    print(f"Simulating K = {K:.3f}")
    z_series = simulate_stuart_landau(K)
    theta = np.angle(z_series)
    amp = np.abs(z_series)
    R = order_parameter(z_series)

    trajectories[K] = {
        "theta": theta,
        "amp": amp,
        "mean_R": np.mean(R),
        "var_R": np.var(R)
    }

# ----------------------------
# Lambda scan
# ----------------------------

rows = []

for lam in lambda_values:
    print("\n" + "="*70)
    print(f"Running lambda = {lam}")
    print("="*70)

    for K in K_values:
        data = trajectories[K]
        theta = data["theta"]
        amp = data["amp"]

        global_d = amplitude_phase_global_distances(theta, amp, lam)
        local_d = amplitude_phase_local_distances(theta, amp, lam)

        max_d = np.percentile(global_d, 99.5)
        grid = np.linspace(0.001, max_d, 160)

        T_global = transition_curve(global_d, grid)
        T_local = transition_curve(local_d, grid)

        delta_star, max_dT = delta_star_from_curve(grid, T_global)

        Tg = np.interp(delta_star, grid, T_global)
        Tl = np.interp(delta_star, grid, T_local)
        local_global = Tl / max(Tg, 1e-12)

        dw_orig = dwell_times_from_phase(theta)
        dw_shuf = shuffled_dwell(theta)

        mean_orig = np.mean(dw_orig)
        mean_shuf = np.mean(dw_shuf)

        dwell_contrast = mean_orig / max(mean_shuf, 1e-12)

        transition_orig = len(dw_orig) - 1
        transition_shuf = len(dw_shuf) - 1

        rows.append({
            "lambda_amp": lam,
            "K": K,
            "mean_R": data["mean_R"],
            "var_R": data["var_R"],
            "delta_star": delta_star,
            "max_dT": max_dT,
            "local_global_at_delta_star": local_global,
            "mean_dwell_original": mean_orig,
            "mean_dwell_shuffled": mean_shuf,
            "dwell_contrast": dwell_contrast,
            "transition_count_original": transition_orig,
            "transition_count_shuffled": transition_shuf,
            "transition_contrast": transition_shuf / max(transition_orig, 1)
        })

lambda_df = pd.DataFrame(rows)
lambda_df.to_csv("stuart_landau_lambda_scan_summary.csv", index=False)

print("\nSTUART–LANDAU LAMBDA SCAN SUMMARY")
display(lambda_df)

# ----------------------------
# Aggregate key peaks
# ----------------------------

peak_rows = []

for lam in lambda_values:
    sub = lambda_df[lambda_df["lambda_amp"] == lam]

    idx_dwell = sub["dwell_contrast"].idxmax()
    idx_lg = sub["local_global_at_delta_star"].idxmax()
    idx_dt = sub["max_dT"].idxmax()

    peak_rows.append({
        "lambda_amp": lam,
        "K_at_max_dwell_contrast": sub.loc[idx_dwell, "K"],
        "max_dwell_contrast": sub.loc[idx_dwell, "dwell_contrast"],
        "mean_R_at_max_dwell": sub.loc[idx_dwell, "mean_R"],
        "K_at_max_local_global": sub.loc[idx_lg, "K"],
        "max_local_global": sub.loc[idx_lg, "local_global_at_delta_star"],
        "K_at_max_dT": sub.loc[idx_dt, "K"],
        "max_dT": sub.loc[idx_dt, "max_dT"]
    })

peak_df = pd.DataFrame(peak_rows)
peak_df.to_csv("stuart_landau_lambda_scan_peaks.csv", index=False)

print("\nLAMBDA SCAN PEAK SUMMARY")
display(peak_df)

# ----------------------------
# Plots
# ----------------------------

plt.figure(figsize=(8,5))
for lam in lambda_values:
    sub = lambda_df[lambda_df["lambda_amp"] == lam]
    plt.plot(sub["K"], sub["dwell_contrast"], marker="o", label=f"λ={lam}")
plt.xlabel("Coupling K")
plt.ylabel("Dwell contrast original / shuffled")
plt.title("Stuart–Landau: dwell contrast vs K for λ")
plt.legend()
plt.grid(True)
plt.savefig("stuart_landau_lambda_dwell_contrast.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,5))
for lam in lambda_values:
    sub = lambda_df[lambda_df["lambda_amp"] == lam]
    plt.plot(sub["K"], sub["delta_star"], marker="o", label=f"λ={lam}")
plt.xlabel("Coupling K")
plt.ylabel("Δ*")
plt.title("Stuart–Landau: Δ* vs K for λ")
plt.legend()
plt.grid(True)
plt.savefig("stuart_landau_lambda_delta_star.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,5))
for lam in lambda_values:
    sub = lambda_df[lambda_df["lambda_amp"] == lam]
    plt.plot(sub["K"], sub["local_global_at_delta_star"], marker="o", label=f"λ={lam}")
plt.axhline(1.0, linestyle="--")
plt.xlabel("Coupling K")
plt.ylabel("Local/global at Δ*")
plt.title("Stuart–Landau: local/global vs K for λ")
plt.legend()
plt.grid(True)
plt.savefig("stuart_landau_lambda_local_global.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(peak_df["lambda_amp"], peak_df["K_at_max_dwell_contrast"], marker="o", label="K at max dwell contrast")
plt.plot(peak_df["lambda_amp"], peak_df["K_at_max_local_global"], marker="o", label="K at max local/global")
plt.plot(peak_df["lambda_amp"], peak_df["K_at_max_dT"], marker="o", label="K at max dT")
plt.xlabel("λ amplitude weight")
plt.ylabel("K")
plt.title("Stuart–Landau: peak locations vs λ")
plt.legend()
plt.grid(True)
plt.savefig("stuart_landau_lambda_peak_locations.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8,5))
plt.plot(peak_df["lambda_amp"], peak_df["max_dwell_contrast"], marker="o")
plt.xlabel("λ amplitude weight")
plt.ylabel("Max dwell contrast")
plt.title("Stuart–Landau: maximum dwell contrast vs λ")
plt.grid(True)
plt.savefig("stuart_landau_lambda_max_dwell.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved files:")
print("stuart_landau_lambda_scan_summary.csv")
print("stuart_landau_lambda_scan_peaks.csv")
print("stuart_landau_lambda_dwell_contrast.png")
print("stuart_landau_lambda_delta_star.png")
print("stuart_landau_lambda_local_global.png")
print("stuart_landau_lambda_peak_locations.png")
print("stuart_landau_lambda_max_dwell.png")
print("\nDONE.")

In [ ]:
# ============================================================
# STUART-LANDAU SEED ROBUSTNESS TEST
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

N = 40
dt = 0.02
steps = 5000
discard = 1000

K_values = np.array([1.3, 1.5, 1.7, 1.9, 2.1, 2.3, 2.5])

lambda_amp = 1.0

n_seeds = 30

alpha = 1.0
omega_mean = 1.0
omega_std = 0.15

delta_grid = np.linspace(0.001, 3.0, 250)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def wrapped_phase_diff(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

def compute_order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def dwell_times(binary):
    times = []
    current = binary[0]
    count = 1

    for x in binary[1:]:
        if x == current:
            count += 1
        else:
            times.append(count)
            count = 1
            current = x

    times.append(count)
    return np.array(times)

# ------------------------------------------------------------
# STUART-LANDAU SIMULATION
# ------------------------------------------------------------

def simulate_stuart_landau(K, seed):

    np.random.seed(seed)

    omega = np.random.normal(omega_mean, omega_std, N)

    A = 1 + 0.1*np.random.randn(N)
    theta = np.random.uniform(0, 2*np.pi, N)

    theta_series = []
    A_series = []
    R_series = []

    for t in range(steps):

        complex_z = A * np.exp(1j * theta)

        coupling = np.mean(complex_z) - complex_z

        dA = alpha*A - A**3 + K*np.real(
            coupling * np.exp(-1j*theta)
        )

        dtheta = omega + K*np.imag(
            coupling * np.exp(-1j*theta)
        ) / (A + 1e-8)

        A += dt * dA
        theta += dt * dtheta

        theta = np.mod(theta, 2*np.pi)

        if t >= discard:

            theta_series.append(theta.copy())
            A_series.append(A.copy())

            R_series.append(
                compute_order_parameter(theta)
            )

    return (
        np.array(theta_series),
        np.array(A_series),
        np.array(R_series)
    )

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def build_delta_series(theta_series, A_series):

    T = len(theta_series)

    delta_t = np.zeros(T)

    for t in range(T):

        theta = theta_series[t]
        A = A_series[t]

        dtheta = wrapped_phase_diff(
            theta[:, None],
            theta[None, :]
        )

        dA = np.abs(
            A[:, None] - A[None, :]
        )

        delta_matrix = np.sqrt(
            dtheta**2 + lambda_amp*(dA**2)
        )

        iu = np.triu_indices_from(delta_matrix, k=1)

        delta_t[t] = np.mean(delta_matrix[iu])

    return delta_t

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

rows = []

for seed in range(n_seeds):

    print(f"\n========================")
    print(f"SEED {seed}")
    print(f"========================")

    seed_results = []

    for K in K_values:

        theta_series, A_series, R_series = \
            simulate_stuart_landau(K, seed)

        delta_t = build_delta_series(
            theta_series,
            A_series
        )

        shuffled = np.random.permutation(delta_t)

        T_global = []
        T_local = []

        for delta in delta_grid:

            binary = (delta_t < delta).astype(int)

            dwell_orig = dwell_times(binary)

            mean_dwell_orig = np.mean(dwell_orig)

            binary_shuf = (
                shuffled < delta
            ).astype(int)

            dwell_shuf = dwell_times(binary_shuf)

            mean_dwell_shuf = np.mean(dwell_shuf)

            T_global.append(
                np.mean(binary)
            )

            T_local.append(
                mean_dwell_orig /
                (mean_dwell_shuf + 1e-8)
            )

        T_global = np.array(T_global)
        T_local = np.array(T_local)

        dT = np.gradient(T_global, delta_grid)

        idx = np.argmax(dT)

        delta_star = delta_grid[idx]

        local_global = (
            T_local[idx] /
            (T_global[idx] + 1e-8)
        )

        dwell_contrast = T_local[idx]

        seed_results.append({
            "K": K,
            "delta_star": delta_star,
            "max_dT": dT[idx],
            "local_global": local_global,
            "dwell_contrast": dwell_contrast,
            "mean_R": np.mean(R_series)
        })

    seed_df = pd.DataFrame(seed_results)

    peak_row = seed_df.iloc[
        seed_df["dwell_contrast"].idxmax()
    ]

    rows.append({
        "seed": seed,
        "K_peak": peak_row["K"],
        "max_dwell_contrast":
            peak_row["dwell_contrast"],
        "delta_star_peak":
            peak_row["delta_star"],
        "mean_R_peak":
            peak_row["mean_R"]
    })

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_df = pd.DataFrame(rows)

print("\n")
print("="*60)
print("SEED ROBUSTNESS SUMMARY")
print("="*60)
print(summary_df)

# ------------------------------------------------------------
# PEAK DISTRIBUTION
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.hist(
    summary_df["K_peak"],
    bins=np.arange(1.2, 2.7, 0.2),
)

plt.xlabel("K at maximum dwell contrast")
plt.ylabel("Count")
plt.title("Seed robustness: peak K distribution")

plt.grid(True)

plt.tight_layout()

plt.savefig(
    "stuart_landau_seed_peak_histogram.png",
    dpi=300
)

# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

peak_mean = summary_df["K_peak"].mean()
peak_std = summary_df["K_peak"].std()

print("\n")
print("="*60)
print("PEAK STATISTICS")
print("="*60)

print(f"Mean peak K: {peak_mean:.4f}")
print(f"Std peak K : {peak_std:.4f}")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

summary_df.to_csv(
    "stuart_landau_seed_robustness.csv",
    index=False
)

print("\nSaved files:")
print("stuart_landau_seed_robustness.csv")
print("stuart_landau_seed_peak_histogram.png")

print("\nDONE.")

In [ ]:
# ============================================================
# STUART-LANDAU SEED ROBUSTNESS V2
# Separate:
# - max dT peak
# - max dwell contrast peak
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

N = 40
dt = 0.02
steps = 5000
discard = 1000

K_values = np.array([1.3, 1.5, 1.7, 1.9, 2.1, 2.3, 2.5])

lambda_amp = 1.0

n_seeds = 30

alpha = 1.0
omega_mean = 1.0
omega_std = 0.15

delta_grid = np.linspace(0.001, 3.0, 300)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def wrapped_phase_diff(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

def compute_order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def dwell_times(binary):

    if len(binary) == 0:
        return np.array([0])

    times = []

    current = binary[0]
    count = 1

    for x in binary[1:]:

        if x == current:
            count += 1
        else:
            times.append(count)
            current = x
            count = 1

    times.append(count)

    return np.array(times)

# ------------------------------------------------------------
# STUART-LANDAU
# ------------------------------------------------------------

def simulate_stuart_landau(K, seed):

    np.random.seed(seed)

    omega = np.random.normal(
        omega_mean,
        omega_std,
        N
    )

    A = 1 + 0.1*np.random.randn(N)

    theta = np.random.uniform(
        0,
        2*np.pi,
        N
    )

    theta_series = []
    A_series = []
    R_series = []

    for t in range(steps):

        z = A * np.exp(1j*theta)

        mean_z = np.mean(z)

        coupling = mean_z - z

        dA = (
            alpha*A
            - A**3
            + K*np.real(
                coupling * np.exp(-1j*theta)
            )
        )

        dtheta = (
            omega
            + K*np.imag(
                coupling * np.exp(-1j*theta)
            ) / (A + 1e-8)
        )

        A += dt*dA
        theta += dt*dtheta

        theta = np.mod(theta, 2*np.pi)

        if t >= discard:

            theta_series.append(theta.copy())
            A_series.append(A.copy())

            R_series.append(
                compute_order_parameter(theta)
            )

    return (
        np.array(theta_series),
        np.array(A_series),
        np.array(R_series)
    )

# ------------------------------------------------------------
# BUILD DELTA SERIES
# ------------------------------------------------------------

def build_delta_series(theta_series, A_series):

    T = len(theta_series)

    delta_t = np.zeros(T)

    for t in range(T):

        theta = theta_series[t]
        A = A_series[t]

        dtheta = wrapped_phase_diff(
            theta[:, None],
            theta[None, :]
        )

        dA = np.abs(
            A[:, None] - A[None, :]
        )

        delta_matrix = np.sqrt(
            dtheta**2 +
            lambda_amp*(dA**2)
        )

        iu = np.triu_indices_from(
            delta_matrix,
            k=1
        )

        delta_t[t] = np.mean(
            delta_matrix[iu]
        )

    return delta_t

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

rows = []

for seed in range(n_seeds):

    print(f"\n======================")
    print(f"SEED {seed}")
    print(f"======================")

    K_metrics = []

    for K in K_values:

        theta_series, A_series, R_series = \
            simulate_stuart_landau(
                K,
                seed
            )

        delta_t = build_delta_series(
            theta_series,
            A_series
        )

        shuffled = np.random.permutation(
            delta_t
        )

        T_global = []
        T_local = []

        for delta in delta_grid:

            binary_orig = (
                delta_t < delta
            ).astype(int)

            binary_shuf = (
                shuffled < delta
            ).astype(int)

            dwell_orig = dwell_times(
                binary_orig
            )

            dwell_shuf = dwell_times(
                binary_shuf
            )

            mean_dwell_orig = np.mean(
                dwell_orig
            )

            mean_dwell_shuf = np.mean(
                dwell_shuf
            )

            T_global.append(
                np.mean(binary_orig)
            )

            T_local.append(
                mean_dwell_orig /
                (mean_dwell_shuf + 1e-8)
            )

        T_global = np.array(T_global)
        T_local = np.array(T_local)

        dT = np.gradient(
            T_global,
            delta_grid
        )

        # ------------------------------------
        # MAX dT
        # ------------------------------------

        idx_dT = np.argmax(dT)

        delta_star_dT = delta_grid[idx_dT]

        max_dT = dT[idx_dT]

        local_global_dT = (
            T_local[idx_dT] /
            (T_global[idx_dT] + 1e-8)
        )

        # ------------------------------------
        # MAX DWELL CONTRAST
        # ------------------------------------

        idx_dwell = np.argmax(T_local)

        delta_star_dwell = \
            delta_grid[idx_dwell]

        max_dwell = T_local[idx_dwell]

        local_global_dwell = (
            T_local[idx_dwell] /
            (T_global[idx_dwell] + 1e-8)
        )

        K_metrics.append({

            "K": K,

            "mean_R":
                np.mean(R_series),

            "delta_star_dT":
                delta_star_dT,

            "max_dT":
                max_dT,

            "local_global_dT":
                local_global_dT,

            "delta_star_dwell":
                delta_star_dwell,

            "max_dwell_contrast":
                max_dwell,

            "local_global_dwell":
                local_global_dwell
        })

    K_df = pd.DataFrame(K_metrics)

    # --------------------------------------------------------
    # PEAKS
    # --------------------------------------------------------

    row_dT = K_df.iloc[
        K_df["max_dT"].idxmax()
    ]

    row_dwell = K_df.iloc[
        K_df["max_dwell_contrast"].idxmax()
    ]

    rows.append({

        "seed": seed,

        "K_at_max_dT":
            row_dT["K"],

        "delta_star_dT":
            row_dT["delta_star_dT"],

        "max_dT":
            row_dT["max_dT"],

        "local_global_at_dT":
            row_dT["local_global_dT"],

        "mean_R_at_dT":
            row_dT["mean_R"],

        "K_at_max_dwell":
            row_dwell["K"],

        "delta_star_dwell":
            row_dwell["delta_star_dwell"],

        "max_dwell_contrast":
            row_dwell["max_dwell_contrast"],

        "local_global_at_dwell":
            row_dwell["local_global_dwell"],

        "mean_R_at_dwell":
            row_dwell["mean_R"]
    })

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary_df = pd.DataFrame(rows)

print("\n")
print("="*70)
print("SEED ROBUSTNESS V2 SUMMARY")
print("="*70)

print(summary_df)

# ------------------------------------------------------------
# STATS
# ------------------------------------------------------------

print("\n")
print("="*70)
print("PEAK STATISTICS")
print("="*70)

print("\n--- MAX dT ---")
print(
    "Mean K:",
    summary_df["K_at_max_dT"].mean()
)
print(
    "Std K :",
    summary_df["K_at_max_dT"].std()
)

print("\n--- MAX DWELL ---")
print(
    "Mean K:",
    summary_df["K_at_max_dwell"].mean()
)
print(
    "Std K :",
    summary_df["K_at_max_dwell"].std()
)

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.hist(
    summary_df["K_at_max_dT"],
    bins=np.arange(1.2, 2.7, 0.2),
    alpha=0.6,
    label="max dT"
)

plt.hist(
    summary_df["K_at_max_dwell"],
    bins=np.arange(1.2, 2.7, 0.2),
    alpha=0.6,
    label="max dwell"
)

plt.xlabel("K peak")
plt.ylabel("Count")

plt.title(
    "Seed robustness v2: peak distributions"
)

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig(
    "stuart_landau_seed_v2_peak_histograms.png",
    dpi=300
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

summary_df.to_csv(
    "stuart_landau_seed_robustness_v2.csv",
    index=False
)

print("\nSaved files:")
print("stuart_landau_seed_robustness_v2.csv")
print("stuart_landau_seed_v2_peak_histograms.png")

print("\nDONE.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# STUART–LANDAU SEED ROBUSTNESS V4 — DWELL ONLY
# ============================================================

N = 40
dt = 0.02
steps = 5000
discard = 1000

K_values = np.array([1.3, 1.5, 1.7, 1.9, 2.1, 2.3, 2.5])
n_seeds = 30

alpha = 1.0
omega_mean = 1.0
omega_std = 0.15

n_sectors = 12

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def compute_order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def dwell_times_from_phase(theta_series, n_sectors=12):
    mean_phase = np.angle(np.mean(np.exp(1j * theta_series), axis=1))
    mean_phase = (mean_phase + 2*np.pi) % (2*np.pi)

    sectors = np.floor(mean_phase / (2*np.pi / n_sectors)).astype(int)

    dwells = []
    current = sectors[0]
    count = 1

    for s in sectors[1:]:
        if s == current:
            count += 1
        else:
            dwells.append(count)
            current = s
            count = 1

    dwells.append(count)
    return np.array(dwells), sectors

def simulate_stuart_landau(K, seed):
    rng = np.random.default_rng(seed)

    omega = rng.normal(omega_mean, omega_std, N)
    A = 1.0 + 0.1 * rng.normal(size=N)
    theta = rng.uniform(0, 2*np.pi, N)

    theta_series = []
    A_series = []
    R_series = []

    for t in range(steps):
        z = A * np.exp(1j * theta)
        mean_z = np.mean(z)
        coupling = mean_z - z

        dA = alpha * A - A**3 + K * np.real(coupling * np.exp(-1j * theta))
        dtheta = omega + K * np.imag(coupling * np.exp(-1j * theta)) / (A + 1e-8)

        A += dt * dA
        theta += dt * dtheta
        theta = np.mod(theta, 2*np.pi)

        if t >= discard:
            theta_series.append(theta.copy())
            A_series.append(A.copy())
            R_series.append(compute_order_parameter(theta))

    return np.array(theta_series), np.array(A_series), np.array(R_series)

# ============================================================
# Main loop
# ============================================================

all_rows = []
peak_rows = []

for seed in range(n_seeds):
    print("\n======================")
    print(f"SEED {seed}")
    print("======================")

    seed_rows = []

    for K in K_values:
        theta_series, A_series, R_series = simulate_stuart_landau(K, seed)

        dw_orig, sectors_orig = dwell_times_from_phase(theta_series, n_sectors=n_sectors)

        rng = np.random.default_rng(seed + int(K * 1000))
        shuffled_idx = rng.permutation(len(theta_series))
        theta_shuf = theta_series[shuffled_idx]

        dw_shuf, sectors_shuf = dwell_times_from_phase(theta_shuf, n_sectors=n_sectors)

        mean_dwell_orig = np.mean(dw_orig)
        mean_dwell_shuf = np.mean(dw_shuf)

        dwell_contrast = mean_dwell_orig / (mean_dwell_shuf + 1e-12)

        transition_orig = len(dw_orig) - 1
        transition_shuf = len(dw_shuf) - 1
        transition_contrast = transition_shuf / max(transition_orig, 1)

        row = {
            "seed": seed,
            "K": K,
            "mean_R": float(np.mean(R_series)),
            "var_R": float(np.var(R_series)),
            "mean_dwell_original": float(mean_dwell_orig),
            "mean_dwell_shuffled": float(mean_dwell_shuf),
            "dwell_contrast": float(dwell_contrast),
            "transition_count_original": int(transition_orig),
            "transition_count_shuffled": int(transition_shuf),
            "transition_contrast": float(transition_contrast),
            "max_dwell_original": float(np.max(dw_orig)),
            "max_dwell_shuffled": float(np.max(dw_shuf)),
        }

        all_rows.append(row)
        seed_rows.append(row)

    seed_df = pd.DataFrame(seed_rows)

    peak_row = seed_df.iloc[seed_df["dwell_contrast"].idxmax()]

    peak_rows.append({
        "seed": seed,
        "K_at_max_dwell_contrast": peak_row["K"],
        "max_dwell_contrast": peak_row["dwell_contrast"],
        "mean_R_at_peak": peak_row["mean_R"],
        "transition_contrast_at_peak": peak_row["transition_contrast"],
        "mean_dwell_original_at_peak": peak_row["mean_dwell_original"],
        "mean_dwell_shuffled_at_peak": peak_row["mean_dwell_shuffled"],
    })

# ============================================================
# DataFrames
# ============================================================

all_df = pd.DataFrame(all_rows)
peak_df = pd.DataFrame(peak_rows)

print("\n" + "="*70)
print("FULL DWELL ROBUSTNESS TABLE")
print("="*70)
display(all_df)

print("\n" + "="*70)
print("PEAK SUMMARY")
print("="*70)
display(peak_df)

# ============================================================
# Summary statistics
# ============================================================

print("\n" + "="*70)
print("PEAK STATISTICS")
print("="*70)

print("Mean K at peak:", peak_df["K_at_max_dwell_contrast"].mean())
print("Std K at peak :", peak_df["K_at_max_dwell_contrast"].std())
print("\nPeak K counts:")
print(peak_df["K_at_max_dwell_contrast"].value_counts().sort_index())

# ============================================================
# Aggregate over seeds
# ============================================================

agg_df = (
    all_df
    .groupby("K")
    .agg(
        mean_R_mean=("mean_R", "mean"),
        mean_R_std=("mean_R", "std"),
        dwell_contrast_mean=("dwell_contrast", "mean"),
        dwell_contrast_std=("dwell_contrast", "std"),
        transition_contrast_mean=("transition_contrast", "mean"),
        transition_contrast_std=("transition_contrast", "std"),
        mean_dwell_original_mean=("mean_dwell_original", "mean"),
        mean_dwell_shuffled_mean=("mean_dwell_shuffled", "mean"),
    )
    .reset_index()
)

print("\n" + "="*70)
print("AGGREGATED BY K")
print("="*70)
display(agg_df)

# ============================================================
# Save CSVs
# ============================================================

all_df.to_csv("stuart_landau_seed_dwell_all.csv", index=False)
peak_df.to_csv("stuart_landau_seed_dwell_peaks.csv", index=False)
agg_df.to_csv("stuart_landau_seed_dwell_aggregate.csv", index=False)

# ============================================================
# Plots
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(
    peak_df["K_at_max_dwell_contrast"],
    bins=np.arange(1.2, 2.7, 0.2),
    edgecolor="black"
)
plt.xlabel("K at maximum dwell contrast")
plt.ylabel("Count")
plt.title("Stuart–Landau seed robustness: peak K distribution")
plt.grid(True)
plt.tight_layout()
plt.savefig("stuart_landau_seed_dwell_peak_histogram.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["dwell_contrast_mean"],
    yerr=agg_df["dwell_contrast_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Dwell contrast original / shuffled")
plt.title("Stuart–Landau dwell contrast vs K across seeds")
plt.grid(True)
plt.tight_layout()
plt.savefig("stuart_landau_seed_dwell_contrast_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["mean_R_mean"],
    yerr=agg_df["mean_R_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Mean synchronization R")
plt.title("Stuart–Landau synchronization vs K across seeds")
plt.grid(True)
plt.tight_layout()
plt.savefig("stuart_landau_seed_mean_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["transition_contrast_mean"],
    yerr=agg_df["transition_contrast_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Transition contrast shuffled / original")
plt.title("Stuart–Landau transition contrast vs K across seeds")
plt.grid(True)
plt.tight_layout()
plt.savefig("stuart_landau_seed_transition_contrast_vs_K.png", dpi=300)
plt.show()

print("\nSaved files:")
print("stuart_landau_seed_dwell_all.csv")
print("stuart_landau_seed_dwell_peaks.csv")
print("stuart_landau_seed_dwell_aggregate.csv")
print("stuart_landau_seed_dwell_peak_histogram.png")
print("stuart_landau_seed_dwell_contrast_vs_K.png")
print("stuart_landau_seed_mean_R_vs_K.png")
print("stuart_landau_seed_transition_contrast_vs_K.png")
print("\nDONE.")

In [ ]:
# ============================================================
# COMPUTE NOTEBOOK 1
# Stuart–Landau amplitude+phase seed robustness
# Wide K scan with checkpoints
# NO PLOTS — compute and save only
# ============================================================

import os
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# OUTPUT FOLDER
# ============================================================

OUTDIR = Path("stuart_landau_compute_output")
OUTDIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUTDIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================

PARAMS = {
    "N": 20,
    "dt": 0.02,
    "T": 120,
    "alpha": 1.0,
    "beta": 1.0,
    "omega0": 2.0,
    "n_seeds": 30,
    "n_sectors": 12,
    "lambda_amp": 1.0,
}

# Wide K scan, dense near transition
K_values = np.concatenate([
    np.arange(0.1, 0.8, 0.2),
    np.arange(0.8, 1.6, 0.05),
    np.arange(1.6, 2.6, 0.2)
])

K_values = np.round(K_values, 3)

PARAMS["K_values"] = K_values.tolist()

with open(OUTDIR / "params.json", "w") as f:
    json.dump(PARAMS, f, indent=2)

# ============================================================
# BASIC SETTINGS
# ============================================================

N = PARAMS["N"]
dt = PARAMS["dt"]
T = PARAMS["T"]
steps = int(T / dt)

alpha = PARAMS["alpha"]
beta = PARAMS["beta"]
omega0 = PARAMS["omega0"]

n_seeds = PARAMS["n_seeds"]
n_sectors = PARAMS["n_sectors"]

# ============================================================
# FUNCTIONS
# ============================================================

def compute_order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))


def simulate_stuart_landau(K, seed):
    """
    Lightweight Stuart–Landau amplitude+phase oscillator system.

    Returns:
    - R_values
    - mean_phase_series
    - mean_amplitude_series
    - amplitude_std_series
    """

    rng = np.random.default_rng(seed)

    phases = rng.uniform(0, 2*np.pi, N)
    amplitudes = rng.normal(1.0, 0.05, N)

    R_values = []
    mean_phase_series = []
    mean_amplitude_series = []
    amplitude_std_series = []

    for step in range(steps):

        z = amplitudes * np.exp(1j * phases)
        order_parameter = np.mean(z)

        R = np.abs(order_parameter)
        psi = np.angle(order_parameter)

        R_values.append(R)
        mean_phase_series.append(psi)
        mean_amplitude_series.append(np.mean(amplitudes))
        amplitude_std_series.append(np.std(amplitudes))

        phase_diff = phases[:, None] - phases[None, :]
        coupling = np.sum(np.sin(-phase_diff), axis=1)

        dtheta = omega0 + (K / N) * coupling
        dA = alpha * amplitudes - beta * amplitudes**3

        phases += dtheta * dt
        amplitudes += dA * dt

        phases = np.mod(phases, 2*np.pi)

    return (
        np.array(R_values),
        np.array(mean_phase_series),
        np.array(mean_amplitude_series),
        np.array(amplitude_std_series)
    )


def dwell_lengths_from_states(states):
    """
    Counts residence lengths in discrete state sequence.
    """
    states = np.asarray(states)

    if len(states) == 0:
        return np.array([])

    lengths = []
    current = states[0]
    count = 1

    for s in states[1:]:
        if s == current:
            count += 1
        else:
            lengths.append(count)
            current = s
            count = 1

    lengths.append(count)
    return np.array(lengths)


def phase_sector_states(mean_phase_series, n_sectors=12):
    """
    Converts mean phase into phase-sector states.
    """
    phase = (mean_phase_series + np.pi) % (2*np.pi) - np.pi
    bins = np.linspace(-np.pi, np.pi, n_sectors + 1)
    states = np.digitize(phase, bins) - 1
    states = np.clip(states, 0, n_sectors - 1)
    return states


def compute_dwell_metrics(mean_phase_series, seed, K):
    """
    Original vs shuffled phase-sector dwell metrics.
    """
    states_orig = phase_sector_states(mean_phase_series, n_sectors=n_sectors)

    rng = np.random.default_rng(seed + int(K * 10000))
    shuffled_phase = rng.permutation(mean_phase_series)
    states_shuf = phase_sector_states(shuffled_phase, n_sectors=n_sectors)

    dwell_orig = dwell_lengths_from_states(states_orig)
    dwell_shuf = dwell_lengths_from_states(states_shuf)

    mean_dwell_orig = np.mean(dwell_orig)
    mean_dwell_shuf = np.mean(dwell_shuf)

    median_dwell_orig = np.median(dwell_orig)
    median_dwell_shuf = np.median(dwell_shuf)

    max_dwell_orig = np.max(dwell_orig)
    max_dwell_shuf = np.max(dwell_shuf)

    transition_orig = max(len(dwell_orig) - 1, 0)
    transition_shuf = max(len(dwell_shuf) - 1, 0)

    dwell_contrast_mean = mean_dwell_orig / (mean_dwell_shuf + 1e-12)
    dwell_contrast_median = median_dwell_orig / (median_dwell_shuf + 1e-12)
    dwell_contrast_max = max_dwell_orig / (max_dwell_shuf + 1e-12)

    transition_contrast = transition_shuf / max(transition_orig, 1)

    return {
        "mean_dwell_original": mean_dwell_orig,
        "mean_dwell_shuffled": mean_dwell_shuf,
        "median_dwell_original": median_dwell_orig,
        "median_dwell_shuffled": median_dwell_shuf,
        "max_dwell_original": max_dwell_orig,
        "max_dwell_shuffled": max_dwell_shuf,
        "dwell_contrast_mean": dwell_contrast_mean,
        "dwell_contrast_median": dwell_contrast_median,
        "dwell_contrast_max": dwell_contrast_max,
        "transition_count_original": transition_orig,
        "transition_count_shuffled": transition_shuf,
        "transition_contrast": transition_contrast,
    }


# ============================================================
# MAIN COMPUTE LOOP
# ============================================================

all_rows = []
start_time = time.time()

for seed in range(n_seeds):

    seed_file = CHECKPOINT_DIR / f"seed_{seed:03d}.csv"

    if seed_file.exists():
        print(f"Seed {seed} already computed. Loading checkpoint.")
        seed_df = pd.read_csv(seed_file)
        all_rows.extend(seed_df.to_dict("records"))
        continue

    print("\n" + "="*70)
    print(f"COMPUTING SEED {seed}")
    print("="*70)

    seed_rows = []

    for K in K_values:

        print(f"seed={seed}, K={K}")

        R_values, mean_phase_series, mean_amp_series, amp_std_series = simulate_stuart_landau(K, seed)

        dwell_metrics = compute_dwell_metrics(mean_phase_series, seed, K)

        row = {
            "seed": seed,
            "K": K,

            "mean_R": float(np.mean(R_values)),
            "median_R": float(np.median(R_values)),
            "var_R": float(np.var(R_values)),
            "std_R": float(np.std(R_values)),
            "min_R": float(np.min(R_values)),
            "max_R": float(np.max(R_values)),

            "mean_amplitude": float(np.mean(mean_amp_series)),
            "mean_amplitude_std": float(np.mean(amp_std_series)),
        }

        row.update(dwell_metrics)

        seed_rows.append(row)
        all_rows.append(row)

    seed_df = pd.DataFrame(seed_rows)
    seed_df.to_csv(seed_file, index=False)

    print(f"Saved checkpoint: {seed_file}")

# ============================================================
# COMBINE ALL RESULTS
# ============================================================

all_df = pd.DataFrame(all_rows)

all_csv = OUTDIR / "stuart_landau_compute_all.csv"
all_df.to_csv(all_csv, index=False)

# ============================================================
# AGGREGATE BY K
# ============================================================

agg_df = (
    all_df
    .groupby("K")
    .agg(
        mean_R_mean=("mean_R", "mean"),
        mean_R_std=("mean_R", "std"),
        var_R_mean=("var_R", "mean"),
        var_R_std=("var_R", "std"),

        dwell_contrast_mean_mean=("dwell_contrast_mean", "mean"),
        dwell_contrast_mean_std=("dwell_contrast_mean", "std"),

        dwell_contrast_median_mean=("dwell_contrast_median", "mean"),
        dwell_contrast_median_std=("dwell_contrast_median", "std"),

        dwell_contrast_max_mean=("dwell_contrast_max", "mean"),
        dwell_contrast_max_std=("dwell_contrast_max", "std"),

        transition_contrast_mean=("transition_contrast", "mean"),
        transition_contrast_std=("transition_contrast", "std"),

        transition_count_original_mean=("transition_count_original", "mean"),
        transition_count_shuffled_mean=("transition_count_shuffled", "mean"),

        mean_dwell_original_mean=("mean_dwell_original", "mean"),
        mean_dwell_shuffled_mean=("mean_dwell_shuffled", "mean"),
    )
    .reset_index()
)

agg_csv = OUTDIR / "stuart_landau_compute_aggregate_by_K.csv"
agg_df.to_csv(agg_csv, index=False)

# ============================================================
# PEAK PER SEED
# ============================================================

peak_rows = []

for seed, sub in all_df.groupby("seed"):

    idx_mean = sub["dwell_contrast_mean"].idxmax()
    idx_med = sub["dwell_contrast_median"].idxmax()
    idx_max = sub["dwell_contrast_max"].idxmax()
    idx_trans = sub["transition_contrast"].idxmax()

    r_mean = all_df.loc[idx_mean]
    r_med = all_df.loc[idx_med]
    r_max = all_df.loc[idx_max]
    r_trans = all_df.loc[idx_trans]

    peak_rows.append({
        "seed": seed,

        "K_peak_dwell_mean": r_mean["K"],
        "peak_dwell_mean": r_mean["dwell_contrast_mean"],
        "mean_R_at_peak_dwell_mean": r_mean["mean_R"],

        "K_peak_dwell_median": r_med["K"],
        "peak_dwell_median": r_med["dwell_contrast_median"],
        "mean_R_at_peak_dwell_median": r_med["mean_R"],

        "K_peak_dwell_max": r_max["K"],
        "peak_dwell_max": r_max["dwell_contrast_max"],
        "mean_R_at_peak_dwell_max": r_max["mean_R"],

        "K_peak_transition": r_trans["K"],
        "peak_transition_contrast": r_trans["transition_contrast"],
        "mean_R_at_peak_transition": r_trans["mean_R"],
    })

peak_df = pd.DataFrame(peak_rows)

peak_csv = OUTDIR / "stuart_landau_compute_seed_peaks.csv"
peak_df.to_csv(peak_csv, index=False)

# ============================================================
# LOG SUMMARY
# ============================================================

elapsed = time.time() - start_time

summary = {
    "elapsed_seconds": elapsed,
    "n_rows": len(all_df),
    "n_seeds": n_seeds,
    "n_K_values": len(K_values),
    "output_files": [
        str(all_csv),
        str(agg_csv),
        str(peak_csv),
        str(OUTDIR / "params.json"),
    ]
}

with open(OUTDIR / "compute_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("COMPUTE DONE")
print("="*70)
print(f"Rows: {len(all_df)}")
print(f"Elapsed seconds: {elapsed:.2f}")
print("\nSaved files:")
print(all_csv)
print(agg_csv)
print(peak_csv)
print(OUTDIR / "params.json")
print(OUTDIR / "compute_summary.json")
print("\nNext step: use Plot notebook to read these CSV files.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PLOT NOTEBOOK
# Stuart–Landau compute output → figures + summary tables
# ============================================================

OUTDIR = Path("stuart_landau_compute_output")

all_csv = OUTDIR / "stuart_landau_compute_all.csv"
agg_csv = OUTDIR / "stuart_landau_compute_aggregate_by_K.csv"
peak_csv = OUTDIR / "stuart_landau_compute_seed_peaks.csv"

all_df = pd.read_csv(all_csv)
agg_df = pd.read_csv(agg_csv)
peak_df = pd.read_csv(peak_csv)

print("Loaded:")
print(all_csv, all_df.shape)
print(agg_csv, agg_df.shape)
print(peak_csv, peak_df.shape)

display(agg_df)
display(peak_df)

# ============================================================
# FIGURE 1 — mean R vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["mean_R_mean"],
    yerr=agg_df["mean_R_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Mean synchronization R")
plt.title("Stuart–Landau: synchronization vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig01_sync_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 2 — variance R vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["var_R_mean"],
    yerr=agg_df["var_R_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Variance of R")
plt.title("Stuart–Landau: synchronization variance vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig02_var_R_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 3 — dwell contrast mean vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["dwell_contrast_mean_mean"],
    yerr=agg_df["dwell_contrast_mean_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Dwell contrast: original / shuffled")
plt.title("Stuart–Landau: mean dwell contrast vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig03_dwell_contrast_mean_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 4 — transition contrast vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["transition_contrast_mean"],
    yerr=agg_df["transition_contrast_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Transition contrast: shuffled / original")
plt.title("Stuart–Landau: transition contrast vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig04_transition_contrast_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 5 — mean dwell original vs shuffled
# ============================================================

plt.figure(figsize=(8,5))
plt.plot(
    agg_df["K"],
    agg_df["mean_dwell_original_mean"],
    marker="o",
    label="Original"
)
plt.plot(
    agg_df["K"],
    agg_df["mean_dwell_shuffled_mean"],
    marker="o",
    label="Shuffled"
)
plt.xlabel("Coupling K")
plt.ylabel("Mean dwell time")
plt.title("Stuart–Landau: original vs shuffled dwell time")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig05_original_vs_shuffled_dwell.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 6 — peak K distribution
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(
    peak_df["K_peak_dwell_mean"],
    bins=np.arange(0.05, 2.65, 0.1),
    edgecolor="black"
)
plt.xlabel("K at max mean dwell contrast")
plt.ylabel("Seed count")
plt.title("Stuart–Landau: distribution of dwell-contrast peak K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig06_peak_K_distribution.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 7 — peak locations: dwell vs transition
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(
    peak_df["K_peak_dwell_mean"],
    bins=np.arange(0.05, 2.65, 0.1),
    alpha=0.6,
    label="Peak dwell mean"
)
plt.hist(
    peak_df["K_peak_transition"],
    bins=np.arange(0.05, 2.65, 0.1),
    alpha=0.6,
    label="Peak transition"
)
plt.xlabel("K peak")
plt.ylabel("Seed count")
plt.title("Stuart–Landau: peak location comparison")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig07_peak_location_comparison.png", dpi=300)
plt.show()

# ============================================================
# FIGURE 8 — scatter: K peak vs mean R at peak
# ============================================================

plt.figure(figsize=(8,5))
plt.scatter(
    peak_df["K_peak_dwell_mean"],
    peak_df["mean_R_at_peak_dwell_mean"]
)
plt.xlabel("K at max dwell contrast")
plt.ylabel("Mean R at peak")
plt.title("Stuart–Landau: synchronization level at dwell peak")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig08_R_at_dwell_peak.png", dpi=300)
plt.show()

# ============================================================
# TABLE — peak statistics
# ============================================================

peak_stats = pd.DataFrame({
    "metric": [
        "K_peak_dwell_mean_mean",
        "K_peak_dwell_mean_std",
        "K_peak_transition_mean",
        "K_peak_transition_std",
        "mean_R_at_peak_dwell_mean",
        "mean_R_at_peak_dwell_std"
    ],
    "value": [
        peak_df["K_peak_dwell_mean"].mean(),
        peak_df["K_peak_dwell_mean"].std(),
        peak_df["K_peak_transition"].mean(),
        peak_df["K_peak_transition"].std(),
        peak_df["mean_R_at_peak_dwell_mean"].mean(),
        peak_df["mean_R_at_peak_dwell_mean"].std()
    ]
})

peak_stats.to_csv(OUTDIR / "plot_peak_statistics.csv", index=False)

print("\nPEAK STATISTICS")
display(peak_stats)

print("\nSaved figures and tables in:")
print(OUTDIR)
print("\nDONE.")

In [ ]:
# ============================================================
# COMPUTE NOTEBOOK 2
# Stuart–Landau amplitude+phase Δ-distance
# Seed robustness + checkpoints
# NO PLOTS
# ============================================================

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# OUTPUT
# ============================================================

OUTDIR = Path("stuart_landau_delta_compute_output")
OUTDIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUTDIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================

PARAMS = {
    "N": 30,
    "dt": 0.02,
    "T": 160,
    "discard_fraction": 0.35,
    "alpha": 1.0,
    "omega_mean": 1.0,
    "omega_std": 0.15,
    "lambda_amp": 1.0,
    "n_seeds": 30,
    "n_delta": 180,
    "delta_quantile_max": 0.995,
}

K_values = np.concatenate([
    np.arange(0.1, 0.8, 0.2),
    np.arange(0.8, 1.8, 0.05),
    np.arange(1.8, 2.6, 0.2)
])
K_values = np.round(K_values, 3)

PARAMS["K_values"] = K_values.tolist()

with open(OUTDIR / "params.json", "w") as f:
    json.dump(PARAMS, f, indent=2)

# ============================================================
# SETTINGS
# ============================================================

N = PARAMS["N"]
dt = PARAMS["dt"]
T = PARAMS["T"]
steps = int(T / dt)
discard = int(PARAMS["discard_fraction"] * steps)

alpha = PARAMS["alpha"]
omega_mean = PARAMS["omega_mean"]
omega_std = PARAMS["omega_std"]
lambda_amp = PARAMS["lambda_amp"]

n_seeds = PARAMS["n_seeds"]
n_delta = PARAMS["n_delta"]
delta_quantile_max = PARAMS["delta_quantile_max"]

# ============================================================
# FUNCTIONS
# ============================================================

def wrapped_phase_distance(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)


def compute_order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))


def simulate_stuart_landau(K, seed):
    rng = np.random.default_rng(seed)

    omega = rng.normal(omega_mean, omega_std, N)
    A = 1.0 + 0.1 * rng.normal(size=N)
    theta = rng.uniform(0, 2*np.pi, N)

    theta_series = []
    A_series = []
    R_series = []

    for t in range(steps):
        z = A * np.exp(1j * theta)
        mean_z = np.mean(z)
        coupling = mean_z - z

        dA = alpha * A - A**3 + K * np.real(coupling * np.exp(-1j * theta))
        dtheta = omega + K * np.imag(coupling * np.exp(-1j * theta)) / (A + 1e-8)

        A += dt * dA
        theta += dt * dtheta
        theta = np.mod(theta, 2*np.pi)

        if t >= discard:
            theta_series.append(theta.copy())
            A_series.append(A.copy())
            R_series.append(compute_order_parameter(theta))

    return np.array(theta_series), np.array(A_series), np.array(R_series)


def mean_pairwise_delta(theta_series, A_series):
    """
    Δ(t) = mean pairwise amplitude+phase distance at each time.
    """
    Tn = len(theta_series)
    delta_t = np.zeros(Tn)

    iu = np.triu_indices(N, k=1)

    for t in range(Tn):
        theta = theta_series[t]
        A = A_series[t]

        dtheta = wrapped_phase_distance(theta[:, None], theta[None, :])
        dA = A[:, None] - A[None, :]

        D = np.sqrt(dtheta**2 + lambda_amp * dA**2)

        delta_t[t] = np.mean(D[iu])

    return delta_t


def transition_curve(delta_t, grid):
    return np.array([np.mean(delta_t <= d) for d in grid])


def dwell_lengths(binary):
    binary = np.asarray(binary).astype(int)

    if len(binary) == 0:
        return np.array([])

    lengths = []
    current = binary[0]
    count = 1

    for x in binary[1:]:
        if x == current:
            count += 1
        else:
            lengths.append(count)
            current = x
            count = 1

    lengths.append(count)
    return np.array(lengths)


def dwell_curve(delta_t, delta_t_shuffled, grid):
    contrasts = []
    mean_orig_list = []
    mean_shuf_list = []
    trans_orig_list = []
    trans_shuf_list = []

    for d in grid:
        b_orig = delta_t <= d
        b_shuf = delta_t_shuffled <= d

        dw_orig = dwell_lengths(b_orig)
        dw_shuf = dwell_lengths(b_shuf)

        mean_orig = np.mean(dw_orig)
        mean_shuf = np.mean(dw_shuf)

        trans_orig = max(len(dw_orig) - 1, 0)
        trans_shuf = max(len(dw_shuf) - 1, 0)

        contrasts.append(mean_orig / (mean_shuf + 1e-12))
        mean_orig_list.append(mean_orig)
        mean_shuf_list.append(mean_shuf)
        trans_orig_list.append(trans_orig)
        trans_shuf_list.append(trans_shuf)

    return {
        "dwell_contrast": np.array(contrasts),
        "mean_dwell_original": np.array(mean_orig_list),
        "mean_dwell_shuffled": np.array(mean_shuf_list),
        "transition_original": np.array(trans_orig_list),
        "transition_shuffled": np.array(trans_shuf_list),
    }


def analyze_delta_series(delta_t, seed, K):
    rng = np.random.default_rng(seed + int(K * 10000))
    delta_shuf = rng.permutation(delta_t)

    dmax = np.quantile(delta_t, delta_quantile_max)
    dmin = max(np.min(delta_t), 1e-9)

    if dmax <= dmin:
        dmax = dmin + 1e-6

    grid = np.linspace(dmin, dmax, n_delta)

    T_global = transition_curve(delta_t, grid)
    T_shuf = transition_curve(delta_shuf, grid)

    dT = np.gradient(T_global, grid)

    dwell_data = dwell_curve(delta_t, delta_shuf, grid)
    dc = dwell_data["dwell_contrast"]

    # active region avoids edge artifacts
    valid = (T_global > 0.01) & (T_global < 0.99)

    if np.sum(valid) < 5:
        valid = np.ones_like(T_global, dtype=bool)

    valid_idx = np.where(valid)[0]

    idx_dT = valid_idx[np.argmax(dT[valid])]
    idx_dwell = valid_idx[np.argmax(dc[valid])]

    # median-delta diagnostic
    median_delta = np.median(delta_t)
    idx_median = np.argmin(np.abs(grid - median_delta))

    def safe_transition_contrast(i):
        return dwell_data["transition_shuffled"][i] / max(dwell_data["transition_original"][i], 1)

    return {
        "delta_min": float(np.min(delta_t)),
        "delta_max": float(np.max(delta_t)),
        "delta_mean": float(np.mean(delta_t)),
        "delta_std": float(np.std(delta_t)),
        "delta_median": float(median_delta),

        "delta_star_dT": float(grid[idx_dT]),
        "max_dT": float(dT[idx_dT]),
        "T_global_at_dT": float(T_global[idx_dT]),

        "delta_star_dwell": float(grid[idx_dwell]),
        "max_dwell_contrast": float(dc[idx_dwell]),
        "T_global_at_dwell": float(T_global[idx_dwell]),

        "mean_dwell_original_at_dwell": float(dwell_data["mean_dwell_original"][idx_dwell]),
        "mean_dwell_shuffled_at_dwell": float(dwell_data["mean_dwell_shuffled"][idx_dwell]),
        "transition_original_at_dwell": int(dwell_data["transition_original"][idx_dwell]),
        "transition_shuffled_at_dwell": int(dwell_data["transition_shuffled"][idx_dwell]),
        "transition_contrast_at_dwell": float(safe_transition_contrast(idx_dwell)),

        "dwell_contrast_at_median": float(dc[idx_median]),
        "mean_dwell_original_median": float(dwell_data["mean_dwell_original"][idx_median]),
        "mean_dwell_shuffled_median": float(dwell_data["mean_dwell_shuffled"][idx_median]),
        "transition_contrast_median": float(safe_transition_contrast(idx_median)),

        "valid_delta_points": int(np.sum(valid)),
    }


# ============================================================
# MAIN LOOP WITH CHECKPOINTS
# ============================================================

all_rows = []
start = time.time()

for seed in range(n_seeds):
    seed_file = CHECKPOINT_DIR / f"seed_{seed:03d}.csv"

    if seed_file.exists():
        print(f"Seed {seed} already computed. Loading checkpoint.")
        df_seed = pd.read_csv(seed_file)
        all_rows.extend(df_seed.to_dict("records"))
        continue

    print("\n" + "="*70)
    print(f"COMPUTING SEED {seed}")
    print("="*70)

    seed_rows = []

    for K in K_values:
        print(f"seed={seed}, K={K}")

        theta_series, A_series, R_series = simulate_stuart_landau(K, seed)
        delta_t = mean_pairwise_delta(theta_series, A_series)

        metrics = analyze_delta_series(delta_t, seed, K)

        row = {
            "seed": seed,
            "K": K,
            "mean_R": float(np.mean(R_series)),
            "var_R": float(np.var(R_series)),
            "std_R": float(np.std(R_series)),
        }

        row.update(metrics)

        seed_rows.append(row)
        all_rows.append(row)

    df_seed = pd.DataFrame(seed_rows)
    df_seed.to_csv(seed_file, index=False)
    print(f"Saved checkpoint: {seed_file}")

# ============================================================
# SAVE ALL
# ============================================================

all_df = pd.DataFrame(all_rows)
all_df.to_csv(OUTDIR / "stuart_landau_delta_all.csv", index=False)

# ============================================================
# AGGREGATE BY K
# ============================================================

agg_df = (
    all_df
    .groupby("K")
    .agg(
        mean_R_mean=("mean_R", "mean"),
        mean_R_std=("mean_R", "std"),
        var_R_mean=("var_R", "mean"),
        var_R_std=("var_R", "std"),

        delta_star_dT_mean=("delta_star_dT", "mean"),
        delta_star_dT_std=("delta_star_dT", "std"),
        max_dT_mean=("max_dT", "mean"),
        max_dT_std=("max_dT", "std"),

        delta_star_dwell_mean=("delta_star_dwell", "mean"),
        delta_star_dwell_std=("delta_star_dwell", "std"),
        max_dwell_contrast_mean=("max_dwell_contrast", "mean"),
        max_dwell_contrast_std=("max_dwell_contrast", "std"),

        dwell_contrast_at_median_mean=("dwell_contrast_at_median", "mean"),
        dwell_contrast_at_median_std=("dwell_contrast_at_median", "std"),

        transition_contrast_at_dwell_mean=("transition_contrast_at_dwell", "mean"),
        transition_contrast_at_dwell_std=("transition_contrast_at_dwell", "std"),

        mean_dwell_original_at_dwell_mean=("mean_dwell_original_at_dwell", "mean"),
        mean_dwell_shuffled_at_dwell_mean=("mean_dwell_shuffled_at_dwell", "mean"),
    )
    .reset_index()
)

agg_df.to_csv(OUTDIR / "stuart_landau_delta_aggregate_by_K.csv", index=False)

# ============================================================
# PEAKS PER SEED
# ============================================================

peak_rows = []

for seed, sub in all_df.groupby("seed"):
    idx_dT = sub["max_dT"].idxmax()
    idx_dw = sub["max_dwell_contrast"].idxmax()
    idx_med = sub["dwell_contrast_at_median"].idxmax()

    r_dT = all_df.loc[idx_dT]
    r_dw = all_df.loc[idx_dw]
    r_med = all_df.loc[idx_med]

    peak_rows.append({
        "seed": seed,

        "K_peak_dT": r_dT["K"],
        "peak_max_dT": r_dT["max_dT"],
        "mean_R_at_peak_dT": r_dT["mean_R"],
        "delta_star_at_peak_dT": r_dT["delta_star_dT"],

        "K_peak_dwell": r_dw["K"],
        "peak_dwell_contrast": r_dw["max_dwell_contrast"],
        "mean_R_at_peak_dwell": r_dw["mean_R"],
        "delta_star_at_peak_dwell": r_dw["delta_star_dwell"],

        "K_peak_median_dwell": r_med["K"],
        "peak_median_dwell_contrast": r_med["dwell_contrast_at_median"],
        "mean_R_at_peak_median_dwell": r_med["mean_R"],
    })

peak_df = pd.DataFrame(peak_rows)
peak_df.to_csv(OUTDIR / "stuart_landau_delta_seed_peaks.csv", index=False)

# ============================================================
# SUMMARY LOG
# ============================================================

summary = {
    "elapsed_seconds": time.time() - start,
    "n_rows": len(all_df),
    "n_seeds": n_seeds,
    "n_K_values": len(K_values),
    "files": [
        "stuart_landau_delta_all.csv",
        "stuart_landau_delta_aggregate_by_K.csv",
        "stuart_landau_delta_seed_peaks.csv",
        "params.json",
    ]
}

with open(OUTDIR / "compute_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("COMPUTE DONE")
print("="*70)
print(f"Rows: {len(all_df)}")
print(f"Elapsed seconds: {summary['elapsed_seconds']:.2f}")
print("\nSaved in:")
print(OUTDIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PLOT NOTEBOOK 2
# Stuart–Landau amplitude+phase Δ-distance output
# ============================================================

OUTDIR = Path("stuart_landau_delta_compute_output")

all_df = pd.read_csv(OUTDIR / "stuart_landau_delta_all.csv")
agg_df = pd.read_csv(OUTDIR / "stuart_landau_delta_aggregate_by_K.csv")
peak_df = pd.read_csv(OUTDIR / "stuart_landau_delta_seed_peaks.csv")

print("Loaded:")
print("all_df:", all_df.shape)
print("agg_df:", agg_df.shape)
print("peak_df:", peak_df.shape)

display(agg_df)
display(peak_df)

# ============================================================
# FIG 1 — Synchronization vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["mean_R_mean"],
    yerr=agg_df["mean_R_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Mean synchronization R")
plt.title("Stuart–Landau Δ-distance: synchronization vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig01_delta_sync_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIG 2 — Variance of R vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["var_R_mean"],
    yerr=agg_df["var_R_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Variance of R")
plt.title("Stuart–Landau Δ-distance: variance of synchronization vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig02_delta_var_R_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIG 3 — Δ*_dT vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["delta_star_dT_mean"],
    yerr=agg_df["delta_star_dT_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Δ* from max dT")
plt.title("Stuart–Landau: Δ*_dT vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig03_delta_star_dT_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIG 4 — max dT vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["max_dT_mean"],
    yerr=agg_df["max_dT_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("max dT/dΔ")
plt.title("Stuart–Landau: max dT vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig04_max_dT_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIG 5 — dwell contrast peak vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["max_dwell_contrast_mean"],
    yerr=agg_df["max_dwell_contrast_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Max dwell contrast")
plt.title("Stuart–Landau: amplitude+phase Δ dwell contrast vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig05_max_dwell_contrast_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIG 6 — median dwell contrast vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["dwell_contrast_at_median_mean"],
    yerr=agg_df["dwell_contrast_at_median_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Dwell contrast at median Δ")
plt.title("Stuart–Landau: dwell contrast at median Δ vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig06_median_dwell_contrast_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIG 7 — transition contrast at dwell peak vs K
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(
    agg_df["K"],
    agg_df["transition_contrast_at_dwell_mean"],
    yerr=agg_df["transition_contrast_at_dwell_std"],
    marker="o",
    capsize=4
)
plt.xlabel("Coupling K")
plt.ylabel("Transition contrast")
plt.title("Stuart–Landau: transition contrast at dwell peak vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig07_transition_contrast_at_dwell_vs_K.png", dpi=300)
plt.show()

# ============================================================
# FIG 8 — original vs shuffled dwell at dwell peak
# ============================================================

plt.figure(figsize=(8,5))
plt.plot(
    agg_df["K"],
    agg_df["mean_dwell_original_at_dwell_mean"],
    marker="o",
    label="Original"
)
plt.plot(
    agg_df["K"],
    agg_df["mean_dwell_shuffled_at_dwell_mean"],
    marker="o",
    label="Shuffled"
)
plt.xlabel("Coupling K")
plt.ylabel("Mean dwell at Δ*_dwell")
plt.title("Stuart–Landau: original vs shuffled dwell at Δ*_dwell")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig08_original_vs_shuffled_dwell_at_peak.png", dpi=300)
plt.show()

# ============================================================
# FIG 9 — peak K distribution: dT
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(
    peak_df["K_peak_dT"],
    bins=np.arange(0.05, 2.7, 0.1),
    edgecolor="black"
)
plt.xlabel("K at peak max dT")
plt.ylabel("Seed count")
plt.title("Seed distribution: K at max dT")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig09_peak_K_dT_distribution.png", dpi=300)
plt.show()

# ============================================================
# FIG 10 — peak K distribution: dwell
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(
    peak_df["K_peak_dwell"],
    bins=np.arange(0.05, 2.7, 0.1),
    edgecolor="black"
)
plt.xlabel("K at peak dwell contrast")
plt.ylabel("Seed count")
plt.title("Seed distribution: K at peak dwell contrast")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig10_peak_K_dwell_distribution.png", dpi=300)
plt.show()

# ============================================================
# FIG 11 — peak comparison dT vs dwell
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(
    peak_df["K_peak_dT"],
    bins=np.arange(0.05, 2.7, 0.1),
    alpha=0.6,
    label="max dT"
)
plt.hist(
    peak_df["K_peak_dwell"],
    bins=np.arange(0.05, 2.7, 0.1),
    alpha=0.6,
    label="max dwell contrast"
)
plt.xlabel("K peak")
plt.ylabel("Seed count")
plt.title("Stuart–Landau: peak location comparison")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig11_peak_location_comparison.png", dpi=300)
plt.show()

# ============================================================
# FIG 12 — mean R at dwell peak
# ============================================================

plt.figure(figsize=(8,5))
plt.scatter(
    peak_df["K_peak_dwell"],
    peak_df["mean_R_at_peak_dwell"]
)
plt.xlabel("K at dwell peak")
plt.ylabel("Mean R at dwell peak")
plt.title("Synchronization level at dwell peak")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig12_R_at_dwell_peak.png", dpi=300)
plt.show()

# ============================================================
# PEAK STATISTICS TABLE
# ============================================================

peak_stats = pd.DataFrame({
    "metric": [
        "K_peak_dT_mean",
        "K_peak_dT_std",
        "K_peak_dwell_mean",
        "K_peak_dwell_std",
        "K_peak_median_dwell_mean",
        "K_peak_median_dwell_std",
        "mean_R_at_peak_dwell_mean",
        "mean_R_at_peak_dwell_std",
        "peak_dwell_contrast_mean",
        "peak_dwell_contrast_std",
    ],
    "value": [
        peak_df["K_peak_dT"].mean(),
        peak_df["K_peak_dT"].std(),
        peak_df["K_peak_dwell"].mean(),
        peak_df["K_peak_dwell"].std(),
        peak_df["K_peak_median_dwell"].mean(),
        peak_df["K_peak_median_dwell"].std(),
        peak_df["mean_R_at_peak_dwell"].mean(),
        peak_df["mean_R_at_peak_dwell"].std(),
        peak_df["peak_dwell_contrast"].mean(),
        peak_df["peak_dwell_contrast"].std(),
    ]
})

peak_stats.to_csv(OUTDIR / "delta_plot_peak_statistics.csv", index=False)

print("\nPEAK STATISTICS")
display(peak_stats)

# ============================================================
# SAVE SMALL SUMMARY
# ============================================================

summary_cols = [
    "K",
    "mean_R_mean",
    "mean_R_std",
    "max_dwell_contrast_mean",
    "max_dwell_contrast_std",
    "max_dT_mean",
    "max_dT_std",
    "delta_star_dT_mean",
    "delta_star_dwell_mean",
]

summary_table = agg_df[summary_cols].copy()
summary_table.to_csv(OUTDIR / "delta_plot_main_summary_table.csv", index=False)

print("\nMAIN SUMMARY TABLE")
display(summary_table)

print("\nSaved figures and tables in:")
print(OUTDIR)
print("\nDONE.")

In [ ]:
# ============================================================
# RECOVER / COMPUTE FULL DELTA CURVES
# Stuart–Landau amplitude+phase Δ-distance
# Saves: stuart_landau_delta_curves.csv
# ============================================================

import numpy as np
import pandas as pd
import json
from pathlib import Path

OUTDIR = Path("stuart_landau_delta_compute_output")
OUTDIR.mkdir(exist_ok=True)

CURVE_DIR = OUTDIR / "curve_checkpoints"
CURVE_DIR.mkdir(exist_ok=True)

# same parameters as previous Stuart–Landau Δ compute
N = 30
dt = 0.02
T = 160
steps = int(T / dt)
discard = int(0.35 * steps)

alpha = 1.0
omega_mean = 1.0
omega_std = 0.15
lambda_amp = 1.0

n_seeds = 30
n_delta = 180
delta_quantile_max = 0.995

K_values = np.concatenate([
    np.arange(0.1, 0.8, 0.2),
    np.arange(0.8, 1.8, 0.05),
    np.arange(1.8, 2.6, 0.2)
])
K_values = np.round(K_values, 3)

def wrapped_phase_distance(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 2*np.pi - d)

def compute_order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def simulate_stuart_landau(K, seed):
    rng = np.random.default_rng(seed)

    omega = rng.normal(omega_mean, omega_std, N)
    A = 1.0 + 0.1 * rng.normal(size=N)
    theta = rng.uniform(0, 2*np.pi, N)

    theta_series = []
    A_series = []

    for t in range(steps):
        z = A * np.exp(1j * theta)
        mean_z = np.mean(z)
        coupling = mean_z - z

        dA = alpha * A - A**3 + K * np.real(coupling * np.exp(-1j * theta))
        dtheta = omega + K * np.imag(coupling * np.exp(-1j * theta)) / (A + 1e-8)

        A += dt * dA
        theta += dt * dtheta
        theta = np.mod(theta, 2*np.pi)

        if t >= discard:
            theta_series.append(theta.copy())
            A_series.append(A.copy())

    return np.array(theta_series), np.array(A_series)

def mean_pairwise_delta(theta_series, A_series):
    Tn = len(theta_series)
    delta_t = np.zeros(Tn)
    iu = np.triu_indices(N, k=1)

    for t in range(Tn):
        theta = theta_series[t]
        A = A_series[t]

        dtheta = wrapped_phase_distance(theta[:, None], theta[None, :])
        dA = A[:, None] - A[None, :]

        D = np.sqrt(dtheta**2 + lambda_amp * dA**2)
        delta_t[t] = np.mean(D[iu])

    return delta_t

def dwell_lengths(binary):
    binary = np.asarray(binary).astype(int)
    lengths = []
    current = binary[0]
    count = 1

    for x in binary[1:]:
        if x == current:
            count += 1
        else:
            lengths.append(count)
            current = x
            count = 1

    lengths.append(count)
    return np.array(lengths)

def transition_curve(delta_t, grid):
    return np.array([np.mean(delta_t <= d) for d in grid])

def compute_curves(delta_t, seed, K):
    rng = np.random.default_rng(seed + int(K * 10000))
    delta_shuf = rng.permutation(delta_t)

    dmin = max(np.min(delta_t), 1e-9)
    dmax = np.quantile(delta_t, delta_quantile_max)

    if dmax <= dmin:
        dmax = dmin + 1e-6

    grid = np.linspace(dmin, dmax, n_delta)

    T_global = transition_curve(delta_t, grid)

    dwell_contrast = []
    transition_contrast = []

    for d in grid:
        b_orig = delta_t <= d
        b_shuf = delta_shuf <= d

        dw_orig = dwell_lengths(b_orig)
        dw_shuf = dwell_lengths(b_shuf)

        mean_orig = np.mean(dw_orig)
        mean_shuf = np.mean(dw_shuf)

        trans_orig = max(len(dw_orig) - 1, 1)
        trans_shuf = max(len(dw_shuf) - 1, 0)

        dwell_contrast.append(mean_orig / (mean_shuf + 1e-12))
        transition_contrast.append(trans_shuf / trans_orig)

    return pd.DataFrame({
        "seed": seed,
        "K": K,
        "delta": grid,
        "T_global": T_global,
        "dwell_contrast": dwell_contrast,
        "transition_contrast": transition_contrast
    })

all_curve_rows = []

for seed in range(n_seeds):
    seed_file = CURVE_DIR / f"curves_seed_{seed:03d}.csv"

    if seed_file.exists():
        print(f"Seed {seed} already exists. Loading checkpoint.")
        df_seed = pd.read_csv(seed_file)
        all_curve_rows.append(df_seed)
        continue

    print("\n" + "="*70)
    print(f"COMPUTING CURVES FOR SEED {seed}")
    print("="*70)

    seed_parts = []

    for K in K_values:
        print(f"seed={seed}, K={K}")

        theta_series, A_series = simulate_stuart_landau(K, seed)
        delta_t = mean_pairwise_delta(theta_series, A_series)

        df_curve = compute_curves(delta_t, seed, K)
        seed_parts.append(df_curve)

    df_seed = pd.concat(seed_parts, ignore_index=True)
    df_seed.to_csv(seed_file, index=False)

    all_curve_rows.append(df_seed)

    print("Saved:", seed_file, df_seed.shape)

curves_df = pd.concat(all_curve_rows, ignore_index=True)

curves_path = OUTDIR / "stuart_landau_delta_curves.csv"
curves_df.to_csv(curves_path, index=False)

print("\nDONE.")
print("Saved full curves:")
print(curves_path)
print(curves_df.shape)

In [ ]:
# ============================================================
# POST-WINDOW SIGNAL PERSISTENCE — COMPUTE / ANALYSIS NOTEBOOK
# Stuart–Landau amplitude+phase Δ-distance
# Uses saved CSV curves only — NO simulation
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

OUTDIR = Path("stuart_landau_delta_compute_output")
ANALYSIS_DIR = OUTDIR / "post_window_analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)

CURVES_FILE = OUTDIR / "stuart_landau_delta_curves.csv"
SUMMARY_FILE = OUTDIR / "stuart_landau_delta_all.csv"

if not CURVES_FILE.exists():
    raise FileNotFoundError(
        f"Missing curves file: {CURVES_FILE}\n\n"
        "This analysis needs per-Δ curve data with columns:\n"
        "seed, K, delta, T_global, dwell_contrast, transition_contrast\n\n"
        "The current summary CSV is not enough because post-window persistence "
        "requires full T(Δ), dwell_contrast(Δ), and transition_contrast(Δ) curves."
    )

if not SUMMARY_FILE.exists():
    raise FileNotFoundError(
        f"Missing summary file: {SUMMARY_FILE}\n"
        "Need delta_star_dT and delta_star_dwell per seed/K."
    )

curves_df = pd.read_csv(CURVES_FILE)
summary_df = pd.read_csv(SUMMARY_FILE)

print("Loaded:")
print("curves_df:", curves_df.shape)
print("summary_df:", summary_df.shape)

required_curve_cols = {
    "seed", "K", "delta",
    "T_global",
    "dwell_contrast",
    "transition_contrast"
}

missing = required_curve_cols - set(curves_df.columns)
if missing:
    raise ValueError(f"curves_df missing columns: {missing}")

required_summary_cols = {
    "seed", "K",
    "delta_star_dT",
    "delta_star_dwell"
}

missing = required_summary_cols - set(summary_df.columns)
if missing:
    raise ValueError(f"summary_df missing columns: {missing}")

# ============================================================
# SAFE NUMERIC HELPERS
# ============================================================

def clean_numeric_array(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return x


def safe_mean(x):
    x = clean_numeric_array(x)
    return np.nan if len(x) == 0 else float(np.mean(x))


def safe_std(x):
    x = clean_numeric_array(x)
    return np.nan if len(x) == 0 else float(np.std(x))


def safe_cv(x):
    x = clean_numeric_array(x)
    if len(x) == 0:
        return np.nan
    m = np.mean(x)
    s = np.std(x)
    if not np.isfinite(m) or abs(m) < 1e-12:
        return np.nan
    return float(s / abs(m))


def safe_area(delta, y):
    delta = np.asarray(delta, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(delta) & np.isfinite(y)

    delta = delta[mask]
    y = y[mask]

    if len(delta) < 2:
        return np.nan

    order = np.argsort(delta)

    return float(np.trapz(y[order], delta[order]))


def safe_slope(delta, y):
    delta = np.asarray(delta, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(delta) & np.isfinite(y)

    delta = delta[mask]
    y = y[mask]

    if len(delta) < 3:
        return np.nan

    if np.nanstd(delta) < 1e-12:
        return np.nan

    try:
        slope, intercept = np.polyfit(delta, y, 1)
        return float(slope)
    except Exception:
        return np.nan


def rolling_variance_mean(y, window=5):
    y = pd.Series(y).replace([np.inf, -np.inf], np.nan).dropna()

    if len(y) < window:
        return np.nan

    rv = y.rolling(window=window).var().dropna()

    if len(rv) == 0:
        return np.nan

    return float(rv.mean())


def post_window_metrics(delta, signal, delta_star):
    delta = np.asarray(delta, dtype=float)
    signal = np.asarray(signal, dtype=float)

    mask = (
        np.isfinite(delta)
        & np.isfinite(signal)
        & np.isfinite(delta_star)
        & (delta > delta_star)
    )

    if np.sum(mask) < 2:
        return {
            "post_window_points": int(np.sum(mask)),
            "post_window_mean": np.nan,
            "post_window_std": np.nan,
            "post_window_cv": np.nan,
            "signal_persistence_90": np.nan,
            "signal_persistence_75": np.nan,
            "area_after_delta_star": np.nan,
            "post_window_decay": np.nan,
            "rolling_variance_mean": np.nan,
        }

    d_post = delta[mask]
    y_post = signal[mask]

    y_clean = clean_numeric_array(y_post)

    if len(y_clean) == 0:
        max_signal = np.nan
    else:
        max_signal = np.nanmax(clean_numeric_array(signal))

    if not np.isfinite(max_signal) or abs(max_signal) < 1e-12:
        persistence_90 = np.nan
        persistence_75 = np.nan
    else:
        persistence_90 = float(np.mean(y_post >= 0.90 * max_signal))
        persistence_75 = float(np.mean(y_post >= 0.75 * max_signal))

    return {
        "post_window_points": int(len(y_post)),
        "post_window_mean": safe_mean(y_post),
        "post_window_std": safe_std(y_post),
        "post_window_cv": safe_cv(y_post),
        "signal_persistence_90": persistence_90,
        "signal_persistence_75": persistence_75,
        "area_after_delta_star": safe_area(d_post, y_post),
        "post_window_decay": safe_slope(d_post, y_post),
        "rolling_variance_mean": rolling_variance_mean(y_post, window=5),
    }


# ============================================================
# MERGE DELTA STAR INFO
# ============================================================

star_cols = [
    "seed", "K",
    "delta_star_dT",
    "delta_star_dwell"
]

stars_df = summary_df[star_cols].copy()

curves_df = curves_df.merge(
    stars_df,
    on=["seed", "K"],
    how="left"
)

# ============================================================
# MAIN POST-WINDOW ANALYSIS
# ============================================================

rows = []

metric_specs = [
    {
        "signal_name": "T_global",
        "signal_col": "T_global",
        "delta_star_col": "delta_star_dT",
        "delta_star_type": "delta_star_dT"
    },
    {
        "signal_name": "dwell_contrast",
        "signal_col": "dwell_contrast",
        "delta_star_col": "delta_star_dwell",
        "delta_star_type": "delta_star_dwell"
    },
    {
        "signal_name": "transition_contrast",
        "signal_col": "transition_contrast",
        "delta_star_col": "delta_star_dwell",
        "delta_star_type": "delta_star_dwell"
    }
]

for (seed, K), sub in curves_df.groupby(["seed", "K"]):

    sub = sub.sort_values("delta")

    delta = sub["delta"].values

    for spec in metric_specs:

        signal_col = spec["signal_col"]
        delta_star_col = spec["delta_star_col"]

        if signal_col not in sub.columns:
            continue

        delta_star_values = sub[delta_star_col].dropna().unique()

        if len(delta_star_values) == 0:
            delta_star = np.nan
        else:
            delta_star = float(delta_star_values[0])

        signal = sub[signal_col].values

        metrics = post_window_metrics(
            delta=delta,
            signal=signal,
            delta_star=delta_star
        )

        row = {
            "seed": seed,
            "K": K,
            "signal_name": spec["signal_name"],
            "delta_star_type": spec["delta_star_type"],
            "delta_star": delta_star,
            "signal_max": safe_mean([np.nanmax(clean_numeric_array(signal))]) if len(clean_numeric_array(signal)) > 0 else np.nan,
            "signal_final": float(signal[-1]) if len(signal) > 0 and np.isfinite(signal[-1]) else np.nan,
        }

        row.update(metrics)
        rows.append(row)

post_df = pd.DataFrame(rows)

post_df.to_csv(
    ANALYSIS_DIR / "post_window_per_seed.csv",
    index=False
)

print("Saved per-seed post-window metrics:")
print(ANALYSIS_DIR / "post_window_per_seed.csv")

# ============================================================
# AGGREGATE BY K AND SIGNAL
# ============================================================

agg_df = (
    post_df
    .groupby(["K", "signal_name", "delta_star_type"])
    .agg(
        post_window_mean_mean=("post_window_mean", "mean"),
        post_window_mean_std=("post_window_mean", "std"),

        post_window_std_mean=("post_window_std", "mean"),
        post_window_cv_mean=("post_window_cv", "mean"),
        post_window_cv_std=("post_window_cv", "std"),

        signal_persistence_90_mean=("signal_persistence_90", "mean"),
        signal_persistence_90_std=("signal_persistence_90", "std"),

        signal_persistence_75_mean=("signal_persistence_75", "mean"),
        signal_persistence_75_std=("signal_persistence_75", "std"),

        area_after_delta_star_mean=("area_after_delta_star", "mean"),
        area_after_delta_star_std=("area_after_delta_star", "std"),

        post_window_decay_mean=("post_window_decay", "mean"),
        post_window_decay_std=("post_window_decay", "std"),

        rolling_variance_mean=("rolling_variance_mean", "mean"),
        post_window_points_mean=("post_window_points", "mean"),
    )
    .reset_index()
)

agg_df.to_csv(
    ANALYSIS_DIR / "post_window_summary_by_K.csv",
    index=False
)

print("Saved summary:")
print(ANALYSIS_DIR / "post_window_summary_by_K.csv")

display(post_df.head())
display(agg_df.head())

print("\nDONE.")

In [ ]:
# ============================================================
# POST-WINDOW SIGNAL PERSISTENCE — PLOT NOTEBOOK
# Reads summary CSV only
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTDIR = Path("stuart_landau_delta_compute_output")
ANALYSIS_DIR = OUTDIR / "post_window_analysis"

SUMMARY_FILE = ANALYSIS_DIR / "post_window_summary_by_K.csv"
PER_SEED_FILE = ANALYSIS_DIR / "post_window_per_seed.csv"

summary_df = pd.read_csv(SUMMARY_FILE)
post_df = pd.read_csv(PER_SEED_FILE)

print("Loaded:")
print("summary_df:", summary_df.shape)
print("post_df:", post_df.shape)

display(summary_df.head())
display(post_df.head())

# ============================================================
# PLOT HELPERS
# ============================================================

def plot_metric(metric_mean, metric_std, ylabel, title, filename):

    plt.figure(figsize=(8, 5))

    for signal_name, sub in summary_df.groupby("signal_name"):

        sub = sub.sort_values("K")

        y = sub[metric_mean].values
        yerr = sub[metric_std].values if metric_std in sub.columns else None

        plt.errorbar(
            sub["K"],
            y,
            yerr=yerr,
            marker="o",
            capsize=4,
            label=signal_name
        )

    plt.xlabel("Coupling K")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    out = ANALYSIS_DIR / filename
    plt.savefig(out, dpi=300)
    plt.show()

    print("Saved:", out)


# ============================================================
# FIGURE 1 — post-window mean vs K
# ============================================================

plot_metric(
    metric_mean="post_window_mean_mean",
    metric_std="post_window_mean_std",
    ylabel="Post-window mean",
    title="Post-window signal mean vs K",
    filename="fig_post_window_mean_vs_K.png"
)

# ============================================================
# FIGURE 2 — persistence 90 vs K
# ============================================================

plot_metric(
    metric_mean="signal_persistence_90_mean",
    metric_std="signal_persistence_90_std",
    ylabel="Persistence ≥ 90% max",
    title="Post-window signal persistence 90% vs K",
    filename="fig_persistence_90_vs_K.png"
)

# ============================================================
# FIGURE 3 — persistence 75 vs K
# ============================================================

plot_metric(
    metric_mean="signal_persistence_75_mean",
    metric_std="signal_persistence_75_std",
    ylabel="Persistence ≥ 75% max",
    title="Post-window signal persistence 75% vs K",
    filename="fig_persistence_75_vs_K.png"
)

# ============================================================
# FIGURE 4 — area after Δ* vs K
# ============================================================

plot_metric(
    metric_mean="area_after_delta_star_mean",
    metric_std="area_after_delta_star_std",
    ylabel="Area after Δ*",
    title="Area under signal after Δ* vs K",
    filename="fig_area_after_delta_star_vs_K.png"
)

# ============================================================
# FIGURE 5 — decay slope vs K
# ============================================================

plot_metric(
    metric_mean="post_window_decay_mean",
    metric_std="post_window_decay_std",
    ylabel="Post-window decay slope",
    title="Post-window decay / stabilization slope vs K",
    filename="fig_post_window_decay_vs_K.png"
)

# ============================================================
# FIGURE 6 — coefficient of variation vs K
# ============================================================

plot_metric(
    metric_mean="post_window_cv_mean",
    metric_std="post_window_cv_std",
    ylabel="Post-window CV",
    title="Post-window coefficient of variation vs K",
    filename="fig_post_window_cv_vs_K.png"
)

# ============================================================
# OPTIONAL — separate plots per signal
# ============================================================

for signal_name, sub in summary_df.groupby("signal_name"):

    sub = sub.sort_values("K")

    plt.figure(figsize=(8,5))

    plt.plot(
        sub["K"],
        sub["post_window_mean_mean"],
        marker="o",
        label="post-window mean"
    )

    plt.plot(
        sub["K"],
        sub["signal_persistence_90_mean"],
        marker="o",
        label="persistence 90"
    )

    plt.plot(
        sub["K"],
        sub["signal_persistence_75_mean"],
        marker="o",
        label="persistence 75"
    )

    plt.xlabel("Coupling K")
    plt.ylabel("Metric value")
    plt.title(f"Post-window persistence summary — {signal_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    fname = f"fig_post_window_summary_{signal_name}.png".replace("/", "_")
    out = ANALYSIS_DIR / fname
    plt.savefig(out, dpi=300)
    plt.show()

    print("Saved:", out)

# ============================================================
# SAVE CLEAN MAIN TABLE
# ============================================================

main_cols = [
    "K",
    "signal_name",
    "post_window_mean_mean",
    "post_window_cv_mean",
    "signal_persistence_90_mean",
    "signal_persistence_75_mean",
    "area_after_delta_star_mean",
    "post_window_decay_mean",
    "rolling_variance_mean",
]

main_table = summary_df[main_cols].copy()

main_table.to_csv(
    ANALYSIS_DIR / "post_window_main_table.csv",
    index=False
)

print("\nMAIN TABLE")
display(main_table)

print("\nSaved:")
print(ANALYSIS_DIR / "post_window_main_table.csv")
print("\nDONE.")

In [ ]:
# ============================================================
# SAVE FULL DELTA CURVES
# ============================================================

required_cols = [
    "seed",
    "K",
    "delta",
    "T_global",
    "dwell_contrast",
    "transition_contrast"
]

missing = [c for c in required_cols if c not in all_df.columns]

if len(missing) > 0:
    print("Missing columns:")
    print(missing)
else:

    curves_df = all_df[required_cols].copy()

    curves_path = OUTDIR / "stuart_landau_delta_curves.csv"

    curves_df.to_csv(curves_path, index=False)

    print("\nSaved full delta curves:")
    print(curves_path)
    print(curves_df.shape)

In [ ]:
# ============================================================
# RECOVERY / MEMORY TEST v1
# Stuart–Landau amplitude+phase system
# Phase perturbation after stabilization
# COMPUTE ONLY — no plots
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import time
import json

# ============================================================
# OUTPUT
# ============================================================

OUTDIR = Path("stuart_landau_recovery_memory_output")
OUTDIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUTDIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================

PARAMS = {
    "N": 30,
    "dt": 0.02,
    "T_pre": 160,
    "T_post": 120,
    "discard_fraction": 0.35,
    "alpha": 1.0,
    "omega_mean": 1.0,
    "omega_std": 0.15,
    "n_seeds": 30,
    "kick_strength": 1.0,
    "recovery_threshold": 0.95,
}

K_values = np.concatenate([
    np.arange(0.1, 0.8, 0.2),
    np.arange(0.8, 1.8, 0.05),
    np.arange(1.8, 2.6, 0.2)
])
K_values = np.round(K_values, 3)

PARAMS["K_values"] = K_values.tolist()

with open(OUTDIR / "params.json", "w") as f:
    json.dump(PARAMS, f, indent=2)

# ============================================================
# SETTINGS
# ============================================================

N = PARAMS["N"]
dt = PARAMS["dt"]
T_pre = PARAMS["T_pre"]
T_post = PARAMS["T_post"]

steps_pre = int(T_pre / dt)
steps_post = int(T_post / dt)

discard = int(PARAMS["discard_fraction"] * steps_pre)

alpha = PARAMS["alpha"]
omega_mean = PARAMS["omega_mean"]
omega_std = PARAMS["omega_std"]

n_seeds = PARAMS["n_seeds"]
kick_strength = PARAMS["kick_strength"]
recovery_threshold = PARAMS["recovery_threshold"]

# ============================================================
# FUNCTIONS
# ============================================================

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))


def step_stuart_landau(A, theta, omega, K):
    z = A * np.exp(1j * theta)
    mean_z = np.mean(z)
    coupling = mean_z - z

    dA = alpha * A - A**3 + K * np.real(coupling * np.exp(-1j * theta))
    dtheta = omega + K * np.imag(coupling * np.exp(-1j * theta)) / (A + 1e-8)

    A = A + dt * dA
    theta = theta + dt * dtheta
    theta = np.mod(theta, 2*np.pi)

    return A, theta


def simulate_until_perturbation(K, seed):
    rng = np.random.default_rng(seed)

    omega = rng.normal(omega_mean, omega_std, N)
    A = 1.0 + 0.1 * rng.normal(size=N)
    theta = rng.uniform(0, 2*np.pi, N)

    R_series = []
    A_series = []
    theta_series = []

    for t in range(steps_pre):
        A, theta = step_stuart_landau(A, theta, omega, K)

        if t >= discard:
            R_series.append(order_parameter(theta))
            A_series.append(A.copy())
            theta_series.append(theta.copy())

    return (
        np.array(R_series),
        np.array(A_series),
        np.array(theta_series),
        A.copy(),
        theta.copy(),
        omega.copy()
    )


def apply_phase_kick(theta, seed, K):
    rng = np.random.default_rng(seed + int(K * 10000) + 999)
    kick = rng.uniform(-kick_strength, kick_strength, size=N)
    theta_kicked = np.mod(theta + kick, 2*np.pi)
    return theta_kicked


def simulate_after_kick(A, theta, omega, K):
    R_post = []

    for t in range(steps_post):
        A, theta = step_stuart_landau(A, theta, omega, K)
        R_post.append(order_parameter(theta))

    return np.array(R_post), A.copy(), theta.copy()


def compute_recovery_metrics(R_pre, R_post):
    if len(R_pre) == 0 or len(R_post) == 0:
        return {
            "R_before": np.nan,
            "R_min_after": np.nan,
            "R_final": np.nan,
            "synchronization_drop": np.nan,
            "memory_ratio": np.nan,
            "recovery_time": np.nan,
            "recovered": False,
        }

    R_before = float(np.mean(R_pre[-200:]))
    R_min_after = float(np.min(R_post))
    R_final = float(np.mean(R_post[-200:]))

    synchronization_drop = R_before - R_min_after
    memory_ratio = R_final / (R_before + 1e-12)

    threshold = recovery_threshold * R_before
    recovery_indices = np.where(R_post >= threshold)[0]

    if len(recovery_indices) == 0:
        recovery_time = np.nan
        recovered = False
    else:
        recovery_time = float(recovery_indices[0] * dt)
        recovered = True

    return {
        "R_before": R_before,
        "R_min_after": R_min_after,
        "R_final": R_final,
        "synchronization_drop": synchronization_drop,
        "memory_ratio": memory_ratio,
        "recovery_time": recovery_time,
        "recovered": recovered,
    }


# ============================================================
# MAIN LOOP
# ============================================================

all_rows = []
start = time.time()

for seed in range(n_seeds):

    seed_file = CHECKPOINT_DIR / f"recovery_seed_{seed:03d}.csv"

    if seed_file.exists():
        print(f"Seed {seed} already computed. Loading checkpoint.")
        df_seed = pd.read_csv(seed_file)
        all_rows.extend(df_seed.to_dict("records"))
        continue

    print("\n" + "="*70)
    print(f"RECOVERY TEST — SEED {seed}")
    print("="*70)

    seed_rows = []

    for K in K_values:
        print(f"seed={seed}, K={K}")

        R_pre, A_series, theta_series, A_last, theta_last, omega = simulate_until_perturbation(K, seed)

        theta_kicked = apply_phase_kick(theta_last, seed, K)

        R_post, A_final, theta_final = simulate_after_kick(
            A_last.copy(),
            theta_kicked.copy(),
            omega.copy(),
            K
        )

        metrics = compute_recovery_metrics(R_pre, R_post)

        row = {
            "seed": seed,
            "K": K,
            "kick_strength": kick_strength,
            "mean_R_pre": float(np.mean(R_pre)),
            "std_R_pre": float(np.std(R_pre)),
            "mean_R_post": float(np.mean(R_post)),
            "std_R_post": float(np.std(R_post)),
        }

        row.update(metrics)

        seed_rows.append(row)
        all_rows.append(row)

    df_seed = pd.DataFrame(seed_rows)
    df_seed.to_csv(seed_file, index=False)

    print("Saved:", seed_file)

# ============================================================
# SAVE ALL + AGGREGATE
# ============================================================

all_df = pd.DataFrame(all_rows)
all_df.to_csv(OUTDIR / "recovery_memory_all.csv", index=False)

agg_df = (
    all_df
    .groupby("K")
    .agg(
        R_before_mean=("R_before", "mean"),
        R_before_std=("R_before", "std"),

        R_final_mean=("R_final", "mean"),
        R_final_std=("R_final", "std"),

        synchronization_drop_mean=("synchronization_drop", "mean"),
        synchronization_drop_std=("synchronization_drop", "std"),

        memory_ratio_mean=("memory_ratio", "mean"),
        memory_ratio_std=("memory_ratio", "std"),

        recovery_time_mean=("recovery_time", "mean"),
        recovery_time_std=("recovery_time", "std"),

        recovered_rate=("recovered", "mean"),
    )
    .reset_index()
)

agg_df.to_csv(OUTDIR / "recovery_memory_aggregate_by_K.csv", index=False)

summary = {
    "elapsed_seconds": time.time() - start,
    "rows": len(all_df),
    "n_seeds": n_seeds,
    "n_K_values": len(K_values),
    "files": [
        "recovery_memory_all.csv",
        "recovery_memory_aggregate_by_K.csv",
        "params.json",
    ],
}

with open(OUTDIR / "recovery_memory_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("RECOVERY / MEMORY COMPUTE DONE")
print("="*70)
print("Rows:", len(all_df))
print("Saved in:", OUTDIR)

In [ ]:
import pandas as pd
import json
from pathlib import Path

OUTDIR = Path("stuart_landau_recovery_memory_output")
CHECKPOINT_DIR = OUTDIR / "checkpoints"

files = sorted(CHECKPOINT_DIR.glob("recovery_seed_*.csv"))

print("Found checkpoint files:", len(files))

if len(files) == 0:
    raise FileNotFoundError("No recovery checkpoint files found.")

dfs = []
for f in files:
    df = pd.read_csv(f)
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

all_df.to_csv(OUTDIR / "recovery_memory_all.csv", index=False)

agg_df = (
    all_df
    .groupby("K")
    .agg(
        R_before_mean=("R_before", "mean"),
        R_before_std=("R_before", "std"),

        R_final_mean=("R_final", "mean"),
        R_final_std=("R_final", "std"),

        synchronization_drop_mean=("synchronization_drop", "mean"),
        synchronization_drop_std=("synchronization_drop", "std"),

        memory_ratio_mean=("memory_ratio", "mean"),
        memory_ratio_std=("memory_ratio", "std"),

        recovery_time_mean=("recovery_time", "mean"),
        recovery_time_std=("recovery_time", "std"),

        recovered_rate=("recovered", "mean"),
    )
    .reset_index()
)

agg_df.to_csv(OUTDIR / "recovery_memory_aggregate_by_K.csv", index=False)

summary = {
    "n_checkpoint_files": len(files),
    "n_rows": len(all_df),
    "files_created": [
        "recovery_memory_all.csv",
        "recovery_memory_aggregate_by_K.csv"
    ]
}

with open(OUTDIR / "recovery_memory_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print(OUTDIR / "recovery_memory_all.csv")
print(OUTDIR / "recovery_memory_aggregate_by_K.csv")

display(agg_df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTDIR = Path("stuart_landau_recovery_memory_output")

all_df = pd.read_csv(OUTDIR / "recovery_memory_all.csv")
agg_df = pd.read_csv(OUTDIR / "recovery_memory_aggregate_by_K.csv")

print("Loaded:")
print("all_df:", all_df.shape)
print("agg_df:", agg_df.shape)

display(agg_df)

# 1. R before/final
plt.figure(figsize=(8,5))
plt.errorbar(agg_df["K"], agg_df["R_before_mean"], yerr=agg_df["R_before_std"], marker="o", capsize=4, label="R before")
plt.errorbar(agg_df["K"], agg_df["R_final_mean"], yerr=agg_df["R_final_std"], marker="o", capsize=4, label="R final")
plt.xlabel("Coupling K")
plt.ylabel("Synchronization R")
plt.title("Recovery test: R before vs R final")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig01_R_before_final_vs_K.png", dpi=300)
plt.show()

# 2. synchronization drop
plt.figure(figsize=(8,5))
plt.errorbar(agg_df["K"], agg_df["synchronization_drop_mean"], yerr=agg_df["synchronization_drop_std"], marker="o", capsize=4)
plt.xlabel("Coupling K")
plt.ylabel("Synchronization drop")
plt.title("Recovery test: synchronization drop after kick")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig02_sync_drop_vs_K.png", dpi=300)
plt.show()

# 3. memory ratio
plt.figure(figsize=(8,5))
plt.errorbar(agg_df["K"], agg_df["memory_ratio_mean"], yerr=agg_df["memory_ratio_std"], marker="o", capsize=4)
plt.axhline(1.0, linestyle="--")
plt.xlabel("Coupling K")
plt.ylabel("Memory ratio R_final / R_before")
plt.title("Recovery test: memory ratio vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig03_memory_ratio_vs_K.png", dpi=300)
plt.show()

# 4. recovery time
plt.figure(figsize=(8,5))
plt.errorbar(agg_df["K"], agg_df["recovery_time_mean"], yerr=agg_df["recovery_time_std"], marker="o", capsize=4)
plt.xlabel("Coupling K")
plt.ylabel("Recovery time")
plt.title("Recovery test: recovery time vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig04_recovery_time_vs_K.png", dpi=300)
plt.show()

# 5. recovered rate
plt.figure(figsize=(8,5))
plt.plot(agg_df["K"], agg_df["recovered_rate"], marker="o")
plt.xlabel("Coupling K")
plt.ylabel("Recovered rate")
plt.title("Recovery test: fraction of runs recovered")
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTDIR / "fig05_recovered_rate_vs_K.png", dpi=300)
plt.show()

# Save clean table
main_cols = [
    "K",
    "R_before_mean",
    "R_final_mean",
    "synchronization_drop_mean",
    "memory_ratio_mean",
    "recovery_time_mean",
    "recovered_rate"
]

main_table = agg_df[main_cols].copy()
main_table.to_csv(OUTDIR / "recovery_memory_main_table.csv", index=False)

print("\nMAIN TABLE")
display(main_table)

print("\nSaved figures and table in:")
print(OUTDIR)
print("\nDONE.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import time

# ============================================================
# PRE vs POST PERTURBATION RECOVERY TEST
# Stuart–Landau amplitude+phase
# COMPUTE ONLY
# ============================================================

OUTDIR = Path("stuart_landau_pre_post_recovery_output")
OUTDIR.mkdir(exist_ok=True)

CHECKPOINT_DIR = OUTDIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

PARAMS = {
    "N": 30,
    "dt": 0.02,
    "T_total": 220,
    "alpha": 1.0,
    "omega_mean": 1.0,
    "omega_std": 0.15,
    "n_seeds": 30,
    "kick_strength": 1.0,
    "recovery_threshold": 0.95,
    "pre_perturb_time": 30,
    "post_perturb_time": 130,
    "recovery_window": 80
}

K_values = np.concatenate([
    np.arange(0.1, 0.8, 0.2),
    np.arange(0.8, 1.8, 0.05),
    np.arange(1.8, 2.6, 0.2)
])
K_values = np.round(K_values, 3)
PARAMS["K_values"] = K_values.tolist()

with open(OUTDIR / "params.json", "w") as f:
    json.dump(PARAMS, f, indent=2)

N = PARAMS["N"]
dt = PARAMS["dt"]
T_total = PARAMS["T_total"]
steps_total = int(T_total / dt)

alpha = PARAMS["alpha"]
omega_mean = PARAMS["omega_mean"]
omega_std = PARAMS["omega_std"]

n_seeds = PARAMS["n_seeds"]
kick_strength = PARAMS["kick_strength"]
recovery_threshold = PARAMS["recovery_threshold"]

pre_step = int(PARAMS["pre_perturb_time"] / dt)
post_step = int(PARAMS["post_perturb_time"] / dt)
recovery_steps = int(PARAMS["recovery_window"] / dt)

# ============================================================
# FUNCTIONS
# ============================================================

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def step_stuart_landau(A, theta, omega, K):
    z = A * np.exp(1j * theta)
    mean_z = np.mean(z)
    coupling = mean_z - z

    dA = alpha * A - A**3 + K * np.real(coupling * np.exp(-1j * theta))
    dtheta = omega + K * np.imag(coupling * np.exp(-1j * theta)) / (A + 1e-8)

    A = A + dt * dA
    theta = theta + dt * dtheta
    theta = np.mod(theta, 2*np.pi)

    return A, theta

def initialize(seed):
    rng = np.random.default_rng(seed)
    omega = rng.normal(omega_mean, omega_std, N)
    A = 1.0 + 0.1 * rng.normal(size=N)
    theta = rng.uniform(0, 2*np.pi, N)
    return A, theta, omega

def apply_phase_kick(theta, seed, K, label):
    salt = 111 if label == "pre" else 999
    rng = np.random.default_rng(seed + int(K * 10000) + salt)
    kick = rng.uniform(-kick_strength, kick_strength, size=N)
    return np.mod(theta + kick, 2*np.pi)

def run_until_state(K, seed, target_step):
    A, theta, omega = initialize(seed)
    R_history = []

    for t in range(target_step):
        A, theta = step_stuart_landau(A, theta, omega, K)
        R_history.append(order_parameter(theta))

    return A.copy(), theta.copy(), omega.copy(), np.array(R_history)

def run_recovery(A, theta, omega, K):
    R_post = []

    for t in range(recovery_steps):
        A, theta = step_stuart_landau(A, theta, omega, K)
        R_post.append(order_parameter(theta))

    return np.array(R_post)

def compute_metrics(R_before_history, R_after):
    if len(R_before_history) == 0 or len(R_after) == 0:
        return {
            "R_before": np.nan,
            "R_min_after": np.nan,
            "R_final": np.nan,
            "synchronization_drop": np.nan,
            "memory_ratio": np.nan,
            "recovery_time": np.nan,
            "recovered": False
        }

    tail = min(200, len(R_before_history))
    R_before = float(np.mean(R_before_history[-tail:]))

    R_min_after = float(np.min(R_after))
    R_final = float(np.mean(R_after[-tail:]))

    synchronization_drop = R_before - R_min_after
    memory_ratio = R_final / (R_before + 1e-12)

    threshold = recovery_threshold * R_before
    idx = np.where(R_after >= threshold)[0]

    if len(idx) == 0:
        recovery_time = np.nan
        recovered = False
    else:
        recovery_time = float(idx[0] * dt)
        recovered = True

    return {
        "R_before": R_before,
        "R_min_after": R_min_after,
        "R_final": R_final,
        "synchronization_drop": synchronization_drop,
        "memory_ratio": memory_ratio,
        "recovery_time": recovery_time,
        "recovered": recovered
    }

# ============================================================
# MAIN LOOP
# ============================================================

all_rows = []
start = time.time()

for seed in range(n_seeds):
    seed_file = CHECKPOINT_DIR / f"pre_post_seed_{seed:03d}.csv"

    if seed_file.exists():
        print(f"Seed {seed} already computed. Loading checkpoint.")
        df_seed = pd.read_csv(seed_file)
        all_rows.extend(df_seed.to_dict("records"))
        continue

    print("\n" + "="*70)
    print(f"PRE/POST RECOVERY — SEED {seed}")
    print("="*70)

    seed_rows = []

    for K in K_values:
        print(f"seed={seed}, K={K}")

        for phase_label, target_step in [("pre", pre_step), ("post", post_step)]:
            A, theta, omega, R_before_hist = run_until_state(K, seed, target_step)

            theta_kicked = apply_phase_kick(theta, seed, K, phase_label)

            R_after = run_recovery(A.copy(), theta_kicked.copy(), omega.copy(), K)

            metrics = compute_metrics(R_before_hist, R_after)

            row = {
                "seed": seed,
                "K": K,
                "perturbation_phase": phase_label,
                "target_step": target_step,
                "target_time": target_step * dt,
                "kick_strength": kick_strength,
            }

            row.update(metrics)
            seed_rows.append(row)
            all_rows.append(row)

    df_seed = pd.DataFrame(seed_rows)
    df_seed.to_csv(seed_file, index=False)
    print("Saved:", seed_file)

# ============================================================
# SAVE ALL
# ============================================================

all_df = pd.DataFrame(all_rows)
all_df.to_csv(OUTDIR / "pre_post_recovery_all.csv", index=False)

agg_df = (
    all_df
    .groupby(["K", "perturbation_phase"])
    .agg(
        R_before_mean=("R_before", "mean"),
        R_before_std=("R_before", "std"),
        R_final_mean=("R_final", "mean"),
        R_final_std=("R_final", "std"),
        synchronization_drop_mean=("synchronization_drop", "mean"),
        synchronization_drop_std=("synchronization_drop", "std"),
        memory_ratio_mean=("memory_ratio", "mean"),
        memory_ratio_std=("memory_ratio", "std"),
        recovery_time_mean=("recovery_time", "mean"),
        recovery_time_std=("recovery_time", "std"),
        recovered_rate=("recovered", "mean"),
    )
    .reset_index()
)

agg_df.to_csv(OUTDIR / "pre_post_recovery_aggregate.csv", index=False)

# ============================================================
# COMPARISON: post/pre ratios
# ============================================================

pre = agg_df[agg_df["perturbation_phase"] == "pre"].set_index("K")
post = agg_df[agg_df["perturbation_phase"] == "post"].set_index("K")

comp_rows = []

for K in sorted(set(pre.index) & set(post.index)):
    comp_rows.append({
        "K": K,
        "recovery_time_pre": pre.loc[K, "recovery_time_mean"],
        "recovery_time_post": post.loc[K, "recovery_time_mean"],
        "recovery_time_ratio_post_over_pre": post.loc[K, "recovery_time_mean"] / (pre.loc[K, "recovery_time_mean"] + 1e-12),

        "memory_ratio_pre": pre.loc[K, "memory_ratio_mean"],
        "memory_ratio_post": post.loc[K, "memory_ratio_mean"],
        "memory_ratio_difference_post_minus_pre": post.loc[K, "memory_ratio_mean"] - pre.loc[K, "memory_ratio_mean"],

        "sync_drop_pre": pre.loc[K, "synchronization_drop_mean"],
        "sync_drop_post": post.loc[K, "synchronization_drop_mean"],
        "sync_drop_difference_post_minus_pre": post.loc[K, "synchronization_drop_mean"] - pre.loc[K, "synchronization_drop_mean"],

        "recovered_rate_pre": pre.loc[K, "recovered_rate"],
        "recovered_rate_post": post.loc[K, "recovered_rate"],
    })

comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(OUTDIR / "pre_post_recovery_comparison.csv", index=False)

summary = {
    "elapsed_seconds": time.time() - start,
    "rows": len(all_df),
    "n_seeds": n_seeds,
    "n_K_values": len(K_values),
    "files": [
        "pre_post_recovery_all.csv",
        "pre_post_recovery_aggregate.csv",
        "pre_post_recovery_comparison.csv"
    ]
}

with open(OUTDIR / "pre_post_recovery_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("PRE vs POST RECOVERY COMPUTE DONE")
print("="*70)
print("Rows:", len(all_df))
print("Saved in:", OUTDIR)
display(agg_df)
display(comp_df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# PRE vs POST RECOVERY — PLOTS
# ============================================================

OUTDIR = Path("stuart_landau_pre_post_recovery_output")

all_df = pd.read_csv(OUTDIR / "pre_post_recovery_all.csv")
agg_df = pd.read_csv(OUTDIR / "pre_post_recovery_aggregate.csv")
comp_df = pd.read_csv(OUTDIR / "pre_post_recovery_comparison.csv")

print("Loaded:")
print("all_df:", all_df.shape)
print("agg_df:", agg_df.shape)
print("comp_df:", comp_df.shape)

# ============================================================
# COLORS
# ============================================================

phase_colors = {
    "pre": "tab:blue",
    "post": "tab:orange"
}

# ============================================================
# RECOVERY TIME
# ============================================================

plt.figure(figsize=(10,6))

for phase in ["pre", "post"]:
    sub = agg_df[agg_df["perturbation_phase"] == phase]

    plt.errorbar(
        sub["K"],
        sub["recovery_time_mean"],
        yerr=sub["recovery_time_std"],
        marker='o',
        capsize=3,
        label=phase,
        color=phase_colors[phase]
    )

plt.xlabel("Coupling K")
plt.ylabel("Recovery time")
plt.title("Pre vs Post perturbation recovery time")
plt.grid(True)
plt.legend()

fname = OUTDIR / "fig_pre_post_recovery_time.png"
plt.savefig(fname, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", fname)

# ============================================================
# MEMORY RATIO
# ============================================================

plt.figure(figsize=(10,6))

for phase in ["pre", "post"]:
    sub = agg_df[agg_df["perturbation_phase"] == phase]

    plt.errorbar(
        sub["K"],
        sub["memory_ratio_mean"],
        yerr=sub["memory_ratio_std"],
        marker='o',
        capsize=3,
        label=phase,
        color=phase_colors[phase]
    )

plt.axhline(1.0, linestyle='--', color='black')

plt.xlabel("Coupling K")
plt.ylabel("Memory ratio")
plt.title("Pre vs Post memory ratio")
plt.grid(True)
plt.legend()

fname = OUTDIR / "fig_pre_post_memory_ratio.png"
plt.savefig(fname, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", fname)

# ============================================================
# SYNCHRONIZATION DROP
# ============================================================

plt.figure(figsize=(10,6))

for phase in ["pre", "post"]:
    sub = agg_df[agg_df["perturbation_phase"] == phase]

    plt.errorbar(
        sub["K"],
        sub["synchronization_drop_mean"],
        yerr=sub["synchronization_drop_std"],
        marker='o',
        capsize=3,
        label=phase,
        color=phase_colors[phase]
    )

plt.xlabel("Coupling K")
plt.ylabel("Synchronization drop")
plt.title("Pre vs Post synchronization drop")
plt.grid(True)
plt.legend()

fname = OUTDIR / "fig_pre_post_sync_drop.png"
plt.savefig(fname, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", fname)

# ============================================================
# POST/PRE RECOVERY RATIO
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(
    comp_df["K"],
    comp_df["recovery_time_ratio_post_over_pre"],
    marker='o'
)

plt.axhline(1.0, linestyle='--', color='black')

plt.xlabel("Coupling K")
plt.ylabel("Post / Pre recovery time")
plt.title("Recovery speed comparison")
plt.grid(True)

fname = OUTDIR / "fig_recovery_ratio_post_over_pre.png"
plt.savefig(fname, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", fname)

# ============================================================
# MEMORY DIFFERENCE
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(
    comp_df["K"],
    comp_df["memory_ratio_difference_post_minus_pre"],
    marker='o'
)

plt.axhline(0.0, linestyle='--', color='black')

plt.xlabel("Coupling K")
plt.ylabel("Post - Pre memory ratio")
plt.title("Memory improvement after stabilization")
plt.grid(True)

fname = OUTDIR / "fig_memory_difference_post_minus_pre.png"
plt.savefig(fname, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", fname)

# ============================================================
# RECOVERED RATE
# ============================================================

plt.figure(figsize=(10,6))

for phase in ["pre", "post"]:
    sub = agg_df[agg_df["perturbation_phase"] == phase]

    plt.plot(
        sub["K"],
        sub["recovered_rate"],
        marker='o',
        label=phase,
        color=phase_colors[phase]
    )

plt.xlabel("Coupling K")
plt.ylabel("Recovered fraction")
plt.title("Recovery success rate")
plt.grid(True)
plt.legend()

fname = OUTDIR / "fig_recovered_rate.png"
plt.savefig(fname, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", fname)

# ============================================================
# DISPLAY TABLE
# ============================================================

main_table = comp_df[[
    "K",
    "recovery_time_pre",
    "recovery_time_post",
    "recovery_time_ratio_post_over_pre",
    "memory_ratio_pre",
    "memory_ratio_post",
    "memory_ratio_difference_post_minus_pre",
    "sync_drop_pre",
    "sync_drop_post"
]]

print("\nMAIN TABLE")
display(main_table)

print("\nDONE.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# SUMMARY TRANSITION PANEL
# Recovery + memory + synchronization
# ============================================================

OUTDIR = Path("stuart_landau_recovery_memory_output")
PREPOST_DIR = Path("stuart_landau_pre_post_recovery_output")

recovery_df = pd.read_csv(OUTDIR / "recovery_memory_aggregate_by_K.csv")
prepost_df = pd.read_csv(PREPOST_DIR / "pre_post_recovery_comparison.csv")

# Merge useful columns
df = recovery_df.merge(
    prepost_df[[
        "K",
        "recovery_time_pre",
        "recovery_time_post",
        "recovery_time_ratio_post_over_pre"
    ]],
    on="K",
    how="left"
)

# Save summary
summary_path = PREPOST_DIR / "transition_summary_panel_data.csv"
df.to_csv(summary_path, index=False)

# ============================================================
# PLOT PANEL
# ============================================================

fig, axes = plt.subplots(4, 1, figsize=(10, 16), sharex=True)

# 1. Mean synchronization before perturbation
axes[0].errorbar(
    df["K"],
    df["R_before_mean"],
    yerr=df["R_before_std"],
    marker="o",
    capsize=3
)
axes[0].set_ylabel("R before")
axes[0].set_title("Stuart–Landau transition summary: recovery / memory / stability")
axes[0].grid(True)

# 2. Recovery time
axes[1].errorbar(
    df["K"],
    df["recovery_time_mean"],
    yerr=df["recovery_time_std"],
    marker="o",
    capsize=3
)
axes[1].set_ylabel("Recovery time")
axes[1].grid(True)

# 3. Memory ratio
axes[2].errorbar(
    df["K"],
    df["memory_ratio_mean"],
    yerr=df["memory_ratio_std"],
    marker="o",
    capsize=3
)
axes[2].axhline(1.0, linestyle="--")
axes[2].set_ylabel("Memory ratio")
axes[2].grid(True)

# 4. Post / pre recovery ratio
axes[3].plot(
    df["K"],
    df["recovery_time_ratio_post_over_pre"],
    marker="o"
)
axes[3].axhline(1.0, linestyle="--")
axes[3].set_ylabel("Post / Pre recovery")
axes[3].set_xlabel("Coupling K")
axes[3].grid(True)

# Mark transition band
for ax in axes:
    ax.axvspan(0.3, 0.5, alpha=0.18)

plt.tight_layout()

fig_path = PREPOST_DIR / "fig_transition_summary_panel.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:")
print(summary_path)
print(fig_path)

display(df[[
    "K",
    "R_before_mean",
    "recovery_time_mean",
    "memory_ratio_mean",
    "recovery_time_ratio_post_over_pre"
]])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# KICK STRENGTH ROBUSTNESS TEST
# ============================================================

OUTDIR = Path("stuart_landau_kick_robustness_output")
OUTDIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

kick_strengths = [0.25, 0.5, 1.0, 1.5, 2.0]

# usesmy above policzonego agg_df
# If it does not exist:
try:
    agg_df
except NameError:
    agg_df = pd.read_csv(
        "stuart_landau_recovery_memory_output/recovery_memory_aggregate_by_K.csv"
    )

# ------------------------------------------------------------
# SIMPLE SCALING MODEL
# ------------------------------------------------------------
# Assumptions:
# recovery_time_eff ~ recovery_time * kick_strength
#
# We want to test:
# whether the K≈0.3–0.5 threshold remains stable
# at different perturbation strengths.
# ------------------------------------------------------------

rows = []

for ks in kick_strengths:

    temp = agg_df.copy()

    temp["kick_strength"] = ks

    temp["effective_recovery_time"] = (
        temp["recovery_time_mean"] * ks
    )

    temp["effective_sync_drop"] = (
        temp["synchronization_drop_mean"] * ks
    )

    rows.append(temp)

robust_df = pd.concat(rows, ignore_index=True)

# Save table
csv_path = OUTDIR / "kick_strength_robustness.csv"
robust_df.to_csv(csv_path, index=False)

# ============================================================
# PLOT
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

# ------------------------------------------------------------
# Recovery time
# ------------------------------------------------------------

for ks in kick_strengths:

    sub = robust_df[
        robust_df["kick_strength"] == ks
    ]

    axes[0].plot(
        sub["K"],
        sub["effective_recovery_time"],
        marker="o",
        label=f"kick={ks}"
    )

axes[0].axvspan(0.3, 0.5, alpha=0.18)

axes[0].set_ylabel("Effective recovery time")
axes[0].set_title("Kick strength robustness")
axes[0].grid(True)
axes[0].legend()

# ------------------------------------------------------------
# Sync drop
# ------------------------------------------------------------

for ks in kick_strengths:

    sub = robust_df[
        robust_df["kick_strength"] == ks
    ]

    axes[1].plot(
        sub["K"],
        sub["effective_sync_drop"],
        marker="o",
        label=f"kick={ks}"
    )

axes[1].axvspan(0.3, 0.5, alpha=0.18)

axes[1].set_ylabel("Effective sync drop")
axes[1].set_xlabel("Coupling K")
axes[1].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_kick_strength_robustness.png"

plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:")
print(csv_path)
print(fig_path)

display(
    robust_df[[
        "K",
        "kick_strength",
        "effective_recovery_time",
        "effective_sync_drop"
    ]].head(20)
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# REAL KICK STRENGTH RECOVERY TEST
# ============================================================

OUTDIR = Path("stuart_landau_real_kick_test_output")
CHECKDIR = OUTDIR / "checkpoints"

OUTDIR.mkdir(exist_ok=True)
CHECKDIR.mkdir(exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================

np.random.seed(42)

N = 100
dt = 0.02
steps = 3000

omega_std = 0.1

kick_time = 1500
recovery_threshold = 0.95

kick_strengths = [0.25, 0.5, 1.0, 1.5, 2.0]

K_values = [
    0.1, 0.3, 0.5,
    0.7, 0.8, 0.85, 0.9,
    0.95, 1.0, 1.05, 1.1,
    1.15, 1.2, 1.25, 1.3,
    1.35, 1.4, 1.45, 1.5,
    1.55, 1.6, 1.65, 1.7,
    1.75, 1.8, 2.0, 2.2, 2.4
]

seeds = range(10)

# ============================================================
# ORDER PARAMETER
# ============================================================

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

# ============================================================
# MAIN LOOP
# ============================================================

all_rows = []

for ks in kick_strengths:

    print("\n" + "="*70)
    print(f"KICK STRENGTH = {ks}")
    print("="*70)

    for seed in seeds:

        np.random.seed(seed)

        omega = np.random.normal(0, omega_std, N)

        for K in K_values:

            theta = np.random.uniform(0, 2*np.pi, N)

            R_series = []

            recovered = False
            recovery_time = np.nan

            R_before = np.nan

            for t in range(steps):

                R = order_parameter(theta)
                psi = np.angle(np.mean(np.exp(1j * theta)))

                R_series.append(R)

                coupling = K * R * np.sin(psi - theta)

                theta += (omega + coupling) * dt

                # ------------------------------------------------
                # PERTURBATION
                # ------------------------------------------------

                if t == kick_time:

                    R_before = R

                    theta += np.random.normal(
                        0,
                        ks,
                        size=N
                    )

                # ------------------------------------------------
                # RECOVERY
                # ------------------------------------------------

                if t > kick_time and not recovered:

                    if R >= recovery_threshold * R_before:

                        recovery_time = (
                            (t - kick_time) * dt
                        )

                        recovered = True

            R_final = R_series[-1]

            memory_ratio = (
                R_final / R_before
                if R_before > 0 else np.nan
            )

            sync_drop = (
                R_before - np.min(R_series[kick_time:])
            )

            all_rows.append({

                "seed": seed,
                "K": K,
                "kick_strength": ks,

                "R_before": R_before,
                "R_final": R_final,

                "memory_ratio": memory_ratio,
                "recovery_time": recovery_time,
                "sync_drop": sync_drop,
                "recovered": recovered
            })

# ============================================================
# SAVE
# ============================================================

all_df = pd.DataFrame(all_rows)

all_path = OUTDIR / "real_kick_test_all.csv"
all_df.to_csv(all_path, index=False)

# ============================================================
# AGGREGATE
# ============================================================

agg_df = (
    all_df
    .groupby(["kick_strength", "K"])
    .agg({

        "R_before": ["mean", "std"],
        "R_final": ["mean", "std"],

        "memory_ratio": ["mean", "std"],
        "recovery_time": ["mean", "std"],

        "sync_drop": ["mean", "std"],
        "recovered": "mean"
    })
)

agg_df.columns = [
    "_".join(col).strip("_")
    for col in agg_df.columns
]

agg_df = agg_df.reset_index()

agg_path = OUTDIR / "real_kick_test_aggregate.csv"
agg_df.to_csv(agg_path, index=False)

# ============================================================
# PLOTS
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(10, 14), sharex=True)

# ------------------------------------------------------------
# Recovery time
# ------------------------------------------------------------

for ks in kick_strengths:

    sub = agg_df[
        agg_df["kick_strength"] == ks
    ]

    axes[0].plot(
        sub["K"],
        sub["recovery_time_mean"],
        marker="o",
        label=f"kick={ks}"
    )

axes[0].axvspan(0.3, 0.5, alpha=0.18)

axes[0].set_ylabel("Recovery time")
axes[0].set_title("Real kick robustness test")
axes[0].grid(True)
axes[0].legend()

# ------------------------------------------------------------
# Memory ratio
# ------------------------------------------------------------

for ks in kick_strengths:

    sub = agg_df[
        agg_df["kick_strength"] == ks
    ]

    axes[1].plot(
        sub["K"],
        sub["memory_ratio_mean"],
        marker="o",
        label=f"kick={ks}"
    )

axes[1].axhline(1.0, linestyle="--")
axes[1].axvspan(0.3, 0.5, alpha=0.18)

axes[1].set_ylabel("Memory ratio")
axes[1].grid(True)

# ------------------------------------------------------------
# Sync drop
# ------------------------------------------------------------

for ks in kick_strengths:

    sub = agg_df[
        agg_df["kick_strength"] == ks
    ]

    axes[2].plot(
        sub["K"],
        sub["sync_drop_mean"],
        marker="o",
        label=f"kick={ks}"
    )

axes[2].axvspan(0.3, 0.5, alpha=0.18)

axes[2].set_ylabel("Sync drop")
axes[2].set_xlabel("Coupling K")
axes[2].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_real_kick_robustness.png"

plt.savefig(fig_path, dpi=300)
plt.show()

# ============================================================
# OUTPUT
# ============================================================

print("\nSaved:")
print(all_path)
print(agg_path)
print(fig_path)

display(
    agg_df[[
        "kick_strength",
        "K",
        "recovery_time_mean",
        "memory_ratio_mean",
        "sync_drop_mean",
        "recovered_mean"
    ]].head(30)
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# REAL N ROBUSTNESS TEST — STUART–LANDAU / PHASE RECOVERY
# ============================================================

OUTDIR = Path("stuart_landau_N_robustness_output")
OUTDIR.mkdir(exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================

N_values = [50, 100, 200, 400]

K_values = [
    0.1, 0.3, 0.5,
    0.7, 0.8, 0.9, 1.0,
    1.1, 1.2, 1.3, 1.4,
    1.5, 1.6, 1.8, 2.0, 2.4
]

seeds = range(10)

dt = 0.02
steps = 3000
kick_time = 1500

omega_std = 0.1
kick_strength = 1.0
recovery_threshold = 0.95

# ============================================================
# FUNCTIONS
# ============================================================

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

# ============================================================
# MAIN LOOP
# ============================================================

rows = []

for N in N_values:
    print("\n" + "="*70)
    print(f"N = {N}")
    print("="*70)

    for seed in seeds:
        print(f"seed={seed}")

        rng = np.random.default_rng(seed)
        omega = rng.normal(0, omega_std, N)

        for K in K_values:

            theta = rng.uniform(0, 2*np.pi, N)

            R_series = []
            recovered = False
            recovery_time = np.nan
            R_before = np.nan

            for t in range(steps):

                R = order_parameter(theta)
                psi = np.angle(np.mean(np.exp(1j * theta)))
                R_series.append(R)

                coupling = K * R * np.sin(psi - theta)
                theta += (omega + coupling) * dt

                if t == kick_time:
                    R_before = R
                    theta += rng.normal(0, kick_strength, size=N)

                if t > kick_time and not recovered:
                    if R_before > 0 and R >= recovery_threshold * R_before:
                        recovery_time = (t - kick_time) * dt
                        recovered = True

            R_final = R_series[-1]

            memory_ratio = (
                R_final / R_before
                if R_before and R_before > 0
                else np.nan
            )

            sync_drop = (
                R_before - np.min(R_series[kick_time:])
                if R_before and R_before > 0
                else np.nan
            )

            rows.append({
                "N": N,
                "seed": seed,
                "K": K,
                "R_before": R_before,
                "R_final": R_final,
                "memory_ratio": memory_ratio,
                "recovery_time": recovery_time,
                "sync_drop": sync_drop,
                "recovered": recovered
            })

# ============================================================
# SAVE ALL
# ============================================================

all_df = pd.DataFrame(rows)
all_path = OUTDIR / "N_robustness_all.csv"
all_df.to_csv(all_path, index=False)

# ============================================================
# AGGREGATE
# ============================================================

agg_df = (
    all_df
    .groupby(["N", "K"])
    .agg({
        "R_before": ["mean", "std"],
        "R_final": ["mean", "std"],
        "memory_ratio": ["mean", "std"],
        "recovery_time": ["mean", "std"],
        "sync_drop": ["mean", "std"],
        "recovered": "mean"
    })
)

agg_df.columns = [
    "_".join(col).strip("_")
    for col in agg_df.columns
]

agg_df = agg_df.reset_index()

agg_path = OUTDIR / "N_robustness_aggregate.csv"
agg_df.to_csv(agg_path, index=False)

# ============================================================
# PLOTS
# ============================================================

fig, axes = plt.subplots(4, 1, figsize=(10, 18), sharex=True)

# 1. R before
for N in N_values:
    sub = agg_df[agg_df["N"] == N]
    axes[0].plot(sub["K"], sub["R_before_mean"], marker="o", label=f"N={N}")

axes[0].axvspan(0.3, 0.5, alpha=0.18)
axes[0].set_ylabel("R before")
axes[0].set_title("N-dependence robustness: synchronization / recovery / memory")
axes[0].grid(True)
axes[0].legend()

# 2. Recovery time
for N in N_values:
    sub = agg_df[agg_df["N"] == N]
    axes[1].plot(sub["K"], sub["recovery_time_mean"], marker="o", label=f"N={N}")

axes[1].axvspan(0.3, 0.5, alpha=0.18)
axes[1].set_ylabel("Recovery time")
axes[1].grid(True)

# 3. Memory ratio
for N in N_values:
    sub = agg_df[agg_df["N"] == N]
    axes[2].plot(sub["K"], sub["memory_ratio_mean"], marker="o", label=f"N={N}")

axes[2].axhline(1.0, linestyle="--")
axes[2].axvspan(0.3, 0.5, alpha=0.18)
axes[2].set_ylabel("Memory ratio")
axes[2].grid(True)

# 4. Sync drop
for N in N_values:
    sub = agg_df[agg_df["N"] == N]
    axes[3].plot(sub["K"], sub["sync_drop_mean"], marker="o", label=f"N={N}")

axes[3].axvspan(0.3, 0.5, alpha=0.18)
axes[3].set_ylabel("Sync drop")
axes[3].set_xlabel("Coupling K")
axes[3].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_N_robustness.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("\nSaved:")
print(all_path)
print(agg_path)
print(fig_path)

display(
    agg_df[[
        "N",
        "K",
        "R_before_mean",
        "recovery_time_mean",
        "memory_ratio_mean",
        "sync_drop_mean",
        "recovered_mean"
    ]].head(40)
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# LOCAL K-ZOOM TEST — N=400, K=0.2–0.6
# ============================================================

OUTDIR = Path("stuart_landau_K_zoom_N400_output")
OUTDIR.mkdir(exist_ok=True)

N = 400
seeds = range(20)

K_values = np.round(np.arange(0.20, 0.601, 0.025), 3)

dt = 0.02
steps = 3000
kick_time = 1500

omega_std = 0.1
kick_strength = 1.0
recovery_threshold = 0.95

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

rows = []

for seed in seeds:
    print(f"\nSEED {seed}")
    rng = np.random.default_rng(seed)
    omega = rng.normal(0, omega_std, N)

    for K in K_values:
        print(f"K={K}")

        theta = rng.uniform(0, 2*np.pi, N)

        R_series = []
        recovered = False
        recovery_time = np.nan
        R_before = np.nan

        for t in range(steps):
            R = order_parameter(theta)
            psi = np.angle(np.mean(np.exp(1j * theta)))
            R_series.append(R)

            coupling = K * R * np.sin(psi - theta)
            theta += (omega + coupling) * dt

            if t == kick_time:
                R_before = R
                theta += rng.normal(0, kick_strength, size=N)

            if t > kick_time and not recovered:
                if R_before > 0 and R >= recovery_threshold * R_before:
                    recovery_time = (t - kick_time) * dt
                    recovered = True

        R_final = R_series[-1]

        memory_ratio = R_final / R_before if R_before > 0 else np.nan
        sync_drop = R_before - np.min(R_series[kick_time:]) if R_before > 0 else np.nan

        rows.append({
            "seed": seed,
            "K": K,
            "R_before": R_before,
            "R_final": R_final,
            "memory_ratio": memory_ratio,
            "recovery_time": recovery_time,
            "sync_drop": sync_drop,
            "recovered": recovered
        })

all_df = pd.DataFrame(rows)
all_path = OUTDIR / "K_zoom_N400_all.csv"
all_df.to_csv(all_path, index=False)

agg_df = (
    all_df
    .groupby("K")
    .agg({
        "R_before": ["mean", "std"],
        "R_final": ["mean", "std"],
        "memory_ratio": ["mean", "std"],
        "recovery_time": ["mean", "std"],
        "sync_drop": ["mean", "std"],
        "recovered": "mean"
    })
)

agg_df.columns = ["_".join(c).strip("_") for c in agg_df.columns]
agg_df = agg_df.reset_index()

agg_path = OUTDIR / "K_zoom_N400_aggregate.csv"
agg_df.to_csv(agg_path, index=False)

fig, axes = plt.subplots(4, 1, figsize=(10, 18), sharex=True)

axes[0].errorbar(
    agg_df["K"], agg_df["R_before_mean"],
    yerr=agg_df["R_before_std"],
    marker="o", capsize=3
)
axes[0].set_ylabel("R before")
axes[0].set_title("Local K zoom: N=400, transition window 0.2–0.6")
axes[0].grid(True)

axes[1].errorbar(
    agg_df["K"], agg_df["recovery_time_mean"],
    yerr=agg_df["recovery_time_std"],
    marker="o", capsize=3
)
axes[1].set_ylabel("Recovery time")
axes[1].grid(True)

axes[2].errorbar(
    agg_df["K"], agg_df["memory_ratio_mean"],
    yerr=agg_df["memory_ratio_std"],
    marker="o", capsize=3
)
axes[2].axhline(1.0, linestyle="--")
axes[2].set_ylabel("Memory ratio")
axes[2].grid(True)

axes[3].errorbar(
    agg_df["K"], agg_df["sync_drop_mean"],
    yerr=agg_df["sync_drop_std"],
    marker="o", capsize=3
)
axes[3].set_ylabel("Sync drop")
axes[3].set_xlabel("Coupling K")
axes[3].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_K_zoom_N400.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("\nSaved:")
print(all_path)
print(agg_path)
print(fig_path)

display(
    agg_df[[
        "K",
        "R_before_mean",
        "recovery_time_mean",
        "memory_ratio_mean",
        "sync_drop_mean",
        "recovered_mean"
    ]]
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# OMEGA_STD ROBUSTNESS — LOCAL K ZOOM, N=400
# ============================================================

OUTDIR = Path("stuart_landau_omega_robustness_N400_output")
OUTDIR.mkdir(exist_ok=True)

N = 400
seeds = range(15)

omega_values = [0.05, 0.10, 0.20, 0.40]
K_values = np.round(np.arange(0.20, 0.601, 0.025), 3)

dt = 0.02
steps = 3000
kick_time = 1500

kick_strength = 1.0
recovery_threshold = 0.95

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

rows = []

for omega_std in omega_values:
    print("\n" + "="*70)
    print(f"omega_std = {omega_std}")
    print("="*70)

    for seed in seeds:
        print(f"seed={seed}")
        rng = np.random.default_rng(seed)
        omega = rng.normal(0, omega_std, N)

        for K in K_values:
            theta = rng.uniform(0, 2*np.pi, N)

            R_series = []
            recovered = False
            recovery_time = np.nan
            R_before = np.nan

            for t in range(steps):
                R = order_parameter(theta)
                psi = np.angle(np.mean(np.exp(1j * theta)))
                R_series.append(R)

                coupling = K * R * np.sin(psi - theta)
                theta += (omega + coupling) * dt

                if t == kick_time:
                    R_before = R
                    theta += rng.normal(0, kick_strength, size=N)

                if t > kick_time and not recovered:
                    if R_before > 0 and R >= recovery_threshold * R_before:
                        recovery_time = (t - kick_time) * dt
                        recovered = True

            R_final = R_series[-1]

            memory_ratio = R_final / R_before if R_before > 0 else np.nan
            sync_drop = R_before - np.min(R_series[kick_time:]) if R_before > 0 else np.nan

            rows.append({
                "omega_std": omega_std,
                "seed": seed,
                "K": K,
                "R_before": R_before,
                "R_final": R_final,
                "memory_ratio": memory_ratio,
                "recovery_time": recovery_time,
                "sync_drop": sync_drop,
                "recovered": recovered
            })

all_df = pd.DataFrame(rows)
all_path = OUTDIR / "omega_robustness_N400_all.csv"
all_df.to_csv(all_path, index=False)

agg_df = (
    all_df
    .groupby(["omega_std", "K"])
    .agg({
        "R_before": ["mean", "std"],
        "R_final": ["mean", "std"],
        "memory_ratio": ["mean", "std"],
        "recovery_time": ["mean", "std"],
        "sync_drop": ["mean", "std"],
        "recovered": "mean"
    })
)

agg_df.columns = ["_".join(c).strip("_") for c in agg_df.columns]
agg_df = agg_df.reset_index()

agg_path = OUTDIR / "omega_robustness_N400_aggregate.csv"
agg_df.to_csv(agg_path, index=False)

# ============================================================
# SIMPLE THRESHOLD ESTIMATES
# ============================================================

threshold_rows = []

for omega_std in omega_values:
    sub = agg_df[agg_df["omega_std"] == omega_std].copy()

    # first K where R_before >= 0.9
    r90 = sub[sub["R_before_mean"] >= 0.9]
    K_R90 = r90["K"].iloc[0] if len(r90) else np.nan

    # first K where memory ratio close to 1
    mem = sub[np.abs(sub["memory_ratio_mean"] - 1.0) <= 0.05]
    K_memory = mem["K"].iloc[0] if len(mem) else np.nan

    # first K where recovery time <= 5
    rec = sub[sub["recovery_time_mean"] <= 5.0]
    K_recovery = rec["K"].iloc[0] if len(rec) else np.nan

    # max slope of R_before
    dR = np.gradient(sub["R_before_mean"], sub["K"])
    K_max_dR = sub["K"].iloc[np.nanargmax(dR)]

    threshold_rows.append({
        "omega_std": omega_std,
        "K_R90": K_R90,
        "K_memory_ratio_close_to_1": K_memory,
        "K_recovery_time_below_5": K_recovery,
        "K_max_dR": K_max_dR
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_path = OUTDIR / "omega_robustness_thresholds.csv"
threshold_df.to_csv(threshold_path, index=False)

# ============================================================
# PLOTS
# ============================================================

fig, axes = plt.subplots(4, 1, figsize=(10, 18), sharex=True)

for omega_std in omega_values:
    sub = agg_df[agg_df["omega_std"] == omega_std]

    axes[0].plot(sub["K"], sub["R_before_mean"], marker="o", label=f"ω_std={omega_std}")
    axes[1].plot(sub["K"], sub["recovery_time_mean"], marker="o", label=f"ω_std={omega_std}")
    axes[2].plot(sub["K"], sub["memory_ratio_mean"], marker="o", label=f"ω_std={omega_std}")
    axes[3].plot(sub["K"], sub["sync_drop_mean"], marker="o", label=f"ω_std={omega_std}")

axes[0].set_title("Omega_std robustness: N=400, K zoom 0.2–0.6")
axes[0].set_ylabel("R before")
axes[0].grid(True)
axes[0].legend()

axes[1].set_ylabel("Recovery time")
axes[1].grid(True)

axes[2].axhline(1.0, linestyle="--")
axes[2].set_ylabel("Memory ratio")
axes[2].grid(True)

axes[3].set_ylabel("Sync drop")
axes[3].set_xlabel("Coupling K")
axes[3].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_omega_robustness_N400.png"
plt.savefig(fig_path, dpi=300)
plt.show()

# threshold plot
plt.figure(figsize=(9, 6))
plt.plot(threshold_df["omega_std"], threshold_df["K_R90"], marker="o", label="K where R≥0.9")
plt.plot(threshold_df["omega_std"], threshold_df["K_memory_ratio_close_to_1"], marker="o", label="K where memory≈1")
plt.plot(threshold_df["omega_std"], threshold_df["K_recovery_time_below_5"], marker="o", label="K where recovery≤5")
plt.plot(threshold_df["omega_std"], threshold_df["K_max_dR"], marker="o", label="K at max dR/dK")
plt.xlabel("omega_std")
plt.ylabel("Estimated threshold K")
plt.title("Threshold shift vs frequency dispersion")
plt.grid(True)
plt.legend()
plt.tight_layout()

fig_threshold_path = OUTDIR / "fig_omega_threshold_shift.png"
plt.savefig(fig_threshold_path, dpi=300)
plt.show()

print("\nSaved:")
print(all_path)
print(agg_path)
print(threshold_path)
print(fig_path)
print(fig_threshold_path)

print("\nTHRESHOLD ESTIMATES")
display(threshold_df)

print("\nAGGREGATE PREVIEW")
display(
    agg_df[[
        "omega_std",
        "K",
        "R_before_mean",
        "recovery_time_mean",
        "memory_ratio_mean",
        "sync_drop_mean",
        "recovered_mean"
    ]].head(60)
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# EXTENDED K TEST FOR omega_std = 0.40
# ============================================================

OUTDIR = Path("stuart_landau_omega04_extendedK_output")
OUTDIR.mkdir(exist_ok=True)

omega_std = 0.40
N = 400
seeds = range(20)

K_values = np.round(np.arange(0.40, 1.601, 0.05), 3)

dt = 0.02
steps = 3000
kick_time = 1500

kick_strength = 1.0
recovery_threshold = 0.95

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

rows = []

print("\n" + "="*70)
print("omega_std = 0.40 | EXTENDED K TEST")
print("="*70)

for seed in seeds:

    print(f"seed={seed}")

    rng = np.random.default_rng(seed)
    omega = rng.normal(0, omega_std, N)

    for K in K_values:

        print(f"K={K}")

        theta = rng.uniform(0, 2*np.pi, N)

        R_series = []

        recovered = False
        recovery_time = np.nan
        R_before = np.nan

        for t in range(steps):

            R = order_parameter(theta)
            psi = np.angle(np.mean(np.exp(1j * theta)))

            R_series.append(R)

            coupling = K * R * np.sin(psi - theta)

            theta += (omega + coupling) * dt

            if t == kick_time:
                R_before = R
                theta += rng.normal(0, kick_strength, size=N)

            if t > kick_time and not recovered:
                if R_before > 0 and R >= recovery_threshold * R_before:
                    recovery_time = (t - kick_time) * dt
                    recovered = True

        R_final = R_series[-1]

        memory_ratio = R_final / R_before if R_before > 0 else np.nan

        sync_drop = (
            R_before - np.min(R_series[kick_time:])
            if R_before > 0 else np.nan
        )

        rows.append({
            "seed": seed,
            "K": K,
            "R_before": R_before,
            "R_final": R_final,
            "memory_ratio": memory_ratio,
            "recovery_time": recovery_time,
            "sync_drop": sync_drop,
            "recovered": recovered
        })

# ============================================================
# DATAFRAMES
# ============================================================

all_df = pd.DataFrame(rows)

agg_df = (
    all_df
    .groupby("K")
    .agg({
        "R_before": ["mean", "std"],
        "R_final": ["mean", "std"],
        "memory_ratio": ["mean", "std"],
        "recovery_time": ["mean", "std"],
        "sync_drop": ["mean", "std"],
        "recovered": "mean"
    })
)

agg_df.columns = ["_".join(c).strip("_") for c in agg_df.columns]
agg_df = agg_df.reset_index()

# ============================================================
# SAVE
# ============================================================

all_path = OUTDIR / "omega04_extendedK_all.csv"
agg_path = OUTDIR / "omega04_extendedK_aggregate.csv"

all_df.to_csv(all_path, index=False)
agg_df.to_csv(agg_path, index=False)

# ============================================================
# PLOT
# ============================================================

fig, axes = plt.subplots(4, 1, figsize=(10, 18), sharex=True)

axes[0].errorbar(
    agg_df["K"],
    agg_df["R_before_mean"],
    yerr=agg_df["R_before_std"],
    marker="o",
    capsize=3
)
axes[0].set_ylabel("R before")
axes[0].set_title("omega_std=0.40 | extended K scan")
axes[0].grid(True)

axes[1].errorbar(
    agg_df["K"],
    agg_df["recovery_time_mean"],
    yerr=agg_df["recovery_time_std"],
    marker="o",
    capsize=3
)
axes[1].set_ylabel("Recovery time")
axes[1].grid(True)

axes[2].errorbar(
    agg_df["K"],
    agg_df["memory_ratio_mean"],
    yerr=agg_df["memory_ratio_std"],
    marker="o",
    capsize=3
)
axes[2].axhline(1.0, linestyle="--")
axes[2].set_ylabel("Memory ratio")
axes[2].grid(True)

axes[3].errorbar(
    agg_df["K"],
    agg_df["sync_drop_mean"],
    yerr=agg_df["sync_drop_std"],
    marker="o",
    capsize=3
)
axes[3].set_ylabel("Sync drop")
axes[3].set_xlabel("Coupling K")
axes[3].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_omega04_extendedK.png"

plt.savefig(fig_path, dpi=300)
plt.show()

# ============================================================
# THRESHOLD ESTIMATES
# ============================================================

r90 = agg_df[agg_df["R_before_mean"] >= 0.90]
K_R90 = r90["K"].iloc[0] if len(r90) else np.nan

mem = agg_df[np.abs(agg_df["memory_ratio_mean"] - 1.0) <= 0.05]
K_memory = mem["K"].iloc[0] if len(mem) else np.nan

rec = agg_df[agg_df["recovery_time_mean"] <= 5.0]
K_recovery = rec["K"].iloc[0] if len(rec) else np.nan

summary_df = pd.DataFrame([{
    "omega_std": omega_std,
    "K_R90": K_R90,
    "K_memory_ratio_close_to_1": K_memory,
    "K_recovery_below_5": K_recovery
}])

summary_path = OUTDIR / "omega04_threshold_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nSaved:")
print(all_path)
print(agg_path)
print(fig_path)
print(summary_path)

print("\nTHRESHOLD SUMMARY")
display(summary_df)

print("\nAGGREGATE TABLE")
display(
    agg_df[[
        "K",
        "R_before_mean",
        "recovery_time_mean",
        "memory_ratio_mean",
        "sync_drop_mean",
        "recovered_mean"
    ]]
)

In [ ]:
import os
import zipfile
import pandas as pd
from pathlib import Path

# ============================================================
# EXPORT ALL RESULTS TO CSV + ZIP
# ============================================================

EXPORT_DIR = Path("stuart_landau_full_export")
EXPORT_DIR.mkdir(exist_ok=True)

CSV_DIR = EXPORT_DIR / "csv_from_memory"
CSV_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Save all pandas DataFrames currently in memory
# ------------------------------------------------------------

saved_dfs = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        safe_name = name.replace("/", "_").replace("\\", "_")
        path = CSV_DIR / f"{safe_name}.csv"
        obj.to_csv(path, index=False)
        saved_dfs.append(str(path))

print("Saved DataFrames to CSV:")
for p in saved_dfs:
    print(" -", p)

# ------------------------------------------------------------
# 2. Collect existing result files from notebook folders
# ------------------------------------------------------------

EXTENSIONS = [
    "*.csv", "*.png", "*.jpg", "*.jpeg", "*.pdf",
    "*.txt", "*.npy", "*.npz"
]

result_files = []

for pattern in EXTENSIONS:
    result_files.extend(Path(".").rglob(pattern))

# skip files already inside EXPORT_DIR to avoid duplicates
result_files = [
    f for f in result_files
    if EXPORT_DIR not in f.parents
]

print("\nFound existing result files:", len(result_files))

# ------------------------------------------------------------
# 3. Create ZIP
# ------------------------------------------------------------

zip_path = Path("stuart_landau_all_results_export.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:

    # Add CSVs generated from memory
    for file_path in CSV_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.as_posix())

    # Add existing result files
    for file_path in result_files:
        if file_path.is_file():
            zipf.write(file_path, file_path.as_posix())

print("\nZIP created:")
print(zip_path)

# ------------------------------------------------------------
# 4. Download ZIP
# ------------------------------------------------------------

# Output files remain in the local results directory.

print("\nDONE.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# LOCAL CLUSTER TEST — omega_std=0.40
# Do local clusters emerge before global synchronization?
# ============================================================

OUTDIR = Path("stuart_landau_local_cluster_test_output")
OUTDIR.mkdir(exist_ok=True)

N = 400
omega_std = 0.40
seeds = range(20)

K_values = np.round(np.arange(0.40, 1.601, 0.05), 3)

dt = 0.02
steps = 3000
burn_in = 1500

# Number of phase bins used to detect clusters
phase_bins = 24

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def phase_entropy(theta, bins=24):
    hist, _ = np.histogram(theta % (2*np.pi), bins=bins, range=(0, 2*np.pi), density=False)
    p = hist / np.sum(hist)
    p = p[p > 0]
    H = -np.sum(p * np.log(p))
    H_norm = H / np.log(bins)
    return H_norm

def phase_cluster_count(theta, bins=24, threshold_frac=0.05):
    hist, _ = np.histogram(theta % (2*np.pi), bins=bins, range=(0, 2*np.pi), density=False)
    threshold = threshold_frac * np.sum(hist)
    active = hist > threshold

    if not np.any(active):
        return 0

    # Number of active-bin groups on the phase circle
    count = 0
    for i in range(bins):
        prev_i = (i - 1) % bins
        if active[i] and not active[prev_i]:
            count += 1

    return count

rows = []

for seed in seeds:
    print("\n" + "="*60)
    print(f"SEED {seed}")
    print("="*60)

    rng = np.random.default_rng(seed)
    omega = rng.normal(0, omega_std, N)

    for K in K_values:
        print(f"K={K}")

        theta = rng.uniform(0, 2*np.pi, N)

        R_series = []
        entropy_series = []
        cluster_series = []

        for t in range(steps):
            R = order_parameter(theta)
            psi = np.angle(np.mean(np.exp(1j * theta)))

            coupling = K * R * np.sin(psi - theta)
            theta += (omega + coupling) * dt

            if t >= burn_in:
                R_series.append(R)
                entropy_series.append(phase_entropy(theta, bins=phase_bins))
                cluster_series.append(phase_cluster_count(theta, bins=phase_bins))

        rows.append({
            "seed": seed,
            "K": K,
            "R_mean": np.mean(R_series),
            "R_std": np.std(R_series),
            "phase_entropy_mean": np.mean(entropy_series),
            "phase_entropy_std": np.std(entropy_series),
            "cluster_count_mean": np.mean(cluster_series),
            "cluster_count_std": np.std(cluster_series),
            "cluster_count_final": cluster_series[-1],
            "entropy_final": entropy_series[-1],
            "R_final": R_series[-1],
        })

# ============================================================
# SAVE DATA
# ============================================================

all_df = pd.DataFrame(rows)

agg_df = (
    all_df
    .groupby("K")
    .agg({
        "R_mean": ["mean", "std"],
        "phase_entropy_mean": ["mean", "std"],
        "cluster_count_mean": ["mean", "std"],
        "R_final": ["mean", "std"],
        "entropy_final": ["mean", "std"],
        "cluster_count_final": ["mean", "std"],
    })
)

agg_df.columns = ["_".join(c).strip("_") for c in agg_df.columns]
agg_df = agg_df.reset_index()

all_path = OUTDIR / "local_cluster_test_all.csv"
agg_path = OUTDIR / "local_cluster_test_aggregate.csv"

all_df.to_csv(all_path, index=False)
agg_df.to_csv(agg_path, index=False)

# ============================================================
# PLOTS
# ============================================================

fig, axes = plt.subplots(4, 1, figsize=(10, 18), sharex=True)

axes[0].errorbar(
    agg_df["K"],
    agg_df["R_mean_mean"],
    yerr=agg_df["R_mean_std"],
    marker="o",
    capsize=3
)
axes[0].set_ylabel("Mean R")
axes[0].set_title("Local cluster test: omega_std=0.40")
axes[0].grid(True)

axes[1].errorbar(
    agg_df["K"],
    agg_df["phase_entropy_mean_mean"],
    yerr=agg_df["phase_entropy_mean_std"],
    marker="o",
    capsize=3
)
axes[1].set_ylabel("Phase entropy")
axes[1].grid(True)

axes[2].errorbar(
    agg_df["K"],
    agg_df["cluster_count_mean_mean"],
    yerr=agg_df["cluster_count_mean_std"],
    marker="o",
    capsize=3
)
axes[2].set_ylabel("Cluster count")
axes[2].grid(True)

axes[3].errorbar(
    agg_df["K"],
    agg_df["cluster_count_final_mean"],
    yerr=agg_df["cluster_count_final_std"],
    marker="o",
    capsize=3
)
axes[3].set_ylabel("Final cluster count")
axes[3].set_xlabel("Coupling K")
axes[3].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_local_cluster_test.png"
plt.savefig(fig_path, dpi=300)
plt.show()

# ============================================================
# MAIN TABLE
# ============================================================

main_table = agg_df[[
    "K",
    "R_mean_mean",
    "phase_entropy_mean_mean",
    "cluster_count_mean_mean",
    "cluster_count_final_mean"
]]

main_path = OUTDIR / "local_cluster_main_table.csv"
main_table.to_csv(main_path, index=False)

print("\nSaved:")
print(all_path)
print(agg_path)
print(main_path)
print(fig_path)

print("\nMAIN TABLE")
display(main_table)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# CLUSTER LIFETIME / METASTABILITY TEST
# Stuart–Landau / Kuramoto phase reduction
# omega_std = 0.40
# ============================================================

OUTDIR = Path("stuart_landau_cluster_lifetime_output")
OUTDIR.mkdir(exist_ok=True)

N = 400
omega_std = 0.40
seeds = range(20)

K_values = np.round(np.arange(0.40, 1.201, 0.025), 3)

dt = 0.02
steps = 4000
burn_in = 1500

phase_bins = 24
threshold_frac = 0.05

def order_parameter(theta):
    z = np.mean(np.exp(1j * theta))
    return np.abs(z), np.angle(z)

def phase_cluster_count(theta, bins=24, threshold_frac=0.05):
    hist, _ = np.histogram(theta % (2*np.pi), bins=bins, range=(0, 2*np.pi))
    threshold = threshold_frac * np.sum(hist)
    active = hist > threshold

    if not np.any(active):
        return 0

    count = 0
    for i in range(bins):
        prev_i = (i - 1) % bins
        if active[i] and not active[prev_i]:
            count += 1

    return count

def run_lengths(values):
    values = list(values)
    if len(values) == 0:
        return []

    lengths = []
    current = values[0]
    length = 1

    for v in values[1:]:
        if v == current:
            length += 1
        else:
            lengths.append((current, length))
            current = v
            length = 1

    lengths.append((current, length))
    return lengths

rows = []

for seed in seeds:
    print("\n" + "="*70)
    print(f"SEED {seed}")
    print("="*70)

    rng_base = np.random.default_rng(seed)
    omega = rng_base.normal(0, omega_std, N)

    for K in K_values:
        print(f"K={K}")

        rng = np.random.default_rng(seed + int(K * 10000))
        theta = rng.uniform(0, 2*np.pi, N)

        R_series = []
        cluster_series = []

        for t in range(steps):
            R, psi = order_parameter(theta)
            theta += (omega + K * R * np.sin(psi - theta)) * dt

            if t >= burn_in:
                R_series.append(R)
                cluster_series.append(
                    phase_cluster_count(
                        theta,
                        bins=phase_bins,
                        threshold_frac=threshold_frac
                    )
                )

        cluster_series = np.array(cluster_series)
        R_series = np.array(R_series)

        rl = run_lengths(cluster_series)

        multi_cluster_lengths = [length for value, length in rl if value > 1]
        single_cluster_lengths = [length for value, length in rl if value == 1]

        total_points = len(cluster_series)
        multi_points = np.sum(cluster_series > 1)
        single_points = np.sum(cluster_series == 1)

        rows.append({
            "seed": seed,
            "K": K,
            "R_mean": np.mean(R_series),
            "R_std": np.std(R_series),

            "cluster_mean": np.mean(cluster_series),
            "cluster_std": np.std(cluster_series),
            "cluster_final": cluster_series[-1],

            "multi_cluster_fraction": multi_points / total_points if total_points > 0 else np.nan,
            "single_cluster_fraction": single_points / total_points if total_points > 0 else np.nan,

            "multi_cluster_lifetime_mean": np.mean(multi_cluster_lengths) * dt if len(multi_cluster_lengths) > 0 else 0.0,
            "multi_cluster_lifetime_max": np.max(multi_cluster_lengths) * dt if len(multi_cluster_lengths) > 0 else 0.0,

            "single_cluster_lifetime_mean": np.mean(single_cluster_lengths) * dt if len(single_cluster_lengths) > 0 else 0.0,
            "single_cluster_lifetime_max": np.max(single_cluster_lengths) * dt if len(single_cluster_lengths) > 0 else 0.0,

            "cluster_switch_count": len(rl) - 1,
            "cluster_switch_rate": (len(rl) - 1) / (total_points * dt) if total_points > 0 else np.nan,
        })

# ============================================================
# SAVE DATA
# ============================================================

all_df = pd.DataFrame(rows)

agg_df = (
    all_df
    .groupby("K")
    .agg({
        "R_mean": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "multi_cluster_fraction": ["mean", "std"],
        "single_cluster_fraction": ["mean", "std"],
        "multi_cluster_lifetime_mean": ["mean", "std"],
        "multi_cluster_lifetime_max": ["mean", "std"],
        "single_cluster_lifetime_mean": ["mean", "std"],
        "single_cluster_lifetime_max": ["mean", "std"],
        "cluster_switch_rate": ["mean", "std"],
    })
)

agg_df.columns = ["_".join(c).strip("_") for c in agg_df.columns]
agg_df = agg_df.reset_index()

all_path = OUTDIR / "cluster_lifetime_all.csv"
agg_path = OUTDIR / "cluster_lifetime_aggregate.csv"

all_df.to_csv(all_path, index=False)
agg_df.to_csv(agg_path, index=False)

# ============================================================
# PLOTS
# ============================================================

fig, axes = plt.subplots(5, 1, figsize=(10, 22), sharex=True)

axes[0].errorbar(
    agg_df["K"],
    agg_df["R_mean_mean"],
    yerr=agg_df["R_mean_std"],
    marker="o",
    capsize=3
)
axes[0].set_ylabel("Mean R")
axes[0].set_title("Cluster lifetime / metastability test — omega_std=0.40")
axes[0].grid(True)

axes[1].errorbar(
    agg_df["K"],
    agg_df["cluster_mean_mean"],
    yerr=agg_df["cluster_mean_std"],
    marker="o",
    capsize=3
)
axes[1].set_ylabel("Mean cluster count")
axes[1].grid(True)

axes[2].errorbar(
    agg_df["K"],
    agg_df["multi_cluster_fraction_mean"],
    yerr=agg_df["multi_cluster_fraction_std"],
    marker="o",
    capsize=3,
    label="multi-cluster"
)
axes[2].errorbar(
    agg_df["K"],
    agg_df["single_cluster_fraction_mean"],
    yerr=agg_df["single_cluster_fraction_std"],
    marker="o",
    capsize=3,
    label="single-cluster"
)
axes[2].set_ylabel("Fraction of time")
axes[2].legend()
axes[2].grid(True)

axes[3].errorbar(
    agg_df["K"],
    agg_df["multi_cluster_lifetime_mean_mean"],
    yerr=agg_df["multi_cluster_lifetime_mean_std"],
    marker="o",
    capsize=3,
    label="multi-cluster lifetime"
)
axes[3].errorbar(
    agg_df["K"],
    agg_df["single_cluster_lifetime_mean_mean"],
    yerr=agg_df["single_cluster_lifetime_mean_std"],
    marker="o",
    capsize=3,
    label="single-cluster lifetime"
)
axes[3].set_ylabel("Mean lifetime")
axes[3].legend()
axes[3].grid(True)

axes[4].errorbar(
    agg_df["K"],
    agg_df["cluster_switch_rate_mean"],
    yerr=agg_df["cluster_switch_rate_std"],
    marker="o",
    capsize=3
)
axes[4].set_ylabel("Switch rate")
axes[4].set_xlabel("Coupling K")
axes[4].grid(True)

plt.tight_layout()

fig_path = OUTDIR / "fig_cluster_lifetime_metastability.png"
plt.savefig(fig_path, dpi=300)
plt.show()

# ============================================================
# MAIN TABLE
# ============================================================

main_table = agg_df[[
    "K",
    "R_mean_mean",
    "cluster_mean_mean",
    "multi_cluster_fraction_mean",
    "single_cluster_fraction_mean",
    "multi_cluster_lifetime_mean_mean",
    "single_cluster_lifetime_mean_mean",
    "cluster_switch_rate_mean"
]]

main_path = OUTDIR / "cluster_lifetime_main_table.csv"
main_table.to_csv(main_path, index=False)

print("\nSaved:")
print(all_path)
print(agg_path)
print(main_path)
print(fig_path)

print("\nMAIN TABLE")
display(main_table)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTDIR = Path("stuart_landau_hysteresis_output")
OUTDIR.mkdir(exist_ok=True)

# ======================
# parameters
# ======================
N = 400
omega_std = 0.40
seeds = range(20)

K_up = np.round(np.arange(0.40, 1.201, 0.025), 3)
K_down = K_up[::-1]

dt = 0.02
T_per_K = 50.0
steps_per_K = int(T_per_K / dt)

burn_fraction = 0.5
burn_start = int(steps_per_K * burn_fraction)

sigma = 1.0
cluster_bins = 36
cluster_threshold = 0.03

rng_global = np.random.default_rng(123)


def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))


def count_phase_clusters(theta, bins=36, threshold=0.03):
    phases = (theta % (2*np.pi)) / (2*np.pi)
    hist, _ = np.histogram(phases, bins=bins, range=(0, 1), density=False)
    hist = hist / max(hist.sum(), 1)
    return int(np.sum(hist > threshold))


def step_stuart_landau(z, omega, K, dt):
    mean_z = np.mean(z)
    dz = (1 + 1j * omega - np.abs(z)**2) * z + K * (mean_z - z)
    return z + dt * dz


def run_hysteresis(seed):
    rng = np.random.default_rng(seed)

    omega = rng.normal(0.0, omega_std, N)
    z = 0.1 * (rng.normal(size=N) + 1j * rng.normal(size=N))

    rows = []

    for direction, K_values in [("up", K_up), ("down", K_down)]:
        for K in K_values:
            R_vals = []
            cluster_vals = []

            for t in range(steps_per_K):
                z = step_stuart_landau(z, omega, K, dt)

                if t >= burn_start:
                    theta = np.angle(z)
                    R_vals.append(order_parameter(theta))
                    cluster_vals.append(count_phase_clusters(theta, cluster_bins, cluster_threshold))

            rows.append({
                "seed": seed,
                "direction": direction,
                "K": K,
                "R_mean": np.mean(R_vals),
                "R_std": np.std(R_vals),
                "cluster_mean": np.mean(cluster_vals),
                "cluster_std": np.std(cluster_vals),
                "cluster_final": cluster_vals[-1],
            })

    return pd.DataFrame(rows)


# ======================
# RUN
# ======================
all_rows = []

for seed in seeds:
    print(f"\nSEED {seed}")
    df_seed = run_hysteresis(seed)
    df_seed.to_csv(OUTDIR / f"hysteresis_seed_{seed:03d}.csv", index=False)
    all_rows.append(df_seed)

all_df = pd.concat(all_rows, ignore_index=True)
all_df.to_csv(OUTDIR / "hysteresis_all.csv", index=False)

agg = (
    all_df
    .groupby(["direction", "K"], as_index=False)
    .agg(
        R_mean_mean=("R_mean", "mean"),
        R_mean_std=("R_mean", "std"),
        R_std_mean=("R_std", "mean"),
        cluster_mean_mean=("cluster_mean", "mean"),
        cluster_mean_std=("cluster_mean", "std"),
        cluster_final_mean=("cluster_final", "mean"),
    )
)

agg.to_csv(OUTDIR / "hysteresis_aggregate.csv", index=False)

# Pivot table for the down-minus-up difference
up = agg[agg["direction"] == "up"].copy()
down = agg[agg["direction"] == "down"].copy()

comp = pd.merge(
    up,
    down,
    on="K",
    suffixes=("_up", "_down")
)

comp["R_hysteresis_gap"] = comp["R_mean_mean_down"] - comp["R_mean_mean_up"]
comp["cluster_hysteresis_gap"] = comp["cluster_mean_mean_down"] - comp["cluster_mean_mean_up"]

comp.to_csv(OUTDIR / "hysteresis_gap_table.csv", index=False)

# ======================
# PLOTS
# ======================
fig, axes = plt.subplots(4, 1, figsize=(9, 14), sharex=True)

for direction, label in [("up", "K increasing"), ("down", "K decreasing")]:
    d = agg[agg["direction"] == direction]

    axes[0].errorbar(
        d["K"], d["R_mean_mean"], yerr=d["R_mean_std"],
        marker="o", capsize=3, label=label
    )

    axes[1].errorbar(
        d["K"], d["cluster_mean_mean"], yerr=d["cluster_mean_std"],
        marker="o", capsize=3, label=label
    )

axes[2].plot(comp["K"], comp["R_hysteresis_gap"], marker="o")
axes[3].plot(comp["K"], comp["cluster_hysteresis_gap"], marker="o")

axes[0].set_ylabel("Mean R")
axes[1].set_ylabel("Mean cluster count")
axes[2].set_ylabel("R down - R up")
axes[3].set_ylabel("Cluster down - up")
axes[3].set_xlabel("Coupling K")

axes[0].set_title("Stuart–Landau hysteresis test: increasing vs decreasing K")

for ax in axes:
    ax.grid(True, alpha=0.35)
    ax.axvspan(0.625, 0.8, alpha=0.15)
    ax.legend(loc="best") if ax in [axes[0], axes[1]] else None

plt.tight_layout()
fig_path = OUTDIR / "fig_hysteresis_test.png"
plt.savefig(fig_path, dpi=200)
plt.show()

print("\nSaved:")
print(OUTDIR / "hysteresis_all.csv")
print(OUTDIR / "hysteresis_aggregate.csv")
print(OUTDIR / "hysteresis_gap_table.csv")
print(fig_path)

print("\nHYSTERESIS GAP PREVIEW")
display(comp[[
    "K",
    "R_mean_mean_up",
    "R_mean_mean_down",
    "R_hysteresis_gap",
    "cluster_mean_mean_up",
    "cluster_mean_mean_down",
    "cluster_hysteresis_gap"
]])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

INFILE = Path("stuart_landau_cluster_lifetime_output/cluster_lifetime_all.csv")
OUTDIR = Path("stuart_landau_fluctuation_peak_output")
OUTDIR.mkdir(exist_ok=True)

df = pd.read_csv(INFILE)

agg = (
    df.groupby("K")
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "multi_cluster_fraction": ["mean", "std"],
        "single_cluster_fraction": ["mean", "std"],
        "cluster_switch_rate": ["mean", "std"],
    })
)

agg.columns = ["_".join(c).strip("_") for c in agg.columns]
agg = agg.reset_index()

# Numerical gradients vs K
K = agg["K"].values

for col in [
    "R_mean_mean",
    "cluster_mean_mean",
    "multi_cluster_fraction_mean",
    "single_cluster_fraction_mean",
    "cluster_switch_rate_mean"
]:
    y = agg[col].values
    agg[f"d_{col}_dK"] = np.gradient(y, K)

# Absolute transition intensity
agg["abs_dR_dK"] = np.abs(agg["d_R_mean_mean_dK"])
agg["abs_dcluster_dK"] = np.abs(agg["d_cluster_mean_mean_dK"])
agg["abs_dmulti_dK"] = np.abs(agg["d_multi_cluster_fraction_mean_dK"])
agg["abs_dsingle_dK"] = np.abs(agg["d_single_cluster_fraction_mean_dK"])

# Normalize selected indicators to [0,1]
def norm01(x):
    x = np.asarray(x, dtype=float)
    if np.nanmax(x) == np.nanmin(x):
        return np.zeros_like(x)
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x))

agg["fluctuation_score"] = (
    norm01(agg["R_std_mean"]) +
    norm01(agg["cluster_std_mean"]) +
    norm01(agg["cluster_switch_rate_mean"]) +
    norm01(agg["abs_dR_dK"]) +
    norm01(agg["abs_dcluster_dK"]) +
    norm01(agg["abs_dmulti_dK"])
) / 6

peak_idx = agg["fluctuation_score"].idxmax()
peak_K = agg.loc[peak_idx, "K"]

summary = pd.DataFrame([{
    "peak_K_fluctuation_score": peak_K,
    "peak_fluctuation_score": agg.loc[peak_idx, "fluctuation_score"],
    "R_mean_at_peak": agg.loc[peak_idx, "R_mean_mean"],
    "R_std_at_peak": agg.loc[peak_idx, "R_std_mean"],
    "cluster_mean_at_peak": agg.loc[peak_idx, "cluster_mean_mean"],
    "cluster_std_at_peak": agg.loc[peak_idx, "cluster_std_mean"],
    "multi_cluster_fraction_at_peak": agg.loc[peak_idx, "multi_cluster_fraction_mean"],
    "single_cluster_fraction_at_peak": agg.loc[peak_idx, "single_cluster_fraction_mean"],
    "cluster_switch_rate_at_peak": agg.loc[peak_idx, "cluster_switch_rate_mean"],
}])

# Save CSV
agg.to_csv(OUTDIR / "fluctuation_peak_aggregate.csv", index=False)
summary.to_csv(OUTDIR / "fluctuation_peak_summary.csv", index=False)

# Plot
fig, axes = plt.subplots(5, 1, figsize=(10, 22), sharex=True)

axes[0].errorbar(
    agg["K"], agg["R_mean_mean"], yerr=agg["R_mean_std"],
    marker="o", capsize=3
)
axes[0].axvline(peak_K, linestyle="--")
axes[0].set_ylabel("Mean R")
axes[0].set_title("Fluctuation peak / maximum reorganization test")
axes[0].grid(True)

axes[1].errorbar(
    agg["K"], agg["R_std_mean"], yerr=agg["R_std_std"],
    marker="o", capsize=3, label="R std"
)
axes[1].errorbar(
    agg["K"], agg["cluster_std_mean"], yerr=agg["cluster_std_std"],
    marker="o", capsize=3, label="cluster count std"
)
axes[1].axvline(peak_K, linestyle="--")
axes[1].set_ylabel("Fluctuation")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(agg["K"], agg["abs_dR_dK"], marker="o", label="|dR/dK|")
axes[2].plot(agg["K"], agg["abs_dcluster_dK"], marker="o", label="|d cluster/dK|")
axes[2].plot(agg["K"], agg["abs_dmulti_dK"], marker="o", label="|d multi/dK|")
axes[2].axvline(peak_K, linestyle="--")
axes[2].set_ylabel("Transition slope")
axes[2].legend()
axes[2].grid(True)

axes[3].errorbar(
    agg["K"], agg["cluster_switch_rate_mean"],
    yerr=agg["cluster_switch_rate_std"],
    marker="o", capsize=3
)
axes[3].axvline(peak_K, linestyle="--")
axes[3].set_ylabel("Switch rate")
axes[3].grid(True)

axes[4].plot(agg["K"], agg["fluctuation_score"], marker="o")
axes[4].axvline(peak_K, linestyle="--", label=f"peak K={peak_K:.3f}")
axes[4].set_ylabel("Fluctuation score")
axes[4].set_xlabel("Coupling K")
axes[4].legend()
axes[4].grid(True)

plt.tight_layout()
fig_path = OUTDIR / "fig_fluctuation_peak_test.png"
plt.savefig(fig_path, dpi=300)
plt.show()

print("Saved:")
print(OUTDIR / "fluctuation_peak_aggregate.csv")
print(OUTDIR / "fluctuation_peak_summary.csv")
print(fig_path)

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(40))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTDIR = Path("stuart_landau_hysteresis_output")
OUTDIR.mkdir(exist_ok=True)

# -----------------------------
# PARAMETERS
# -----------------------------
N = 400
omega_std = 0.40
seeds = range(20)

K_up = np.round(np.arange(0.40, 1.225, 0.025), 3)
K_down = K_up[::-1]

dt = 0.02
T_per_K = 20.0
steps_per_K = int(T_per_K / dt)

discard_fraction = 0.5

sigma = 0.05
noise_strength = 0.0

# Stuart-Landau parameters
lam = 1.0
beta = 0.0

# -----------------------------
# HELPERS
# -----------------------------
def order_parameter(z):
    phases = np.angle(z)
    return np.abs(np.mean(np.exp(1j * phases)))

def euler_step(z, omega, K, dt):
    mean_z = np.mean(z)
    dz = (lam + 1j * omega - (1 + 1j * beta) * np.abs(z)**2) * z + K * (mean_z - z)
    return z + dt * dz

def cluster_count(phases, bins=36, threshold=0.03):
    hist, _ = np.histogram(phases, bins=bins, range=(-np.pi, np.pi), density=True)
    active = hist > threshold * np.max(hist)
    if active.sum() == 0:
        return 0
    count = 0
    for i in range(len(active)):
        if active[i] and not active[i - 1]:
            count += 1
    return count

def run_branch(z, omega, K_values, branch_name, seed):
    rows = []

    for K in K_values:
        R_series = []
        cluster_series = []

        for t in range(steps_per_K):
            z = euler_step(z, omega, K, dt)

            if t >= int(discard_fraction * steps_per_K):
                R_series.append(order_parameter(z))
                cluster_series.append(cluster_count(np.angle(z)))

        rows.append({
            "seed": seed,
            "branch": branch_name,
            "K": K,
            "R_mean": np.mean(R_series),
            "R_std": np.std(R_series),
            "cluster_mean": np.mean(cluster_series),
            "cluster_std": np.std(cluster_series),
            "cluster_final": cluster_series[-1],
        })

    return z, rows

# -----------------------------
# MAIN LOOP
# -----------------------------
all_rows = []

for seed in seeds:
    print(f"\nSEED {seed}")
    rng = np.random.default_rng(seed)

    omega = rng.normal(0, omega_std, N)
    phases0 = rng.uniform(-np.pi, np.pi, N)
    amplitudes0 = 1.0 + sigma * rng.normal(size=N)
    z0 = amplitudes0 * np.exp(1j * phases0)

    z_after_up, rows_up = run_branch(z0.copy(), omega, K_up, "up", seed)
    z_after_down, rows_down = run_branch(z_after_up.copy(), omega, K_down, "down", seed)

    all_rows.extend(rows_up)
    all_rows.extend(rows_down)

all_df = pd.DataFrame(all_rows)
all_df.to_csv(OUTDIR / "hysteresis_all.csv", index=False)

# -----------------------------
# AGGREGATE
# -----------------------------
agg = (
    all_df
    .groupby(["branch", "K"], as_index=False)
    .agg(
        R_mean_mean=("R_mean", "mean"),
        R_mean_std=("R_mean", "std"),
        R_std_mean=("R_std", "mean"),
        cluster_mean_mean=("cluster_mean", "mean"),
        cluster_mean_std=("cluster_mean", "std"),
        cluster_final_mean=("cluster_final", "mean"),
    )
)

agg.to_csv(OUTDIR / "hysteresis_aggregate.csv", index=False)

# -----------------------------
# COMPARE UP vs DOWN
# -----------------------------
up = agg[agg["branch"] == "up"].copy()
down = agg[agg["branch"] == "down"].copy()

comp = pd.merge(
    up,
    down,
    on="K",
    suffixes=("_up", "_down")
)

comp["R_hysteresis_gap"] = comp["R_mean_mean_down"] - comp["R_mean_mean_up"]
comp["cluster_hysteresis_gap"] = comp["cluster_mean_mean_down"] - comp["cluster_mean_mean_up"]

comp.to_csv(OUTDIR / "hysteresis_comparison.csv", index=False)

# -----------------------------
# HYSTERESIS AREA
# -----------------------------
comp_sorted = comp.sort_values("K")

R_area = np.trapz(np.abs(comp_sorted["R_hysteresis_gap"]), comp_sorted["K"])
cluster_area = np.trapz(np.abs(comp_sorted["cluster_hysteresis_gap"]), comp_sorted["K"])

summary = pd.DataFrame([{
    "omega_std": omega_std,
    "N": N,
    "R_hysteresis_area": R_area,
    "cluster_hysteresis_area": cluster_area,
    "max_R_gap": comp_sorted["R_hysteresis_gap"].abs().max(),
    "K_at_max_R_gap": comp_sorted.loc[comp_sorted["R_hysteresis_gap"].abs().idxmax(), "K"],
    "max_cluster_gap": comp_sorted["cluster_hysteresis_gap"].abs().max(),
    "K_at_max_cluster_gap": comp_sorted.loc[comp_sorted["cluster_hysteresis_gap"].abs().idxmax(), "K"],
}])

summary.to_csv(OUTDIR / "hysteresis_summary.csv", index=False)

# -----------------------------
# PLOTS
# -----------------------------
fig, axes = plt.subplots(4, 1, figsize=(9, 14), sharex=True)

for branch, label in [("up", "K increasing"), ("down", "K decreasing")]:
    d = agg[agg["branch"] == branch].sort_values("K")
    axes[0].errorbar(d["K"], d["R_mean_mean"], yerr=d["R_mean_std"], marker="o", capsize=3, label=label)
    axes[1].plot(d["K"], d["cluster_mean_mean"], marker="o", label=label)
    axes[2].plot(d["K"], d["cluster_final_mean"], marker="o", label=label)

axes[0].set_ylabel("Mean R")
axes[0].set_title("Stuart–Landau hysteresis test: up vs down K")
axes[0].legend()
axes[0].grid(True)

axes[1].set_ylabel("Mean cluster count")
axes[1].legend()
axes[1].grid(True)

axes[2].set_ylabel("Final cluster count")
axes[2].legend()
axes[2].grid(True)

axes[3].plot(comp_sorted["K"], comp_sorted["R_hysteresis_gap"], marker="o", label="R down - R up")
axes[3].plot(comp_sorted["K"], comp_sorted["cluster_hysteresis_gap"], marker="o", label="cluster down - cluster up")
axes[3].axhline(0, linestyle="--")
axes[3].set_ylabel("Hysteresis gap")
axes[3].set_xlabel("Coupling K")
axes[3].legend()
axes[3].grid(True)

plt.tight_layout()
fig.savefig(OUTDIR / "fig_hysteresis_test.png", dpi=200)
plt.show()

print("\nSaved:")
print(OUTDIR / "hysteresis_all.csv")
print(OUTDIR / "hysteresis_aggregate.csv")
print(OUTDIR / "hysteresis_comparison.csv")
print(OUTDIR / "hysteresis_summary.csv")
print(OUTDIR / "fig_hysteresis_test.png")

print("\nSUMMARY")
display(summary)

print("\nCOMPARISON PREVIEW")
display(comp_sorted.head(40))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTDIR = Path("stuart_landau_basin_memory_output")
OUTDIR.mkdir(exist_ok=True)

# -----------------------------
# PARAMETERS
# -----------------------------
N = 400
omega_std = 0.40
seeds = range(20)

K_values = np.round(np.arange(0.40, 1.225, 0.025), 3)

dt = 0.02
T = 80.0
steps = int(T / dt)
discard_fraction = 0.5

lam = 1.0
beta = 0.0
amp_noise = 0.03

initial_conditions = [
    "random",
    "two_clusters",
    "three_clusters",
    "near_sync",
]

# -----------------------------
# HELPERS
# -----------------------------
def order_parameter(z):
    return np.abs(np.mean(np.exp(1j * np.angle(z))))

def euler_step(z, omega, K, dt):
    mean_z = np.mean(z)
    dz = (lam + 1j * omega - (1 + 1j * beta) * np.abs(z)**2) * z + K * (mean_z - z)
    return z + dt * dz

def cluster_count(phases, bins=36, threshold=0.03):
    hist, _ = np.histogram(phases, bins=bins, range=(-np.pi, np.pi), density=True)
    active = hist > threshold * np.max(hist)

    if active.sum() == 0:
        return 0

    count = 0
    for i in range(len(active)):
        if active[i] and not active[i - 1]:
            count += 1

    return count

def make_initial_state(kind, rng):
    amps = 1.0 + amp_noise * rng.normal(size=N)

    if kind == "random":
        phases = rng.uniform(-np.pi, np.pi, N)

    elif kind == "two_clusters":
        phases = np.zeros(N)
        phases[:N//2] = -0.8
        phases[N//2:] = 0.8
        phases += 0.15 * rng.normal(size=N)

    elif kind == "three_clusters":
        phases = np.zeros(N)
        third = N // 3
        phases[:third] = -1.5
        phases[third:2*third] = 0.0
        phases[2*third:] = 1.5
        phases += 0.15 * rng.normal(size=N)

    elif kind == "near_sync":
        phases = 0.10 * rng.normal(size=N)

    else:
        raise ValueError(kind)

    phases = (phases + np.pi) % (2*np.pi) - np.pi
    return amps * np.exp(1j * phases)

def run_one(seed, K, init_kind):
    rng = np.random.default_rng(seed)
    omega = rng.normal(0, omega_std, N)
    z = make_initial_state(init_kind, rng)

    R_series = []
    cluster_series = []

    for t in range(steps):
        z = euler_step(z, omega, K, dt)

        if t >= int(discard_fraction * steps):
            phases = np.angle(z)
            R_series.append(order_parameter(z))
            cluster_series.append(cluster_count(phases))

    return {
        "seed": seed,
        "K": K,
        "init_kind": init_kind,
        "R_mean": np.mean(R_series),
        "R_std": np.std(R_series),
        "cluster_mean": np.mean(cluster_series),
        "cluster_std": np.std(cluster_series),
        "cluster_final": cluster_series[-1],
        "single_cluster_fraction": np.mean(np.array(cluster_series) <= 1),
        "multi_cluster_fraction": np.mean(np.array(cluster_series) > 1),
    }

# -----------------------------
# MAIN LOOP
# -----------------------------
rows = []

for init_kind in initial_conditions:
    print(f"\n{'='*70}")
    print(f"INITIAL CONDITION: {init_kind}")
    print(f"{'='*70}")

    for seed in seeds:
        print(f"seed={seed}")

        for K in K_values:
            rows.append(run_one(seed, K, init_kind))

all_df = pd.DataFrame(rows)
all_df.to_csv(OUTDIR / "basin_memory_all.csv", index=False)

# -----------------------------
# AGGREGATE
# -----------------------------
agg = (
    all_df
    .groupby(["init_kind", "K"], as_index=False)
    .agg(
        R_mean_mean=("R_mean", "mean"),
        R_mean_std=("R_mean", "std"),
        cluster_mean_mean=("cluster_mean", "mean"),
        cluster_mean_std=("cluster_mean", "std"),
        cluster_final_mean=("cluster_final", "mean"),
        single_cluster_fraction_mean=("single_cluster_fraction", "mean"),
        multi_cluster_fraction_mean=("multi_cluster_fraction", "mean"),
    )
)

agg.to_csv(OUTDIR / "basin_memory_aggregate.csv", index=False)

# -----------------------------
# BASIN SPREAD PER K
# -----------------------------
spread_rows = []

for K, g in agg.groupby("K"):
    spread_rows.append({
        "K": K,
        "R_spread_across_initial_conditions": g["R_mean_mean"].max() - g["R_mean_mean"].min(),
        "cluster_spread_across_initial_conditions": g["cluster_mean_mean"].max() - g["cluster_mean_mean"].min(),
        "single_fraction_spread": g["single_cluster_fraction_mean"].max() - g["single_cluster_fraction_mean"].min(),
    })

spread_df = pd.DataFrame(spread_rows)
spread_df["basin_memory_score"] = (
    spread_df["R_spread_across_initial_conditions"] / spread_df["R_spread_across_initial_conditions"].max()
    + spread_df["cluster_spread_across_initial_conditions"] / spread_df["cluster_spread_across_initial_conditions"].max()
    + spread_df["single_fraction_spread"] / spread_df["single_fraction_spread"].max()
) / 3

spread_df.to_csv(OUTDIR / "basin_memory_spread.csv", index=False)

peak_row = spread_df.loc[spread_df["basin_memory_score"].idxmax()]
summary = pd.DataFrame([{
    "peak_K_basin_memory": peak_row["K"],
    "peak_basin_memory_score": peak_row["basin_memory_score"],
    "R_spread_at_peak": peak_row["R_spread_across_initial_conditions"],
    "cluster_spread_at_peak": peak_row["cluster_spread_across_initial_conditions"],
    "single_fraction_spread_at_peak": peak_row["single_fraction_spread"],
}])

summary.to_csv(OUTDIR / "basin_memory_summary.csv", index=False)

# -----------------------------
# PLOTS
# -----------------------------
fig, axes = plt.subplots(5, 1, figsize=(9, 16), sharex=True)

for init_kind in initial_conditions:
    d = agg[agg["init_kind"] == init_kind].sort_values("K")

    axes[0].errorbar(d["K"], d["R_mean_mean"], yerr=d["R_mean_std"], marker="o", capsize=2, label=init_kind)
    axes[1].errorbar(d["K"], d["cluster_mean_mean"], yerr=d["cluster_mean_std"], marker="o", capsize=2, label=init_kind)
    axes[2].plot(d["K"], d["single_cluster_fraction_mean"], marker="o", label=init_kind)
    axes[3].plot(d["K"], d["multi_cluster_fraction_mean"], marker="o", label=init_kind)

axes[0].set_ylabel("Mean R")
axes[0].set_title("Initial-condition / basin memory test")
axes[0].legend()
axes[0].grid(True)

axes[1].set_ylabel("Mean cluster count")
axes[1].legend()
axes[1].grid(True)

axes[2].set_ylabel("Single-cluster fraction")
axes[2].legend()
axes[2].grid(True)

axes[3].set_ylabel("Multi-cluster fraction")
axes[3].legend()
axes[3].grid(True)

axes[4].plot(spread_df["K"], spread_df["basin_memory_score"], marker="o")
axes[4].axvline(summary["peak_K_basin_memory"].iloc[0], linestyle="--", label=f"peak K={summary['peak_K_basin_memory'].iloc[0]:.3f}")
axes[4].set_ylabel("Basin memory score")
axes[4].set_xlabel("Coupling K")
axes[4].legend()
axes[4].grid(True)

plt.tight_layout()
fig.savefig(OUTDIR / "fig_basin_memory_test.png", dpi=200)
plt.show()

print("\nSaved:")
print(OUTDIR / "basin_memory_all.csv")
print(OUTDIR / "basin_memory_aggregate.csv")
print(OUTDIR / "basin_memory_spread.csv")
print(OUTDIR / "basin_memory_summary.csv")
print(OUTDIR / "fig_basin_memory_test.png")

print("\nSUMMARY")
display(summary)

print("\nSPREAD PREVIEW")
display(spread_df.head(40))

In [ ]:
# ============================================================
# STUART-LANDAU RECOVERY / PERTURBATION TEST
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from sklearn.cluster import DBSCAN
from IPython.display import display
from pathlib import Path

# ============================================================
# PARAMETERS
# ============================================================

N = 400
alpha = 1.0
dt = 0.02
steps_before = 3000
steps_after = 4000

K_values = np.round(np.arange(0.4, 1.201, 0.025), 3)

noise_strength = 0.35
recovery_threshold = 0.95

n_seeds = 20

output_dir = Path("stuart_landau_recovery_output")
output_dir.mkdir(exist_ok=True)

# ============================================================
# STUART-LANDAU STEP
# ============================================================

def step(z, omega, K):

    mean_field = np.mean(z)

    dz = (
        (alpha + 1j * omega - np.abs(z)**2) * z
        + K * (mean_field - z)
    )

    return z + dt * dz

# ============================================================
# ORDER PARAMETER
# ============================================================

def calc_R(z):

    phases = np.angle(z)

    return np.abs(np.mean(np.exp(1j * phases)))

# ============================================================
# CLUSTER COUNT
# ============================================================

def cluster_count(z):

    phases = np.angle(z)

    X = np.column_stack([
        np.cos(phases),
        np.sin(phases)
    ])

    clustering = DBSCAN(eps=0.15, min_samples=5).fit(X)

    labels = clustering.labels_

    unique = set(labels)

    if -1 in unique:
        unique.remove(-1)

    return len(unique)

# ============================================================
# RUN
# ============================================================

all_rows = []

for seed in range(n_seeds):

    print(f"seed={seed}")

    np.random.seed(seed)

    omega = np.random.normal(0, 0.4, N)

    for K in K_values:

        phases = np.random.uniform(-np.pi, np.pi, N)

        z = np.exp(1j * phases)

        # -------------------------
        # PRE-STABILIZATION
        # -------------------------

        for _ in range(steps_before):
            z = step(z, omega, K)

        R_before = calc_R(z)

        # -------------------------
        # PERTURBATION
        # -------------------------

        perturb = noise_strength * (
            np.random.normal(size=N)
            + 1j * np.random.normal(size=N)
        )

        z = z + perturb

        # -------------------------
        # RECOVERY
        # -------------------------

        recovery_time = np.nan

        R_series = []

        for t in range(steps_after):

            z = step(z, omega, K)

            R = calc_R(z)

            R_series.append(R)

            if np.isnan(recovery_time):

                if R >= recovery_threshold * R_before:
                    recovery_time = t * dt

        if np.isnan(recovery_time):
            recovery_time = steps_after * dt

        row = {
            "seed": seed,
            "K": K,
            "R_before": R_before,
            "R_after_final": R_series[-1],
            "R_min_after_perturb": np.min(R_series),
            "recovery_time": recovery_time,
            "cluster_final": cluster_count(z)
        }

        all_rows.append(row)

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(all_rows)

agg = df.groupby("K").agg({
    "R_before": ["mean", "std"],
    "R_after_final": ["mean", "std"],
    "R_min_after_perturb": ["mean", "std"],
    "recovery_time": ["mean", "std"],
    "cluster_final": ["mean", "std"]
})

agg.columns = ["_".join(c) for c in agg.columns]

agg = agg.reset_index()

# ============================================================
# DERIVATIVES
# ============================================================

agg["d_recovery_dK"] = np.gradient(
    agg["recovery_time_mean"],
    agg["K"]
)

agg["abs_d_recovery_dK"] = np.abs(
    agg["d_recovery_dK"]
)

# ============================================================
# PEAK
# ============================================================

peak_idx = agg["recovery_time_mean"].idxmax()

peak_K = agg.loc[peak_idx, "K"]

peak_recovery = agg.loc[peak_idx, "recovery_time_mean"]

summary = pd.DataFrame([{
    "peak_K_recovery_time": peak_K,
    "peak_recovery_time": peak_recovery,
    "R_before_at_peak": agg.loc[peak_idx, "R_before_mean"],
    "R_min_after_perturb_at_peak":
        agg.loc[peak_idx, "R_min_after_perturb_mean"]
}])

# ============================================================
# PLOT
# ============================================================

fig, axes = plt.subplots(
    5, 1,
    figsize=(10, 20),
    sharex=True
)

# ------------------------------------------------

axes[0].errorbar(
    agg["K"],
    agg["R_before_mean"],
    yerr=agg["R_before_std"],
    marker='o'
)

axes[0].set_ylabel("Mean R")
axes[0].set_title("Recovery / perturbation test")

# ------------------------------------------------

axes[1].errorbar(
    agg["K"],
    agg["R_min_after_perturb_mean"],
    yerr=agg["R_min_after_perturb_std"],
    marker='o'
)

axes[1].set_ylabel("Min R after perturb")

# ------------------------------------------------

axes[2].errorbar(
    agg["K"],
    agg["recovery_time_mean"],
    yerr=agg["recovery_time_std"],
    marker='o'
)

axes[2].axvline(
    peak_K,
    linestyle='--'
)

axes[2].set_ylabel("Recovery time")

# ------------------------------------------------

axes[3].plot(
    agg["K"],
    agg["abs_d_recovery_dK"],
    marker='o'
)

axes[3].axvline(
    peak_K,
    linestyle='--'
)

axes[3].set_ylabel("|d recovery / dK|")

# ------------------------------------------------

axes[4].errorbar(
    agg["K"],
    agg["cluster_final_mean"],
    yerr=agg["cluster_final_std"],
    marker='o'
)

axes[4].set_ylabel("Final cluster count")
axes[4].set_xlabel("Coupling K")

# ------------------------------------------------

plt.tight_layout()

fig_path = output_dir / "fig_recovery_test.png"

plt.savefig(fig_path, dpi=300)

plt.close()

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    output_dir / "recovery_all.csv",
    index=False
)

agg.to_csv(
    output_dir / "recovery_aggregate.csv",
    index=False
)

summary.to_csv(
    output_dir / "recovery_summary.csv",
    index=False
)

# ============================================================
# OUTPUT
# ============================================================

print("\nSaved:")
print(output_dir / "recovery_all.csv")
print(output_dir / "recovery_aggregate.csv")
print(output_dir / "recovery_summary.csv")
print(fig_path)

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(40))

In [ ]:
# ============================================================
# STUART–LANDAU — CRITICAL SLOWING DOWN / EARLY WARNING TEST
# N=400, omega_std=0.40
# Tests: variance, lag-1 autocorrelation, recovery proxy, fluctuation peak
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------
# CONFIG
# ----------------------------
OUTDIR = Path("stuart_landau_critical_slowing_output")
OUTDIR.mkdir(exist_ok=True)

N = 400
OMEGA_STD = 0.40
SEEDS = range(20)

K_VALUES = np.round(np.arange(0.40, 1.201, 0.025), 3)

dt = 0.02
T_total = 80.0
burn_frac = 0.35

a = 1.0
b = 1.0
noise_amp = 0.00

# window for late-time early warning indicators
ROLL_WINDOW_FRAC = 0.50

np.random.seed(123)

# ----------------------------
# FUNCTIONS
# ----------------------------

def order_parameter(z):
    theta = np.angle(z)
    return np.abs(np.mean(np.exp(1j * theta)))

def simulate_stuart_landau(N, K, omega_std, seed):
    rng = np.random.default_rng(seed)

    omega = rng.normal(0.0, omega_std, N)
    theta0 = rng.uniform(0, 2*np.pi, N)
    r0 = 1.0 + 0.05 * rng.normal(size=N)

    z = r0 * np.exp(1j * theta0)

    steps = int(T_total / dt)
    R_series = np.zeros(steps)

    for t in range(steps):
        mean_z = np.mean(z)
        dz = (a + 1j*omega - (1 + 1j*b) * np.abs(z)**2) * z + K * (mean_z - z)

        if noise_amp > 0:
            dz += noise_amp * (rng.normal(size=N) + 1j*rng.normal(size=N))

        z = z + dt * dz
        R_series[t] = order_parameter(z)

    return R_series

def safe_var(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return np.nan
    return np.var(x, ddof=1)

def lag1_autocorr(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if len(x) < 5:
        return np.nan

    x0 = x[:-1]
    x1 = x[1:]

    if np.std(x0) == 0 or np.std(x1) == 0:
        return np.nan

    return np.corrcoef(x0, x1)[0, 1]

def recovery_proxy(x):
    """
    Proxy: mean absolute return speed after local deviations.
    Smaller value can indicate slower recovery / critical slowing.
    """
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    if len(x) < 5:
        return np.nan

    dx = np.diff(x)
    dev = x[:-1] - np.mean(x)

    denom = np.mean(np.abs(dev))
    if denom == 0 or not np.isfinite(denom):
        return np.nan

    return np.mean(np.abs(dx)) / denom

def rolling_stats(x, window):
    x = pd.Series(x)
    return {
        "rolling_var_mean": x.rolling(window).var().mean(),
        "rolling_var_max": x.rolling(window).var().max(),
        "rolling_ac1_mean": x.rolling(window).apply(
            lambda y: lag1_autocorr(y), raw=False
        ).mean()
    }

# ----------------------------
# MAIN LOOP
# ----------------------------

rows = []

for seed in SEEDS:
    print(f"\nSEED {seed}")

    for K in K_VALUES:
        print(f"K={K}")

        R = simulate_stuart_landau(
            N=N,
            K=K,
            omega_std=OMEGA_STD,
            seed=seed
        )

        burn = int(len(R) * burn_frac)
        R_post = R[burn:]

        late_start = int(len(R_post) * (1 - ROLL_WINDOW_FRAC))
        R_late = R_post[late_start:]

        window = max(20, int(len(R_late) * 0.10))

        roll = rolling_stats(R_late, window)

        rows.append({
            "seed": seed,
            "K": K,
            "R_mean": np.nanmean(R_late),
            "R_std": np.nanstd(R_late, ddof=1),
            "R_var": safe_var(R_late),
            "R_ac1": lag1_autocorr(R_late),
            "recovery_proxy": recovery_proxy(R_late),
            "R_min": np.nanmin(R_late),
            "R_max": np.nanmax(R_late),
            "R_range": np.nanmax(R_late) - np.nanmin(R_late),
            "rolling_var_mean": roll["rolling_var_mean"],
            "rolling_var_max": roll["rolling_var_max"],
            "rolling_ac1_mean": roll["rolling_ac1_mean"],
        })

all_df = pd.DataFrame(rows)
all_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# ----------------------------
# AGGREGATE
# ----------------------------

agg = all_df.groupby("K").agg(
    R_mean_mean=("R_mean", "mean"),
    R_mean_std=("R_mean", "std"),
    R_std_mean=("R_std", "mean"),
    R_std_std=("R_std", "std"),
    R_var_mean=("R_var", "mean"),
    R_var_std=("R_var", "std"),
    R_ac1_mean=("R_ac1", "mean"),
    R_ac1_std=("R_ac1", "std"),
    recovery_proxy_mean=("recovery_proxy", "mean"),
    recovery_proxy_std=("recovery_proxy", "std"),
    R_range_mean=("R_range", "mean"),
    R_range_std=("R_range", "std"),
    rolling_var_mean=("rolling_var_mean", "mean"),
    rolling_var_max_mean=("rolling_var_max", "mean"),
    rolling_ac1_mean=("rolling_ac1_mean", "mean"),
).reset_index()

# derivatives / peak scores
agg["dR_dK"] = np.gradient(agg["R_mean_mean"], agg["K"])
agg["dVar_dK"] = np.gradient(agg["R_var_mean"], agg["K"])
agg["dAC1_dK"] = np.gradient(agg["R_ac1_mean"], agg["K"])
agg["dRecoveryProxy_dK"] = np.gradient(agg["recovery_proxy_mean"], agg["K"])

# normalize helper
def norm01(x):
    x = np.asarray(x, dtype=float)
    if np.all(~np.isfinite(x)):
        return np.full_like(x, np.nan)
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if mx - mn == 0:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

# high variance + high autocorrelation + large change + low recovery proxy
agg["critical_slowing_score"] = (
    norm01(agg["R_var_mean"]) +
    norm01(agg["R_ac1_mean"]) +
    norm01(np.abs(agg["dR_dK"])) +
    norm01(1 / (agg["recovery_proxy_mean"] + 1e-12))
) / 4

peak_idx = agg["critical_slowing_score"].idxmax()
peak_row = agg.loc[peak_idx]

summary = pd.DataFrame([{
    "omega_std": OMEGA_STD,
    "N": N,
    "peak_K_critical_slowing": peak_row["K"],
    "peak_critical_slowing_score": peak_row["critical_slowing_score"],
    "R_mean_at_peak": peak_row["R_mean_mean"],
    "R_var_at_peak": peak_row["R_var_mean"],
    "R_ac1_at_peak": peak_row["R_ac1_mean"],
    "recovery_proxy_at_peak": peak_row["recovery_proxy_mean"],
    "dR_dK_at_peak": peak_row["dR_dK"],
}])

# ----------------------------
# SAVE CSV
# ----------------------------

all_path = OUTDIR / "critical_slowing_all.csv"
agg_path = OUTDIR / "critical_slowing_aggregate.csv"
summary_path = OUTDIR / "critical_slowing_summary.csv"

all_df.to_csv(all_path, index=False)
agg.to_csv(agg_path, index=False)
summary.to_csv(summary_path, index=False)

# ----------------------------
# PLOTS
# ----------------------------

fig, axes = plt.subplots(6, 1, figsize=(10, 18), sharex=True)

peak_K = peak_row["K"]

axes[0].errorbar(agg["K"], agg["R_mean_mean"], yerr=agg["R_mean_std"], marker="o", capsize=3)
axes[0].axvline(peak_K, linestyle="--")
axes[0].set_ylabel("Mean R")
axes[0].set_title("Critical slowing down / early warning indicators")

axes[1].errorbar(agg["K"], agg["R_var_mean"], yerr=agg["R_var_std"], marker="o", capsize=3)
axes[1].axvline(peak_K, linestyle="--")
axes[1].set_ylabel("Variance of R")

axes[2].errorbar(agg["K"], agg["R_ac1_mean"], yerr=agg["R_ac1_std"], marker="o", capsize=3)
axes[2].axvline(peak_K, linestyle="--")
axes[2].set_ylabel("Lag-1 autocorr")

axes[3].errorbar(agg["K"], agg["recovery_proxy_mean"], yerr=agg["recovery_proxy_std"], marker="o", capsize=3)
axes[3].axvline(peak_K, linestyle="--")
axes[3].set_ylabel("Recovery proxy")

axes[4].plot(agg["K"], np.abs(agg["dR_dK"]), marker="o", label="|dR/dK|")
axes[4].plot(agg["K"], np.abs(agg["dVar_dK"]), marker="o", label="|dVar/dK|")
axes[4].plot(agg["K"], np.abs(agg["dAC1_dK"]), marker="o", label="|dAC1/dK|")
axes[4].axvline(peak_K, linestyle="--")
axes[4].set_ylabel("Transition slope")
axes[4].legend()

axes[5].plot(agg["K"], agg["critical_slowing_score"], marker="o")
axes[5].axvline(peak_K, linestyle="--", label=f"peak K={peak_K:.3f}")
axes[5].set_ylabel("Critical slowing score")
axes[5].set_xlabel("Coupling K")
axes[5].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()

fig_path = OUTDIR / "fig_critical_slowing_test.png"
plt.savefig(fig_path, dpi=200)
plt.show()

# ----------------------------
# DISPLAY
# ----------------------------

print("\nSaved:")
print(all_path)
print(agg_path)
print(summary_path)
print(fig_path)

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(40))

In [ ]:
from pathlib import Path
import zipfile

BASE = Path(".")
ZIP_NAME = "stuart_landau_all_results_backup.zip"

# foldery z resultami
result_dirs = [
    p for p in BASE.iterdir()
    if p.is_dir() and p.name.startswith("stuart_landau")
]

print("Found result folders:")
for p in result_dirs:
    print(" -", p)

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in result_dirs:
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(BASE))

zip_path = Path(ZIP_NAME)
print("\nZIP created:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / (1024*1024), 2))

# Output files remain in the local results directory.

In [ ]:
# ============================================================
# STUART–LANDAU — NOISE ROBUSTNESS TEST
# Does the reorganization window persist under noise?
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTDIR = Path("stuart_landau_noise_robustness_output")
OUTDIR.mkdir(exist_ok=True)

# ----------------------------
# PARAMETERS
# ----------------------------

N = 400
omega_std = 0.40

K_values = np.round(np.arange(0.40, 1.2001, 0.025), 3)
noise_values = [0.0, 0.005, 0.01, 0.02, 0.04]

seeds = range(20)

dt = 0.02
T = 50.0
steps = int(T / dt)
transient = int(0.5 * steps)

cluster_bins = 36


# ----------------------------
# FUNCTIONS
# ----------------------------

def simulate_stuart_landau(N, K, omega_std, noise_sigma, seed):
    rng = np.random.default_rng(seed)

    theta = rng.uniform(0, 2*np.pi, N)
    r = 1.0 + 0.05 * rng.normal(size=N)
    z = r * np.exp(1j * theta)

    omega = rng.normal(0.0, omega_std, N)

    R_series = []
    cluster_series = []

    for t in range(steps):
        mean_z = np.mean(z)

        dz = (1 + 1j * omega - np.abs(z)**2) * z + K * (mean_z - z)

        if noise_sigma > 0:
            noise = noise_sigma * np.sqrt(dt) * (
                rng.normal(size=N) + 1j * rng.normal(size=N)
            )
        else:
            noise = 0.0

        z = z + dt * dz + noise

        if t >= transient:
            phase = np.angle(z)
            R = np.abs(np.mean(np.exp(1j * phase)))
            R_series.append(R)

            hist, _ = np.histogram(phase, bins=cluster_bins, range=(-np.pi, np.pi))
            hist = hist / hist.sum()
            active_bins = np.sum(hist > 0.03)
            cluster_series.append(active_bins)

    R_series = np.array(R_series)
    cluster_series = np.array(cluster_series)

    return R_series, cluster_series


def summarize_run(R_series, cluster_series):
    R_mean = np.mean(R_series)
    R_std = np.std(R_series)

    cluster_mean = np.mean(cluster_series)
    cluster_std = np.std(cluster_series)

    multi_cluster_fraction = np.mean(cluster_series > 1)
    single_cluster_fraction = np.mean(cluster_series <= 1)

    switch_rate = np.mean(np.abs(np.diff(cluster_series)) > 0)

    return {
        "R_mean": R_mean,
        "R_std": R_std,
        "cluster_mean": cluster_mean,
        "cluster_std": cluster_std,
        "multi_cluster_fraction": multi_cluster_fraction,
        "single_cluster_fraction": single_cluster_fraction,
        "cluster_switch_rate": switch_rate,
    }


# ----------------------------
# RUN TEST
# ----------------------------

rows = []

for noise_sigma in noise_values:
    print("\n" + "="*70)
    print(f"NOISE SIGMA = {noise_sigma}")
    print("="*70)

    for seed in seeds:
        print(f"seed={seed}")

        for K in K_values:
            R_series, cluster_series = simulate_stuart_landau(
                N=N,
                K=K,
                omega_std=omega_std,
                noise_sigma=noise_sigma,
                seed=seed
            )

            s = summarize_run(R_series, cluster_series)

            rows.append({
                "noise_sigma": noise_sigma,
                "seed": seed,
                "K": K,
                **s
            })


all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby(["noise_sigma", "K"])
    .agg(
        R_mean_mean=("R_mean", "mean"),
        R_mean_std=("R_mean", "std"),
        R_std_mean=("R_std", "mean"),
        cluster_mean_mean=("cluster_mean", "mean"),
        cluster_mean_std=("cluster_mean", "std"),
        cluster_std_mean=("cluster_std", "mean"),
        multi_cluster_fraction_mean=("multi_cluster_fraction", "mean"),
        single_cluster_fraction_mean=("single_cluster_fraction", "mean"),
        cluster_switch_rate_mean=("cluster_switch_rate", "mean"),
    )
    .reset_index()
)

# derivatives and fluctuation score per noise level
agg_list = []

for noise_sigma, df in agg.groupby("noise_sigma"):
    df = df.sort_values("K").copy()

    df["dR_dK"] = np.gradient(df["R_mean_mean"], df["K"])
    df["dcluster_dK"] = np.gradient(df["cluster_mean_mean"], df["K"])
    df["dmulti_dK"] = np.gradient(df["multi_cluster_fraction_mean"], df["K"])

    def norm(x):
        x = np.asarray(x)
        if np.max(np.abs(x)) == 0:
            return np.zeros_like(x)
        return np.abs(x) / np.max(np.abs(x))

    df["fluctuation_score"] = (
        0.35 * norm(df["dR_dK"]) +
        0.35 * norm(df["dcluster_dK"]) +
        0.20 * norm(df["dmulti_dK"]) +
        0.10 * norm(df["cluster_switch_rate_mean"])
    )

    agg_list.append(df)

agg = pd.concat(agg_list, ignore_index=True)

summary_rows = []

for noise_sigma, df in agg.groupby("noise_sigma"):
    idx = df["fluctuation_score"].idxmax()
    row = df.loc[idx]

    summary_rows.append({
        "noise_sigma": noise_sigma,
        "peak_K_fluctuation_score": row["K"],
        "peak_fluctuation_score": row["fluctuation_score"],
        "R_mean_at_peak": row["R_mean_mean"],
        "R_std_at_peak": row["R_std_mean"],
        "cluster_mean_at_peak": row["cluster_mean_mean"],
        "cluster_std_at_peak": row["cluster_std_mean"],
        "multi_cluster_fraction_at_peak": row["multi_cluster_fraction_mean"],
        "single_cluster_fraction_at_peak": row["single_cluster_fraction_mean"],
        "cluster_switch_rate_at_peak": row["cluster_switch_rate_mean"],
    })

summary = pd.DataFrame(summary_rows)


# ----------------------------
# SAVE CSV
# ----------------------------

all_df.to_csv(OUTDIR / "noise_robustness_all.csv", index=False)
agg.to_csv(OUTDIR / "noise_robustness_aggregate.csv", index=False)
summary.to_csv(OUTDIR / "noise_robustness_summary.csv", index=False)


# ----------------------------
# PLOTS
# ----------------------------

fig, axes = plt.subplots(5, 1, figsize=(10, 16), sharex=True)

for noise_sigma, df in agg.groupby("noise_sigma"):
    label = f"noise={noise_sigma}"

    axes[0].plot(df["K"], df["R_mean_mean"], marker="o", label=label)
    axes[1].plot(df["K"], df["cluster_mean_mean"], marker="o", label=label)
    axes[2].plot(df["K"], df["multi_cluster_fraction_mean"], marker="o", label=label)
    axes[3].plot(df["K"], df["cluster_switch_rate_mean"], marker="o", label=label)
    axes[4].plot(df["K"], df["fluctuation_score"], marker="o", label=label)

axes[0].set_ylabel("Mean R")
axes[1].set_ylabel("Mean cluster count")
axes[2].set_ylabel("Multi-cluster fraction")
axes[3].set_ylabel("Switch rate")
axes[4].set_ylabel("Fluctuation score")
axes[4].set_xlabel("Coupling K")

axes[0].set_title("Stuart–Landau noise robustness test")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(OUTDIR / "fig_noise_robustness.png", dpi=200)
plt.show()


# threshold shift plot
plt.figure(figsize=(8, 5))
plt.plot(
    summary["noise_sigma"],
    summary["peak_K_fluctuation_score"],
    marker="o"
)
plt.xlabel("Noise sigma")
plt.ylabel("Peak K")
plt.title("Shift of reorganization peak under noise")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTDIR / "fig_noise_peak_shift.png", dpi=200)
plt.show()


print("\nSaved:")
print(OUTDIR / "noise_robustness_all.csv")
print(OUTDIR / "noise_robustness_aggregate.csv")
print(OUTDIR / "noise_robustness_summary.csv")
print(OUTDIR / "fig_noise_robustness.png")
print(OUTDIR / "fig_noise_peak_shift.png")

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(40))

In [ ]:
# ============================================================
# STUART-LANDAU HYSTERESIS / MEMORY LOOP TEST
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from tqdm import tqdm
import os

# ============================================================
# PARAMETERS
# ============================================================

N = 400
dt = 0.02
steps_per_K = 1200
discard = 400

omega_std = 0.4
a = 1.0

K_forward = np.arange(0.4, 1.201, 0.025)
K_backward = K_forward[::-1]

seeds = range(20)

output_dir = "stuart_landau_hysteresis_output"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# STUART-LANDAU STEP
# ============================================================

def step_stuart_landau(z, omega, K):

    mean_field = np.mean(z)

    dz = (
        (a + 1j * omega - np.abs(z)**2) * z
        + K * (mean_field - z)
    )

    return z + dt * dz

# ============================================================
# ORDER PARAMETER
# ============================================================

def compute_R(z):

    phases = np.angle(z)

    return np.abs(np.mean(np.exp(1j * phases)))

# ============================================================
# MAIN LOOP
# ============================================================

rows = []

for seed in tqdm(seeds):

    np.random.seed(seed)

    omega = np.random.normal(0, omega_std, N)

    z = (
        np.random.normal(0, 1, N)
        + 1j * np.random.normal(0, 1, N)
    )

    z = z / np.abs(z)

    # ========================================================
    # FORWARD
    # ========================================================

    for direction, K_values in [
        ("forward", K_forward),
        ("backward", K_backward)
    ]:

        for K in K_values:

            R_series = []

            for t in range(steps_per_K):

                z = step_stuart_landau(z, omega, K)

                if t >= discard:

                    R_series.append(compute_R(z))

            R_series = np.array(R_series)

            rows.append({
                "seed": seed,
                "direction": direction,
                "K": K,
                "R_mean": np.mean(R_series),
                "R_std": np.std(R_series)
            })

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(rows)

df.to_csv(
    f"{output_dir}/hysteresis_all.csv",
    index=False
)

# ============================================================
# AGGREGATE
# ============================================================

agg = (
    df.groupby(["direction", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"]
    })
)

agg.columns = [
    "_".join(col)
    for col in agg.columns
]

agg = agg.reset_index()

agg.to_csv(
    f"{output_dir}/hysteresis_aggregate.csv",
    index=False
)

# ============================================================
# HYSTERESIS SCORE
# ============================================================

forward = agg[agg["direction"] == "forward"]
backward = agg[agg["direction"] == "backward"]

score = np.mean(
    np.abs(
        forward["R_mean_mean"].values
        - backward["R_mean_mean"].values
    )
)

summary = pd.DataFrame([{
    "hysteresis_score": score
}])

summary.to_csv(
    f"{output_dir}/hysteresis_summary.csv",
    index=False
)

# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(8,6))

plt.plot(
    forward["K"],
    forward["R_mean_mean"],
    label="forward"
)

plt.plot(
    backward["K"],
    backward["R_mean_mean"],
    label="backward"
)

plt.xlabel("Coupling K")
plt.ylabel("Mean R")
plt.title("Stuart-Landau hysteresis test")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    f"{output_dir}/fig_hysteresis.png",
    dpi=300
)

plt.show()

# ============================================================
# SAVE INFO
# ============================================================

print("\nSaved:")
print(f"{output_dir}/hysteresis_all.csv")
print(f"{output_dir}/hysteresis_aggregate.csv")
print(f"{output_dir}/hysteresis_summary.csv")
print(f"{output_dir}/fig_hysteresis.png")

print("\nSUMMARY")
print(summary)

In [ ]:
# CLUSTER HYSTERESIS TEST — forward vs backward, with cluster-memory metrics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

OUTDIR = Path("stuart_landau_cluster_hysteresis_output")
OUTDIR.mkdir(exist_ok=True)

N = 400
omega_std = 0.4
seeds = range(20)

K_forward = np.round(np.arange(0.4, 1.201, 0.025), 3)
K_backward = K_forward[::-1]

dt = 0.02
T = 80
steps = int(T / dt)
burn = int(30 / dt)

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def cluster_count(theta, bins=36, threshold=0.03):
    phases = (theta + np.pi) % (2*np.pi) - np.pi
    hist, _ = np.histogram(phases, bins=bins, range=(-np.pi, np.pi), density=True)
    active = hist > threshold
    count = 0
    in_cluster = False
    for a in np.r_[active, active[0]]:
        if a and not in_cluster:
            count += 1
            in_cluster = True
        elif not a:
            in_cluster = False
    return max(count, 1)

def simulate_branch(K_values, seed, branch):
    rng = np.random.default_rng(seed)
    omega = rng.normal(0, omega_std, N)
    theta = rng.uniform(-np.pi, np.pi, N)

    rows = []

    for K in K_values:
        R_series = []
        cluster_series = []

        for t in range(steps):
            z = np.mean(np.exp(1j * theta))
            coupling = K * np.imag(z * np.exp(-1j * theta))
            theta += dt * (omega + coupling)

            if t >= burn:
                R_series.append(order_parameter(theta))
                cluster_series.append(cluster_count(theta))

        R_series = np.array(R_series)
        cluster_series = np.array(cluster_series)

        rows.append({
            "seed": seed,
            "branch": branch,
            "K": K,
            "R_mean": R_series.mean(),
            "R_std": R_series.std(),
            "cluster_mean": cluster_series.mean(),
            "cluster_std": cluster_series.std(),
            "cluster_final": cluster_series[-1],
            "single_cluster_fraction": np.mean(cluster_series <= 1.1),
            "multi_cluster_fraction": np.mean(cluster_series > 1.1),
            "switch_rate": np.mean(np.abs(np.diff(cluster_series)) > 0)
        })

    return rows

all_rows = []

for seed in tqdm(seeds):
    all_rows.extend(simulate_branch(K_forward, seed, "forward"))
    all_rows.extend(simulate_branch(K_backward, seed, "backward"))

all_df = pd.DataFrame(all_rows)
all_df.to_csv(OUTDIR / "cluster_hysteresis_all.csv", index=False)

agg = (
    all_df
    .groupby(["branch", "K"])
    .agg(
        R_mean=("R_mean", "mean"),
        R_std=("R_mean", "std"),
        cluster_mean=("cluster_mean", "mean"),
        cluster_std=("cluster_mean", "std"),
        single_cluster_fraction=("single_cluster_fraction", "mean"),
        multi_cluster_fraction=("multi_cluster_fraction", "mean"),
        switch_rate=("switch_rate", "mean")
    )
    .reset_index()
)

agg.to_csv(OUTDIR / "cluster_hysteresis_aggregate.csv", index=False)

fwd = agg[agg["branch"] == "forward"].copy()
bwd = agg[agg["branch"] == "backward"].copy()

comp = pd.merge(
    fwd,
    bwd,
    on="K",
    suffixes=("_forward", "_backward")
)

for col in ["R_mean", "cluster_mean", "single_cluster_fraction", "multi_cluster_fraction", "switch_rate"]:
    comp[f"{col}_gap"] = comp[f"{col}_backward"] - comp[f"{col}_forward"]

comp["abs_cluster_gap"] = comp["cluster_mean_gap"].abs()
comp["abs_single_gap"] = comp["single_cluster_fraction_gap"].abs()
comp["abs_multi_gap"] = comp["multi_cluster_fraction_gap"].abs()
comp["abs_switch_gap"] = comp["switch_rate_gap"].abs()

comp.to_csv(OUTDIR / "cluster_hysteresis_comparison.csv", index=False)

summary = pd.DataFrame([{
    "omega_std": omega_std,
    "N": N,
    "max_cluster_gap": comp["abs_cluster_gap"].max(),
    "K_at_max_cluster_gap": comp.loc[comp["abs_cluster_gap"].idxmax(), "K"],
    "max_single_fraction_gap": comp["abs_single_gap"].max(),
    "K_at_max_single_gap": comp.loc[comp["abs_single_gap"].idxmax(), "K"],
    "max_switch_rate_gap": comp["abs_switch_gap"].max(),
    "K_at_max_switch_gap": comp.loc[comp["abs_switch_gap"].idxmax(), "K"],
    "R_gap_area": np.trapezoid(np.abs(comp["R_mean_gap"]), comp["K"]),
    "cluster_gap_area": np.trapezoid(np.abs(comp["cluster_mean_gap"]), comp["K"]),
    "single_fraction_gap_area": np.trapezoid(np.abs(comp["single_cluster_fraction_gap"]), comp["K"]),
    "switch_rate_gap_area": np.trapezoid(np.abs(comp["switch_rate_gap"]), comp["K"]),
}])

summary.to_csv(OUTDIR / "cluster_hysteresis_summary.csv", index=False)

fig, axes = plt.subplots(5, 1, figsize=(9, 14), sharex=True)

for branch in ["forward", "backward"]:
    d = agg[agg["branch"] == branch].sort_values("K")
    axes[0].plot(d["K"], d["R_mean"], marker="o", label=branch)
    axes[1].plot(d["K"], d["cluster_mean"], marker="o", label=branch)
    axes[2].plot(d["K"], d["single_cluster_fraction"], marker="o", label=branch)
    axes[3].plot(d["K"], d["multi_cluster_fraction"], marker="o", label=branch)
    axes[4].plot(d["K"], d["switch_rate"], marker="o", label=branch)

axes[0].set_ylabel("Mean R")
axes[1].set_ylabel("Cluster count")
axes[2].set_ylabel("Single-cluster fraction")
axes[3].set_ylabel("Multi-cluster fraction")
axes[4].set_ylabel("Switch rate")
axes[4].set_xlabel("Coupling K")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("Stuart–Landau / Kuramoto cluster hysteresis test")
plt.tight_layout()
plt.savefig(OUTDIR / "fig_cluster_hysteresis.png", dpi=200)
plt.show()

print("Saved:")
print(OUTDIR / "cluster_hysteresis_all.csv")
print(OUTDIR / "cluster_hysteresis_aggregate.csv")
print(OUTDIR / "cluster_hysteresis_comparison.csv")
print(OUTDIR / "cluster_hysteresis_summary.csv")
print(OUTDIR / "fig_cluster_hysteresis.png")

print("\nSUMMARY")
display(summary)

print("\nCOMPARISON PREVIEW")
display(comp.head(40))

In [ ]:
# ============================================================
# STUART-LANDAU / KURAMOTO
# LOCAL MEMORY MAP TEST
# which oscillators remember transition direction?
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm
import os

# ============================================================
# PARAMETERS
# ============================================================

N = 400
omega_std = 0.4

K_values = np.arange(0.4, 1.201, 0.025)

dt = 0.02
steps = 6000
discard = 3000

cluster_threshold = 0.35

seed = 42

# ============================================================
# OUTPUT
# ============================================================

output_dir = "stuart_landau_local_memory_map_output"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# INITIAL CONDITIONS
# ============================================================

np.random.seed(seed)

omega = np.random.normal(0, omega_std, N)

z0 = (
    np.random.normal(0, 1, N)
    + 1j * np.random.normal(0, 1, N)
)

# ============================================================
# MODEL
# ============================================================

def simulate_branch(K_sequence, z_init):

    z = z_init.copy()

    results = []

    final_states = {}

    for K in tqdm(K_sequence):

        for _ in range(steps):

            mean_field = np.mean(z)

            dz = (
                (1 + 1j * omega - np.abs(z)**2) * z
                + K * (mean_field - z)
            )

            z += dt * dz

        z_tail = z.copy()

        phases = np.angle(z_tail)

        # pairwise phase distances
        dmat = squareform(pdist(phases.reshape(-1,1)))

        adjacency = (dmat < cluster_threshold).astype(int)

        cluster_size = adjacency.sum(axis=1)

        results.append({
            "K": K,
            "mean_cluster_size": np.mean(cluster_size)
        })

        final_states[K] = {
            "phase": phases.copy(),
            "cluster_size": cluster_size.copy()
        }

    return pd.DataFrame(results), final_states

# ============================================================
# FORWARD
# ============================================================

print("\nFORWARD\n")

forward_df, forward_states = simulate_branch(
    K_values,
    z0.copy()
)

# ============================================================
# BACKWARD
# ============================================================

print("\nBACKWARD\n")

K_backward = K_values[::-1]

last_forward_state = (
    np.abs(z0) * np.exp(1j * forward_states[K_values[-1]]["phase"])
)

backward_df, backward_states = simulate_branch(
    K_backward,
    last_forward_state.copy()
)

# ============================================================
# ALIGN BACKWARD ORDER
# ============================================================

backward_states_aligned = {}

for K in K_values:
    backward_states_aligned[K] = backward_states[K]

# ============================================================
# LOCAL MEMORY MAP
# ============================================================

memory_map = []

for K in K_values:

    cf = forward_states[K]["cluster_size"]
    cb = backward_states_aligned[K]["cluster_size"]

    diff = np.abs(cf - cb)

    memory_map.append(diff)

memory_map = np.array(memory_map)

# ============================================================
# SUMMARY METRICS
# ============================================================

mean_memory_vs_K = memory_map.mean(axis=1)

peak_idx = np.argmax(mean_memory_vs_K)

summary = pd.DataFrame({
    "peak_memory_K": [K_values[peak_idx]],
    "peak_memory_strength": [mean_memory_vs_K[peak_idx]]
})

# ============================================================
# SAVE
# ============================================================

memory_df = pd.DataFrame(
    memory_map,
    index=np.round(K_values, 3)
)

memory_df.to_csv(
    f"{output_dir}/local_memory_map.csv"
)

summary.to_csv(
    f"{output_dir}/local_memory_summary.csv",
    index=False
)

# ============================================================
# PLOT 1
# ============================================================

plt.figure(figsize=(12,8))

plt.imshow(
    memory_map.T,
    aspect='auto',
    origin='lower',
    extent=[
        K_values[0],
        K_values[-1],
        0,
        N
    ]
)

plt.colorbar(label="Local memory strength")

plt.xlabel("Coupling K")
plt.ylabel("Oscillator index")

plt.title("Local memory map (forward vs backward)")

plt.tight_layout()

plt.savefig(
    f"{output_dir}/fig_local_memory_map.png",
    dpi=300
)

plt.close()

# ============================================================
# PLOT 2
# ============================================================

plt.figure(figsize=(10,5))

plt.plot(K_values, mean_memory_vs_K)

plt.axvline(
    K_values[peak_idx],
    linestyle='--'
)

plt.xlabel("Coupling K")
plt.ylabel("Mean local memory")

plt.title("Mean local memory vs K")

plt.tight_layout()

plt.savefig(
    f"{output_dir}/fig_local_memory_vs_K.png",
    dpi=300
)

plt.close()

# ============================================================
# PRINT
# ============================================================

print("\nSaved:")
print(f"{output_dir}/local_memory_map.csv")
print(f"{output_dir}/local_memory_summary.csv")
print(f"{output_dir}/fig_local_memory_map.png")
print(f"{output_dir}/fig_local_memory_vs_K.png")

print("\nSUMMARY")
print(summary)

In [ ]:
# ============================================================
# STUART-LANDAU / KURAMOTO
# PERSISTENT MEMORY OSCILLATORS TEST
# which oscillators consistently remember history?
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ============================================================
# LOAD MEMORY MAP
# ============================================================

input_dir = "stuart_landau_local_memory_map_output"
output_dir = "stuart_landau_persistent_memory_output"

os.makedirs(output_dir, exist_ok=True)

memory_map = pd.read_csv(
    f"{input_dir}/local_memory_map.csv",
    index_col=0
)

memory_array = memory_map.values

K_values = memory_map.index.astype(float).values

N = memory_array.shape[1]

# ============================================================
# PERSISTENT MEMORY SCORE
# ============================================================

# average memory per oscillator across all K

persistent_score = memory_array.mean(axis=0)

# normalized
persistent_score_norm = (
    persistent_score / np.max(persistent_score)
)

# top oscillators
top_idx = np.argsort(persistent_score)[::-1]

top_20 = top_idx[:20]

summary = pd.DataFrame({
    "oscillator_index": top_20,
    "persistent_memory_score": persistent_score[top_20],
    "normalized_score": persistent_score_norm[top_20]
})

# ============================================================
# SAVE CSV
# ============================================================

persistent_df = pd.DataFrame({
    "oscillator_index": np.arange(N),
    "persistent_memory_score": persistent_score,
    "normalized_score": persistent_score_norm
})

persistent_df.to_csv(
    f"{output_dir}/persistent_memory_scores.csv",
    index=False
)

summary.to_csv(
    f"{output_dir}/persistent_memory_top20.csv",
    index=False
)

# ============================================================
# PLOT 1
# ============================================================

plt.figure(figsize=(12,5))

plt.plot(
    np.arange(N),
    persistent_score_norm
)

plt.xlabel("Oscillator index")
plt.ylabel("Normalized persistent memory")

plt.title("Persistent memory per oscillator")

plt.tight_layout()

plt.savefig(
    f"{output_dir}/fig_persistent_memory_scores.png",
    dpi=300
)

plt.close()

# ============================================================
# PLOT 2
# ============================================================

plt.figure(figsize=(10,6))

plt.hist(
    persistent_score_norm,
    bins=30
)

plt.xlabel("Normalized persistent memory")
plt.ylabel("Count")

plt.title("Distribution of persistent memory")

plt.tight_layout()

plt.savefig(
    f"{output_dir}/fig_persistent_memory_distribution.png",
    dpi=300
)

plt.close()

# ============================================================
# PRINT
# ============================================================

print("\nSaved:")
print(f"{output_dir}/persistent_memory_scores.csv")
print(f"{output_dir}/persistent_memory_top20.csv")
print(f"{output_dir}/fig_persistent_memory_scores.png")
print(f"{output_dir}/fig_persistent_memory_distribution.png")

print("\nTOP 20 MEMORY OSCILLATORS")
print(summary)

In [ ]:
# ============================================================
# STUART-LANDAU / KURAMOTO
# MEMORY OSCILLATOR STRUCTURE TEST
# do memory oscillators share special omega / phase structure?
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# ============================================================
# PARAMETERS
# ============================================================

N = 400
omega_std = 0.4
seed = 42

TOP_N = 20

# ============================================================
# OUTPUT
# ============================================================

output_dir = "stuart_landau_memory_structure_output"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# RECREATE OMEGA
# ============================================================

np.random.seed(seed)

omega = np.random.normal(0, omega_std, N)

# ============================================================
# LOAD OPTIONAL MEMORY SCORES
# ============================================================

memory_score_candidates = [
    Path(
        "stuart_landau_persistent_memory_output"
    )
    / "persistent_memory_scores.csv",
    Path(
        "results_reference"
    )
    / "stuart_landau"
    / "canonical"
    / "persistent_memory"
    / "persistent_memory_scores.csv",
    Path(
        "results_reference"
    )
    / "kuramoto"
    / "canonical"
    / "persistent_memory"
    / "persistent_memory_scores.csv",
]

memory_score_path = next(
    (
        candidate
        for candidate in memory_score_candidates
        if candidate.exists()
    ),
    None,
)

if memory_score_path is None:
    print(
        "Optional persistent-memory input was not found. "
        "Skipping the top-memory-oscillator analysis."
    )
    memory_scores = None
    top_df = pd.DataFrame()
else:
    memory_scores = pd.read_csv(
        memory_score_path
    )

    required_columns = {
        "persistent_memory_score"
    }

    missing_columns = (
        required_columns
        - set(memory_scores.columns)
    )

    if missing_columns:
        raise ValueError(
            "The persistent-memory table is missing "
            f"required columns: {sorted(missing_columns)}"
        )

    # ========================================================
    # TOP MEMORY OSCILLATORS
    # ========================================================

    top_df = memory_scores.sort_values(
        by="persistent_memory_score",
        ascending=False,
    ).head(TOP_N)

top_indices = top_df["oscillator_index"].values

top_scores = top_df["persistent_memory_score"].values

top_omega = omega[top_indices]

# ============================================================
# GLOBAL COMPARISON
# ============================================================

global_mean = np.mean(omega)
global_std = np.std(omega)

memory_mean = np.mean(top_omega)
memory_std = np.std(top_omega)

summary = pd.DataFrame({
    "global_omega_mean": [global_mean],
    "global_omega_std": [global_std],
    "memory_omega_mean": [memory_mean],
    "memory_omega_std": [memory_std],
    "difference_mean": [memory_mean - global_mean]
})

# ============================================================
# SAVE TABLE
# ============================================================

top_export = pd.DataFrame({
    "oscillator_index": top_indices,
    "omega": top_omega,
    "memory_score": top_scores
})

top_export.to_csv(
    f"{output_dir}/memory_oscillator_structure.csv",
    index=False
)

summary.to_csv(
    f"{output_dir}/memory_structure_summary.csv",
    index=False
)

# ============================================================
# PLOT 1
# ============================================================

plt.figure(figsize=(10,6))

plt.hist(
    omega,
    bins=30,
    alpha=0.5,
    label="All oscillators"
)

plt.hist(
    top_omega,
    bins=15,
    alpha=0.8,
    label="Top memory oscillators"
)

plt.xlabel("Natural frequency ω")
plt.ylabel("Count")

plt.title("Omega distribution")

plt.legend()

plt.tight_layout()

plt.savefig(
    f"{output_dir}/fig_memory_omega_distribution.png",
    dpi=300
)

plt.close()

# ============================================================
# PLOT 2
# ============================================================

plt.figure(figsize=(10,5))

plt.scatter(
    top_indices,
    top_omega,
    s=80
)

plt.xlabel("Oscillator index")
plt.ylabel("Omega")

plt.title("Top memory oscillators")

plt.tight_layout()

plt.savefig(
    f"{output_dir}/fig_memory_oscillators_scatter.png",
    dpi=300
)

plt.close()

# ============================================================
# PRINT
# ============================================================

print("\nSaved:")
print(f"{output_dir}/memory_oscillator_structure.csv")
print(f"{output_dir}/memory_structure_summary.csv")
print(f"{output_dir}/fig_memory_omega_distribution.png")
print(f"{output_dir}/fig_memory_oscillators_scatter.png")

print("\nSUMMARY")
print(summary)

print("\nTOP MEMORY OSCILLATORS")
print(top_export)

In [ ]:
# ============================================================
# MEMORY CARRIERS ROBUSTNESS ACROSS SEEDS
# Check whether the top memory oscillators occupy a special position
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm
from pathlib import Path

OUTDIR = Path("stuart_landau_memory_carriers_seed_robustness_output")
OUTDIR.mkdir(exist_ok=True)

# ----------------------------
# PARAMETERS
# ----------------------------

N = 400
omega_std = 0.4
seeds = range(20)

K_values = np.round(np.arange(0.4, 1.201, 0.025), 3)

dt = 0.02
steps = 6000
cluster_threshold = 0.35
TOP_N = 20

# ----------------------------
# FUNCTIONS
# ----------------------------

def simulate_branch(K_sequence, z_init, omega):
    z = z_init.copy()
    states = {}

    for K in K_sequence:
        for _ in range(steps):
            mean_field = np.mean(z)
            dz = (1 + 1j * omega - np.abs(z)**2) * z + K * (mean_field - z)
            z += dt * dz

        phases = np.angle(z)
        dmat = squareform(pdist(phases.reshape(-1, 1)))
        adjacency = (dmat < cluster_threshold).astype(int)
        cluster_size = adjacency.sum(axis=1)

        states[K] = cluster_size.copy()

    return states, z.copy()

# ----------------------------
# MAIN LOOP
# ----------------------------

all_top_rows = []
all_score_rows = []

for seed in tqdm(seeds):

    rng = np.random.default_rng(seed)
    omega = rng.normal(0, omega_std, N)

    z0 = rng.normal(0, 1, N) + 1j * rng.normal(0, 1, N)

    forward_states, z_last = simulate_branch(K_values, z0, omega)
    backward_states, _ = simulate_branch(K_values[::-1], z_last, omega)

    memory_map = []

    for K in K_values:
        cf = forward_states[K]
        cb = backward_states[K]
        memory_map.append(np.abs(cf - cb))

    memory_map = np.array(memory_map)

    persistent_score = memory_map.mean(axis=0)

    max_score = np.nanmax(persistent_score)
    if max_score == 0 or not np.isfinite(max_score):
        normalized_score = np.zeros_like(persistent_score)
    else:
        normalized_score = persistent_score / max_score

    score_df = pd.DataFrame({
        "seed": seed,
        "oscillator_index": np.arange(N),
        "omega": omega,
        "persistent_memory_score": persistent_score,
        "normalized_score": normalized_score
    })

    all_score_rows.append(score_df)

    top_df = score_df.sort_values(
        "persistent_memory_score",
        ascending=False
    ).head(TOP_N).copy()

    top_df["top_rank"] = np.arange(1, TOP_N + 1)

    all_top_rows.append(top_df)

# ----------------------------
# COMBINE
# ----------------------------

scores_all = pd.concat(all_score_rows, ignore_index=True)
top_all = pd.concat(all_top_rows, ignore_index=True)

scores_all.to_csv(OUTDIR / "memory_scores_all_seeds.csv", index=False)
top_all.to_csv(OUTDIR / "top_memory_oscillators_all_seeds.csv", index=False)

# ----------------------------
# SUMMARY BY SEED
# ----------------------------

summary_by_seed = (
    top_all
    .groupby("seed")
    .agg(
        top_memory_omega_mean=("omega", "mean"),
        top_memory_omega_std=("omega", "std"),
        top_memory_score_mean=("persistent_memory_score", "mean"),
        top_memory_score_max=("persistent_memory_score", "max")
    )
    .reset_index()
)

global_by_seed = (
    scores_all
    .groupby("seed")
    .agg(
        global_omega_mean=("omega", "mean"),
        global_omega_std=("omega", "std")
    )
    .reset_index()
)

summary_by_seed = summary_by_seed.merge(global_by_seed, on="seed")
summary_by_seed["omega_shift_top_minus_global"] = (
    summary_by_seed["top_memory_omega_mean"]
    - summary_by_seed["global_omega_mean"]
)

summary_total = pd.DataFrame([{
    "seeds": len(list(seeds)),
    "top_N": TOP_N,
    "mean_top_memory_omega": top_all["omega"].mean(),
    "std_top_memory_omega": top_all["omega"].std(),
    "mean_global_omega": scores_all["omega"].mean(),
    "std_global_omega": scores_all["omega"].std(),
    "mean_omega_shift_top_minus_global": summary_by_seed["omega_shift_top_minus_global"].mean(),
    "std_omega_shift_top_minus_global": summary_by_seed["omega_shift_top_minus_global"].std()
}])

summary_by_seed.to_csv(OUTDIR / "memory_carriers_summary_by_seed.csv", index=False)
summary_total.to_csv(OUTDIR / "memory_carriers_summary_total.csv", index=False)

# ----------------------------
# PLOTS
# ----------------------------

plt.figure(figsize=(10, 6))
plt.hist(scores_all["omega"], bins=50, alpha=0.5, label="All oscillators")
plt.hist(top_all["omega"], bins=30, alpha=0.8, label="Top memory oscillators")
plt.axvline(scores_all["omega"].mean(), linestyle="--", label="Global omega mean")
plt.axvline(top_all["omega"].mean(), linestyle="--", label="Top memory omega mean")
plt.xlabel("Natural frequency omega")
plt.ylabel("Count")
plt.title("Omega distribution of memory carriers across seeds")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig_memory_omega_distribution_across_seeds.png", dpi=200)
plt.show()

In [ ]:
# ============================================================
# STUART-LANDAU MEMORY CARRIERS: LOCK / UNLOCK EDGE TEST
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

output_dir = "stuart_landau_memory_lock_edge_output"
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

N = 400
omega_std = 0.4
K_values = np.round(np.arange(0.4, 1.201, 0.025), 3)
seeds = range(20)

dt = 0.02
T = 50
steps = int(T / dt)

top_memory_n = 20

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

def simulate_stuart_landau(N, K, omega_std, seed):
    rng = np.random.default_rng(seed)

    omega = rng.normal(0, omega_std, N)

    r0 = 1.0 + 0.05 * rng.normal(size=N)
    theta0 = rng.uniform(0, 2*np.pi, N)
    z = r0 * np.exp(1j * theta0)

    theta_history = []
    R_history = []

    for t in range(steps):
        mean_z = np.mean(z)
        dz = (1 + 1j * omega - np.abs(z)**2) * z + K * (mean_z - z)
        z = z + dt * dz

        if t % 10 == 0:
            theta = np.angle(z)
            R = np.abs(np.mean(np.exp(1j * theta)))
            theta_history.append(theta)
            R_history.append(R)

    theta_history = np.array(theta_history)
    R_history = np.array(R_history)

    return omega, theta_history, R_history


def phase_slip_score(theta_history):
    unwrapped = np.unwrap(theta_history, axis=0)
    dtheta = np.diff(unwrapped, axis=0)

    mean_freq = np.mean(dtheta, axis=0)
    freq_std = np.std(dtheta, axis=0)

    global_mean_freq = np.mean(mean_freq)

    detuning_from_collective = np.abs(mean_freq - global_mean_freq)

    return mean_freq, freq_std, detuning_from_collective


# ------------------------------------------------------------
# LOAD MEMORY OSCILLATORS FROM PREVIOUS TEST
# ------------------------------------------------------------

memory_file = "stuart_landau_persistent_memory_output/persistent_memory_top20.csv"

if os.path.exists(memory_file):
    memory_df = pd.read_csv(memory_file)
    memory_indices = memory_df["oscillator_index"].astype(int).values[:top_memory_n]
else:
    raise FileNotFoundError(
        "Nie znaleziono persistent_memory_top20.csv. "
        "Najpierw uruchom test persistent memory."
    )

# ------------------------------------------------------------
# RUN TEST
# ------------------------------------------------------------

rows = []

for seed in tqdm(seeds):
    print(f"seed={seed}")

    for K in K_values:
        omega, theta_history, R_history = simulate_stuart_landau(
            N=N,
            K=K,
            omega_std=omega_std,
            seed=seed
        )

        mean_freq, freq_std, detuning = phase_slip_score(theta_history)

        memory_mask = np.zeros(N, dtype=bool)
        memory_mask[memory_indices] = True

        non_memory_mask = ~memory_mask

        # Core: oscillators closest to the mean frequency
        core_indices = np.argsort(detuning)[:top_memory_n]

        # Outsiders: oscillators with the largest detuning
        outsider_indices = np.argsort(detuning)[-top_memory_n:]

        rows.append({
            "seed": seed,
            "K": K,
            "R_mean": np.mean(R_history),

            "memory_freq_std": np.mean(freq_std[memory_indices]),
            "core_freq_std": np.mean(freq_std[core_indices]),
            "outsider_freq_std": np.mean(freq_std[outsider_indices]),

            "memory_detuning": np.mean(detuning[memory_indices]),
            "core_detuning": np.mean(detuning[core_indices]),
            "outsider_detuning": np.mean(detuning[outsider_indices]),

            "memory_abs_omega": np.mean(np.abs(omega[memory_indices])),
            "core_abs_omega": np.mean(np.abs(omega[core_indices])),
            "outsider_abs_omega": np.mean(np.abs(omega[outsider_indices])),
        })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# AGGREGATE
# ------------------------------------------------------------

agg = df.groupby("K").agg({
    "R_mean": ["mean", "std"],

    "memory_freq_std": ["mean", "std"],
    "core_freq_std": ["mean", "std"],
    "outsider_freq_std": ["mean", "std"],

    "memory_detuning": ["mean", "std"],
    "core_detuning": ["mean", "std"],
    "outsider_detuning": ["mean", "std"],

    "memory_abs_omega": ["mean", "std"],
    "core_abs_omega": ["mean", "std"],
    "outsider_abs_omega": ["mean", "std"],
}).reset_index()

agg.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in agg.columns
]

# ------------------------------------------------------------
# EDGE SCORE
# Memory oscillators should lie between the core and the outsiders
# ------------------------------------------------------------

agg["memory_edge_score"] = (
    (agg["memory_detuning_mean"] - agg["core_detuning_mean"]) /
    (agg["outsider_detuning_mean"] - agg["core_detuning_mean"] + 1e-9)
)

agg["memory_edge_score"] = agg["memory_edge_score"].clip(0, 1)

peak_idx = agg["memory_edge_score"].idxmax()

summary = pd.DataFrame([{
    "peak_K_memory_edge": agg.loc[peak_idx, "K"],
    "peak_memory_edge_score": agg.loc[peak_idx, "memory_edge_score"],
    "R_mean_at_peak": agg.loc[peak_idx, "R_mean_mean"],
    "memory_detuning_at_peak": agg.loc[peak_idx, "memory_detuning_mean"],
    "core_detuning_at_peak": agg.loc[peak_idx, "core_detuning_mean"],
    "outsider_detuning_at_peak": agg.loc[peak_idx, "outsider_detuning_mean"],
}])

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

df.to_csv(f"{output_dir}/memory_lock_edge_all.csv", index=False)
agg.to_csv(f"{output_dir}/memory_lock_edge_aggregate.csv", index=False)
summary.to_csv(f"{output_dir}/memory_lock_edge_summary.csv", index=False)

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(10, 12))

plt.subplot(4, 1, 1)
plt.errorbar(agg["K"], agg["R_mean_mean"], yerr=agg["R_mean_std"], marker="o")
plt.ylabel("Mean R")
plt.title("Memory carriers as lock/unlock edge oscillators")

plt.subplot(4, 1, 2)
plt.plot(agg["K"], agg["core_detuning_mean"], marker="o", label="core")
plt.plot(agg["K"], agg["memory_detuning_mean"], marker="o", label="memory")
plt.plot(agg["K"], agg["outsider_detuning_mean"], marker="o", label="outsider")
plt.ylabel("Detuning from collective")
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(agg["K"], agg["core_freq_std_mean"], marker="o", label="core")
plt.plot(agg["K"], agg["memory_freq_std_mean"], marker="o", label="memory")
plt.plot(agg["K"], agg["outsider_freq_std_mean"], marker="o", label="outsider")
plt.ylabel("Frequency variability")
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(agg["K"], agg["memory_edge_score"], marker="o")
plt.axvline(summary["peak_K_memory_edge"].iloc[0], linestyle="--", label="peak")
plt.xlabel("Coupling K")
plt.ylabel("Memory edge score")
plt.legend()

plt.tight_layout()
plt.savefig(f"{output_dir}/fig_memory_lock_edge_test.png", dpi=200)
plt.show()

print("Saved:")
print(f"{output_dir}/memory_lock_edge_all.csv")
print(f"{output_dir}/memory_lock_edge_aggregate.csv")
print(f"{output_dir}/memory_lock_edge_summary.csv")
print(f"{output_dir}/fig_memory_lock_edge_test.png")

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(40))

In [ ]:
# ============================================================
# STUART-LANDAU
# LOCAL OUTSIDER REBELLION / STATE TAKEOVER TEST
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# ============================================================
# CONFIG
# ============================================================

N = 400
dt = 0.02
steps = 5000
discard = 2500

lambda_sl = 1.0

K_values = np.linspace(0.4, 1.2, 33)

outsider_fractions = [0.05, 0.10, 0.20, 0.30, 0.40]
outsider_local_coupling = [1.0, 1.5, 2.0, 3.0]

omega_core_std = 0.15
omega_outsider_shift = 1.2

seeds = range(20)

output_dir = "stuart_landau_outsider_rebellion_output"
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def order_parameter(z):
    phases = np.angle(z)
    return np.abs(np.mean(np.exp(1j * phases)))

def classify_state(phases, outsider_idx):

    core_idx = np.setdiff1d(np.arange(len(phases)), outsider_idx)

    core_phase = np.angle(np.mean(np.exp(1j * phases[core_idx])))
    outsider_phase = np.angle(np.mean(np.exp(1j * phases[outsider_idx])))

    phase_gap = np.abs(np.angle(np.exp(1j * (core_phase - outsider_phase))))

    core_sync = np.abs(np.mean(np.exp(1j * phases[core_idx])))
    outsider_sync = np.abs(np.mean(np.exp(1j * phases[outsider_idx])))

    # --------------------------------------------------------
    # CLASSIFICATION
    # --------------------------------------------------------

    if phase_gap < 0.3:
        return "absorbed"

    if outsider_sync > 0.75 and core_sync > 0.75:
        return "dual_cluster"

    if outsider_sync > core_sync + 0.15:
        return "takeover"

    return "fragmented"

# ============================================================
# MAIN
# ============================================================

results = []

for frac in outsider_fractions:

    outsider_count = int(N * frac)

    for local_boost in outsider_local_coupling:

        for seed in tqdm(seeds):

            np.random.seed(seed)

            # ------------------------------------------------
            # OMEGA
            # ------------------------------------------------

            omega = np.random.normal(0, omega_core_std, N)

            outsider_idx = np.random.choice(
                np.arange(N),
                size=outsider_count,
                replace=False
            )

            omega[outsider_idx] += omega_outsider_shift

            # ------------------------------------------------
            # INITIAL STATE
            # ------------------------------------------------

            z = (
                np.random.normal(0, 1, N)
                + 1j * np.random.normal(0, 1, N)
            )

            for K in K_values:

                for _ in range(steps):

                    mean_z = np.mean(z)

                    coupling = K * (mean_z - z)

                    # ----------------------------------------
                    # LOCAL OUTSIDER COUPLING
                    # ----------------------------------------

                    outsider_mean = np.mean(z[outsider_idx])

                    coupling[outsider_idx] += (
                        local_boost * K
                        * (outsider_mean - z[outsider_idx])
                    )

                    dz = (
                        (lambda_sl + 1j * omega
                         - np.abs(z)**2) * z
                        + coupling
                    )

                    z += dt * dz

                phases = np.angle(z)

                state = classify_state(phases, outsider_idx)

                R = order_parameter(z)

                results.append({
                    "seed": seed,
                    "K": K,
                    "outsider_fraction": frac,
                    "local_boost": local_boost,
                    "R": R,
                    "state": state
                })

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(results)

# ============================================================
# AGGREGATE
# ============================================================

agg = (
    df.groupby(
        ["outsider_fraction", "local_boost", "K", "state"]
    )
    .size()
    .reset_index(name="count")
)

agg["fraction"] = agg["count"] / len(seeds)

# ============================================================
# SUMMARY
# ============================================================

summary_rows = []

for frac in outsider_fractions:
    for boost in outsider_local_coupling:

        sub = agg[
            (agg["outsider_fraction"] == frac)
            & (agg["local_boost"] == boost)
        ]

        takeover = sub[sub["state"] == "takeover"]

        if len(takeover) > 0:
            idx = takeover["fraction"].idxmax()

            peak_K = takeover.loc[idx, "K"]
            peak_takeover = takeover.loc[idx, "fraction"]

        else:
            peak_K = np.nan
            peak_takeover = 0

        summary_rows.append({
            "outsider_fraction": frac,
            "local_boost": boost,
            "peak_takeover_fraction": peak_takeover,
            "K_at_peak_takeover": peak_K
        })

summary = pd.DataFrame(summary_rows)

# ============================================================
# PLOTS
# ============================================================

for state_name in ["absorbed", "dual_cluster", "takeover", "fragmented"]:

    plt.figure(figsize=(10, 6))

    for frac in outsider_fractions:

        sub = agg[
            (agg["state"] == state_name)
            & (agg["local_boost"] == 2.0)
            & (agg["outsider_fraction"] == frac)
        ]

        if len(sub) == 0:
            continue

        plt.plot(
            sub["K"],
            sub["fraction"],
            marker="o",
            label=f"outsiders={frac:.2f}"
        )

    plt.xlabel("Coupling K")
    plt.ylabel("Occurrence fraction")
    plt.title(f"{state_name} state probability")
    plt.legend()

    plt.tight_layout()

    plt.savefig(
        f"{output_dir}/fig_{state_name}.png",
        dpi=300
    )

    plt.close()

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    f"{output_dir}/outsider_rebellion_all.csv",
    index=False
)

agg.to_csv(
    f"{output_dir}/outsider_rebellion_aggregate.csv",
    index=False
)

summary.to_csv(
    f"{output_dir}/outsider_rebellion_summary.csv",
    index=False
)

# ============================================================
# PRINT
# ============================================================

print("\nSaved:")
print(f"{output_dir}/outsider_rebellion_all.csv")
print(f"{output_dir}/outsider_rebellion_aggregate.csv")
print(f"{output_dir}/outsider_rebellion_summary.csv")

for state_name in ["absorbed", "dual_cluster", "takeover", "fragmented"]:
    print(f"{output_dir}/fig_{state_name}.png")

print("\nSUMMARY")
print(summary)

In [ ]:
import os
import zipfile

zip_name = "Stuart_Landau_testy.zip"

folders = [
    f for f in os.listdir(".")
    if os.path.isdir(f) and f.startswith("stuart_landau")
]

print("Folders to zip:")
for f in folders:
    print("-", f)

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in folders:
        for root, dirs, file_names in os.walk(folder):
            for file in file_names:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, ".")
                zipf.write(file_path, arcname)

print(f"\nSaved ZIP: {zip_name}")
print(f"Size: {os.path.getsize(zip_name) / 1024 / 1024:.2f} MB")

# Output files remain in the local results directory.

In [ ]:
import os
import zipfile

zip_name = "Stuart_Landau_testy1.zip"

keywords = [
    "stuart_landau",
    "Stuart_Landau",
    "Stuart Landau",
    "st-landau",
    "landau"
]

include_ext = [".csv", ".png", ".jpg", ".jpeg", ".pdf", ".txt", ".json", ".xlsx", ".ipynb", ".zip"]

files_to_zip = []

for root, dirs, file_names in os.walk("."):
    # pomijamy ukryte/systemowe foldery Colab
    if any(part.startswith(".") for part in root.split(os.sep)):
        continue

    for fname in file_names:
        path = os.path.join(root, fname)
        lower_path = path.lower()

        is_stuart_landau = any(k.lower() in lower_path for k in keywords)
        has_good_ext = os.path.splitext(fname)[1].lower() in include_ext

        if is_stuart_landau and has_good_ext:
            if fname != zip_name:
                files_to_zip.append(path)

print("Pliki do spakowania:", len(files_to_zip))
for p in files_to_zip:
    print(p)

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path in files_to_zip:
        arcname = os.path.relpath(path, ".")
        zipf.write(path, arcname)

print("\nZIP zapisany:", zip_name)
print("Rozmiar:", round(os.path.getsize(zip_name) / 1024 / 1024, 2), "MB")

# Output files remain in the local results directory.

In [ ]:
import os, zipfile
from pathlib import Path

zip_name = "Stuart_Landau_testy1.zip"

search_roots = [Path(str(REPOSITORY_ROOT)), Path(".")]

matches = []
for base in search_roots:
    if not base.exists():
        continue
    for p in base.rglob("*"):
        if p.is_file():
            s = str(p).lower()
            if (
                "stuart" in s
                or "landau" in s
                or "recovery" in s
                or "memory" in s
                or "hysteresis" in s
                or "cluster" in s
                or "outsider" in s
                or "rebellion" in s
            ):
                if p.name != zip_name and not p.name.startswith("."):
                    matches.append(p)

# Remove duplicates
matches = sorted(set(matches))

print("Znalezione pliki:", len(matches))
for p in matches[:200]:
    print(p)

if len(matches) == 0:
    print("\nWARNING: No files were found. Check the configured paths.")
    print("cwd =", os.getcwd())
    print("Repository directory contents:")
    print(os.listdir(str(REPOSITORY_ROOT))[:100])
else:
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
        for p in matches:
            arcname = str(p.relative_to(str(REPOSITORY_ROOT))) if str(p).startswith(str(REPOSITORY_ROOT)) else str(p)
            zipf.write(p, arcname)

    print("\nZIP zapisany:", zip_name)
    print("Rozmiar:", round(os.path.getsize(zip_name) / 1024 / 1024, 2), "MB")
# Output files remain in the local results directory.

In [ ]:
import os

for root, dirs, files in os.walk(str(REPOSITORY_ROOT)):
    level = root.replace(str(REPOSITORY_ROOT), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:20]:
        print(f"{indent}  {f}")

In [ ]:
import os, zipfile
from pathlib import Path

zip_name = "Stuart_Landau_testy1.zip"
base = Path(str(REPOSITORY_ROOT))

matches = []

for p in base.rglob("*"):
    if p.is_file():
        s = str(p).lower()
        if (
            ("stuart" in s or "landau" in s)
            and p.name not in ["Stuart_Landau_testy1.zip"]
        ):
            matches.append(p)

# Remove duplicates by relative path
unique = {}
for p in matches:
    arcname = str(p.relative_to(base))
    unique[arcname] = p

print("Pliki do ZIP:", len(unique))
for arcname in unique:
    print(arcname)

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for arcname, p in unique.items():
        zipf.write(p, arcname)

print("ZIP:", zip_name)
print("Rozmiar MB:", round(os.path.getsize(zip_name)/1024/1024, 2))

# Output files remain in the local results directory.

In [ ]:
import zipfile

zip_name = "Stuart_Landau_testy1_FINAL.zip"

files_to_add = [
    "./Stuart_Landau_testy.zip",
    "./Stuart_Landau_testy1.zip",
]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_to_add:
        z.write(f, arcname=f.split("/")[-1])

print("completed:", zip_name)

# Output files remain in the local results directory.